## 0. Pip installs (run first)


In [ ]:
# YHUA-Fusion deps + unrar for rs-FMRI site archives and neuroimaging.rar
import subprocess, sys
subprocess.run(["apt-get", "-qq", "update"], check=False)
subprocess.run(["apt-get", "-qq", "install", "-y", "unrar"], check=False)
%pip -q install numpy pandas scikit-learn matplotlib joblib scipy rarfile nibabel nilearn
try:
    import torch  # noqa: F401
except Exception:
    %pip -q install torch --index-url https://download.pytorch.org/whl/cpu
print("unrar which:", subprocess.getoutput("which unrar") or "(not found)")
try:
    import torch
    print("torch", torch.__version__)
except Exception as exc:
    print("torch not available", exc)


## 1. Mount Google Drive

Colab binds whichever Google account the **runtime** is signed into. Sign in as the account that owns `MyDrive/Project`.


In [ ]:
from pathlib import Path

IN_COLAB = False
DRIVE_MOUNTED = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

MYDRIVE = Path("/content/drive/MyDrive")

if IN_COLAB:
    from google.colab import drive  # type: ignore
    print("Mounting Drive. Sign in as the Google account that owns MyDrive/Project.")
    print("drive.mount uses the runtime Google account.")
    drive.mount("/content/drive", force_remount=False)
    DRIVE_MOUNTED = MYDRIVE.exists()
else:
    print("Not Colab — skipping drive.mount.")
    DRIVE_MOUNTED = MYDRIVE.exists()

print("IN_COLAB", IN_COLAB, "| DRIVE_MOUNTED", DRIVE_MOUNTED, "| MYDRIVE", MYDRIVE, "exists", MYDRIVE.exists())


## 2. Confirm Project streams

Linux paths are case-sensitive: `Phenotypic`, `rs-FMRI` (capital FMRI), `s-MRI` (hyphen).


In [ ]:

import os
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/Project")
PHENO_DIR = DATA_ROOT / "Phenotypic"
RS_DIR = DATA_ROOT / "rs-FMRI"
SMRI_DIR = DATA_ROOT / "s-MRI"

print("DATA_ROOT", DATA_ROOT, "exists", DATA_ROOT.exists())
for label, p in (("Phenotypic", PHENO_DIR), ("rs-FMRI", RS_DIR), ("s-MRI", SMRI_DIR)):
    print(f"{label}: {p} exists={p.exists()}")
    if p.exists():
        try:
            print("  ", sorted(os.listdir(p))[:30])
        except Exception as exc:
            print("  listdir error", exc)

print("--- Project depth ≤ 3 ---")
if DATA_ROOT.exists():
    for dirpath, dirnames, filenames in os.walk(DATA_ROOT, followlinks=False):
        rel = Path(dirpath).relative_to(DATA_ROOT)
        depth = 0 if str(rel) == "." else len(rel.parts)
        dirnames[:] = sorted(d for d in dirnames if not d.startswith("."))
        if depth > 3:
            dirnames[:] = []
            continue
        indent = "  " * depth
        label = DATA_ROOT.name if depth == 0 else Path(dirpath).name
        print(f"{indent}{label}/  files={len(filenames)} subdirs={len(dirnames)}")


## 3. Configuration, seeds, versions


In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
TEST_SIZE = 0.20  # mixed-site held-out fraction; n_correct is counted, never hardcoded
VAL_FRACTION_OF_TRAINVAL = 0.1875
MIN_SITE_N = 8
PCA_RANKS = (1,)  # n_pca=1 lock; model selection = best val metric
C_GRID = (0.01, 0.05)
L1_RATIO = 0.15
KAN_WIDTHS = (2, 3)
KAN_C_GRID = (0.01, 0.05, 0.1)
KAN_TEMP_GRID = (1.0, 0.85, 1.15)
KAN_BIAS_GRID = (0.0, -0.05, 0.05, -0.08, 0.08)
SKIP_LOSO = False  # True skips section 8 LOSO (slow with graph readout)


SITE_FILTER = True  # ALWAYS four base-paper sites. Not a switch.
SITE_FILTER_GROUPS = ("NYU", "USM", "UM", "UCLA")  # UM=UM_1+UM_2; UCLA=UCLA_1+UCLA_2


def four_site_label(s):
    """Map SITE_ID to NYU / USM / UM / UCLA or None."""
    s = str(s).strip().upper().replace("-", "_").replace(" ", "")
    if s.startswith("NYU"):
        return "NYU"
    if s.startswith("USM"):
        return "USM"
    if s in ("UM", "UM_1", "UM_2", "UM2") or s.startswith("UM_") or s.startswith("UM2"):
        return "UM"
    if s.startswith("UCLA"):
        return "UCLA"
    return None


def site_filter_keep(s) -> bool:
    return four_site_label(s) is not None


def clipped_sigmoid_gate(rs, sm):
    """Region-wise (or broadcast) logistic gate; clip logits to avoid exp overflow."""
    z = np.clip(np.asarray(rs, float) - np.asarray(sm, float), -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-z))


def assert_four_site_cohort(n, sites=None, where=""):
    """SITE_FILTER always on. Abort all-site 1007 / 1112 / N>600."""
    globals()["SITE_FILTER"] = True
    n = int(n)
    if n in (1007, 1112):
        raise RuntimeError(
            f"ABORT {where}: banner N {n} — all-site ABIDE-I is forbidden. "
            "Re-run cell 6 load with four-site filter (NYU, USM, UM_1+UM_2, UCLA_1+UCLA_2)."
        )
    if n > 600:
        raise RuntimeError(
            f"ABORT {where}: N={n} > 600; expected four-site NYU/USM/UM/UCLA cohort, not all-site 1112."
        )
    if sites is not None:
        labs = [four_site_label(s) for s in np.asarray(sites).ravel().astype(str)]
        n_other = int(sum(x is None for x in labs))
        if n_other:
            raise RuntimeError(
                f"ABORT {where}: {n_other}/{n} rows are not NYU/USM/UM/UCLA. SITE_FILTER must run before N."
            )
    print("four-site cohort OK", where, "N", n, "SITE_FILTER=True always")
    return n


def apply_site_filter_pheno(pheno: pd.DataFrame) -> pd.DataFrame:
    """Filter SITE_ID before cohort N. Always on (no all-site path)."""
    globals()["SITE_FILTER"] = True
    if pheno is None or len(pheno) == 0:
        return pheno
    n0 = len(pheno)
    col = "site" if "site" in pheno.columns else None
    if col is None:
        raise RuntimeError("SITE_FILTER: no site column — cannot keep NYU/USM/UM/UCLA only")
    keep = pheno[col].map(site_filter_keep)
    out = pheno.loc[keep].reset_index(drop=True)
    vc = out[col].map(four_site_label).value_counts()
    print(
        "SITE_FILTER=True ALWAYS", SITE_FILTER_GROUPS,
        "pheno", n0, "->", len(out), "by group", dict(vc),
        "(NYU, USM, UM=UM_1+UM_2, UCLA=UCLA_1+UCLA_2).",
    )
    assert_four_site_cohort(len(out), out[col], where="apply_site_filter_pheno")
    return out


def mixed_site_kfold_splits(y, site_names, n_splits=5, seed=42):
    """5-fold y-stratified splits where every test site appears in train (avoids fold-5 ComBat collapse)."""
    from sklearn.model_selection import StratifiedKFold

    y = np.asarray(y).ravel()
    s4 = np.array([four_site_label(s) or str(s) for s in np.asarray(site_names).ravel().astype(str)])
    for attempt in range(40):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=int(seed) + attempt)
        splits = list(skf.split(np.zeros(len(y)), y))
        ok = True
        for tr, te in splits:
            if not set(s4[te]).issubset(set(s4[tr])):
                ok = False
                break
        if ok:
            print("mixed-site kfold: attempt", attempt, "n_splits", n_splits, "all test sites seen in train")
            return splits
    print("WARN mixed-site kfold: using y-stratified fallback")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=int(seed))
    return list(skf.split(np.zeros(len(y)), y))

USE_FULL_FC = False
DEMO_N = 1000
DEMO_T = 48
DEMO_SEED = 2
DEMO_SNR = 0.52
N_YEO = 7
N_REGIONS = 246
ATLAS_NAME = "Brainnetome"
YEO_NAMES = ["Vis", "SomMot", "DorsAttn", "SalVentAttn", "Limbic", "Cont", "Default"]

DATA_ROOT = Path("/content/drive/MyDrive/Project")
FORBIDDEN_LOWERCASE = Path("/content/drive/MyDrive/project")
if DATA_ROOT.name == "project" or Path(str(DATA_ROOT)) == FORBIDDEN_LOWERCASE:
    raise RuntimeError("Refusing lowercase MyDrive/project as DATA_ROOT")
USE_SYNTHETIC = False  # True only if Project has zero files and you want the labeled demo
DATA_ROOT_CANDIDATES = [DATA_ROOT]  # Project only

OUTPUT_DIR = Path("/content/yeo_neuro_outputs")
try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
except OSError:
    OUTPUT_DIR = Path.cwd() / "yeo_neuro_outputs"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Python", sys.version.split()[0])
print("numpy", np.__version__, "| pandas", pd.__version__)
import sklearn
print("sklearn", sklearn.__version__)
print("LOCKED DATA_ROOT", DATA_ROOT)
print("N_REGIONS", N_REGIONS, "atlas", ATLAS_NAME, "model selection = best val metric; test n_correct is counted")
print("SITE_FILTER=True ALWAYS groups", SITE_FILTER_GROUPS, "abort if N>600 or N in (1007, 1112)")
print("CANDIDATES", [str(p) for p in DATA_ROOT_CANDIDATES])


In [ ]:
# Working atlas: Brainnetome 246 (Fan et al. 2016). Never AAL / Schaefer / 90 / 116.
N_REGIONS = 246
ATLAS_NAME = "Brainnetome"
print("N_REGIONS", N_REGIONS, "atlas", ATLAS_NAME, "| licensed neuroimaging then AFNI/HTTPS then FALLBACK_BNA; never AAL")
# Copy C:\\neuroimaging to Drive or /content if Colab cannot see C:.

# --- Brainnetome 246 → 48 gyri → Yeo-7 (Fan et al. 2016; odd=L even=R) ---
# --- Brainnetome 246 → 48 gyri → Yeo-7 (Fan et al. 2016; odd=L even=R) ---
# Official 24 gyri × 2 hemispheres = 48 nodes. Sai asked ~46; we print both.
BNA_GYRUS_COUNTS_HEMI = np.array(
    [6, 7, 6, 6, 6, 2, 6, 4, 7, 3, 6, 2, 5, 6, 4, 4, 6, 7, 5, 7, 2, 2, 6, 8], dtype=int
)
assert int(BNA_GYRUS_COUNTS_HEMI.sum()) == 123, BNA_GYRUS_COUNTS_HEMI.sum()
BNA_GYRUS_NAMES_HEMI = [
    "SFG", "MFG", "IFG", "OrG", "PrG", "PCL", "STG", "MTG", "ITG", "FuG", "PhG", "pSTS",
    "SPL", "IPL", "PCun", "PoG", "INS", "CG", "MVOcC", "LOcC", "Amyg", "Hipp", "BG", "Tha",
]
N_GYRUS_OFFICIAL = 48
N_GYRUS_REQUESTED = 46
BNA_ATLAS_URLS = [
    "https://afni.nimh.nih.gov/pub/dist/atlases/afni_atlases_dist/BN_Atlas_246_1mm.nii.gz",
    "https://afni-dev.nimh.nih.gov/pub/dist/atlases/afni_atlases_dist/BN_Atlas_246_1mm.nii.gz",
    "https://github.com/BrainnetomeAtlas/HUMAN-BRAINNETOME-ATLAS/raw/master/BNA_MPM_thr25_1.25mm.nii.gz",
    "https://github.com/BrainnetomeAtlas/HUMAN-BRAINNETOME-ATLAS/raw/master/BN_Atlas_246_2mm.nii.gz",
]

# Licensed Brainnetome (Sai, Windows folder C:\neuroimaging).
# Copy C:\neuroimaging to Drive or /content if Colab cannot see C:.
# Licensed Brainnetome (Sai, Windows folder C:\neuroimaging).
# Copy C:\neuroimaging to Drive or /content if Colab cannot see C:.
# Licensed Brainnetome (Sai). Drive upload is often neuroimaging.rar.
# Copy C:\neuroimaging to Drive or /content if Colab cannot see C:.
# If Colab cannot see C:, upload neuroimaging.rar to Drive; this extracts it to /content/neuroimaging.


def _neuroimaging_archive_names():
    return (
        "neuroimaging.rar",
        "Neuroimaging.rar",
        "NEUROIMAGING.rar",
        "neuroimaging.RAR",
        "Neuroimaging.RAR",
        "neuroimaging.zip",
        "Neuroimaging.zip",
        "NEUROIMAGING.zip",
    )


def neuroimaging_search_dirs():
    dirs = [
        Path("/content/drive/MyDrive/Project"),
        Path("/content/drive/MyDrive"),
        Path("/content"),
    ]
    dr = globals().get("DATA_ROOT")
    if dr is not None:
        drp = Path(dr)
        dirs = [drp, drp.parent] + dirs
    out, seen = [], set()
    for d in dirs:
        key = str(d)
        if key not in seen:
            seen.add(key)
            out.append(d)
    return out


def find_neuroimaging_archive():
    """Find neuroimaging.rar / Neuroimaging.rar under Project and MyDrive (not a full Drive walk)."""
    hits = []
    names = _neuroimaging_archive_names()
    for d in neuroimaging_search_dirs():
        exists = d.exists()
        print("neuroimaging.rar search dir:", d, "exists=", exists)
        if not exists:
            continue
        for n in names:
            p = d / n
            if p.is_file() and p.stat().st_size > 1000:
                hits.append(p)
        try:
            for p in d.glob("*neuroimaging*.rar"):
                if p.is_file() and p.stat().st_size > 1000:
                    hits.append(p)
            for p in d.glob("*Neuroimaging*.rar"):
                if p.is_file() and p.stat().st_size > 1000:
                    hits.append(p)
            for p in d.glob("*neuroimaging*.zip"):
                if p.is_file() and p.stat().st_size > 1000:
                    hits.append(p)
        except OSError as exc:
            print("neuroimaging glob skip", d, exc)
        # shallow recurse (depth ≤ 2) so Project/outputs/neuroimaging.rar still hits
        try:
            for p in d.rglob("*neuroimaging*.rar"):
                try:
                    rel = p.relative_to(d)
                except ValueError:
                    continue
                if len(rel.parts) > 3:
                    continue
                if p.is_file() and p.stat().st_size > 1000:
                    hits.append(p)
        except OSError as exc:
            print("neuroimaging rglob skip", d, exc)
    uniq, seen = [], set()
    for p in hits:
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen:
            seen.add(key)
            uniq.append(p)

    def _score(p: Path) -> tuple:
        n = p.name.lower()
        rar = 0 if n.endswith(".rar") else 1
        exact = 0 if n in ("neuroimaging.rar", "neuroimaging.zip") else 1
        return (exact, rar, n)

    uniq.sort(key=_score)
    if uniq:
        print("found neuroimaging archive:", uniq[0], "candidates=", [str(x) for x in uniq[:8]])
        return uniq[0]
    print("found neuroimaging archive: None")
    return None


def _ensure_unrar_binary():
    import shutil
    import subprocess

    if shutil.which("unrar") or shutil.which("unrar-nonfree"):
        return
    print("apt-get install unrar (needed for neuroimaging.rar, same as rs-FMRI rars)")
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "unrar"], check=False)
    print("unrar which:", shutil.which("unrar") or shutil.which("unrar-nonfree") or "(not found)")


def _extract_archive_to(archive: Path, dest: Path) -> bool:
    import shutil
    import subprocess
    import zipfile

    dest.mkdir(parents=True, exist_ok=True)
    suf = archive.suffix.lower()
    if suf == ".zip":
        print("EXTRACT zip", archive, "->", dest)
        with zipfile.ZipFile(str(archive)) as zf:
            zf.extractall(dest)
        return True
    _ensure_unrar_binary()
    unrar = shutil.which("unrar") or shutil.which("unrar-nonfree")
    seven = shutil.which("7z") or shutil.which("7za")
    cmd = None
    if unrar:
        cmd = ["unrar", "x", "-o+", str(archive), str(dest) + "/"]
    elif seven:
        cmd = ["7z", "x", "-y", f"-o{dest}", str(archive)]
    print("EXTRACT:", " ".join(cmd) if cmd else "python-rarfile", "->", dest)
    if cmd:
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            return True
        print("unrar/7z failed", r.returncode, (r.stderr or r.stdout or "")[-400:])
    try:
        import rarfile

        with rarfile.RarFile(str(archive)) as rf:
            rf.extractall(dest)
        return True
    except Exception as exc:
        print("rarfile failed", archive.name, exc)
        return False


def _neuroimaging_has_bna(root: Path) -> bool:
    if not root.exists():
        return False
    for pat in ("**/BN_Atlas_246*.nii.gz", "**/BN_Atlas_246*.nii", "**/*BNA246*.nii.gz", "**/*BNA_246*.nii.gz"):
        try:
            for p in root.glob(pat):
                if p.is_file() and p.stat().st_size > 10000:
                    return True
        except OSError:
            continue
    return False


def ensure_neuroimaging_extracted(dest: Path | None = None) -> Path:
    """Find Drive neuroimaging.rar and unrar/unzip to /content/neuroimaging before atlas load."""
    dest = Path(dest or "/content/neuroimaging")
    if _neuroimaging_has_bna(dest):
        print("skip neuroimaging.rar extract; BNA already under", dest)
        return dest
    archive = find_neuroimaging_archive()
    if archive is None:
        print("no neuroimaging.rar / Neuroimaging.rar found under Project or MyDrive")
        dest.mkdir(parents=True, exist_ok=True)
        return dest
    print("found rar path:", archive)
    ok = _extract_archive_to(archive, dest)
    print("neuroimaging extract ok=", ok, "dest=", dest)
    return dest


def licensed_neuroimaging_roots():
    """(1) extracted /content/neuroimaging (2) Drive neuroimaging (3) Windows C:/neuroimaging."""
    roots = [
        Path("/content/neuroimaging"),
        Path("/content/drive/MyDrive/neuroimaging"),
    ]
    dr = globals().get("DATA_ROOT")
    if dr is not None:
        drp = Path(dr)
        roots.append(drp.parent / "neuroimaging")  # e.g. MyDrive/neuroimaging next to Project
        roots.append(drp / "neuroimaging")
    roots.extend(
        [
            Path("C:/neuroimaging"),
            Path("C:\\neuroimaging"),
            Path("/mnt/c/neuroimaging"),
        ]
    )
    out, seen = [], set()
    for r in roots:
        key = str(r)
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out


def find_licensed_bna_nifti():
    """Prefer BN_Atlas_246*.nii.gz / BNA246 nifti inside neuroimaging. Print which file is used later."""
    ensure_neuroimaging_extracted(Path("/content/neuroimaging"))
    patterns = (
        "BN_Atlas_246*.nii.gz",
        "BN_Atlas_246*.nii",
        "*BNA246*.nii.gz",
        "*BNA_246*.nii.gz",
        "*BNA*246*.nii.gz",
        "BNA_MPM*.nii.gz",
        "*Brainnetome*246*.nii.gz",
    )
    exact_names = (
        "BN_Atlas_246_1mm.nii.gz",
        "BN_Atlas_246_2mm.nii.gz",
        "BNA246.nii.gz",
        "BNA_246.nii.gz",
        "BNA_MPM_thr25_1.25mm.nii.gz",
    )
    hits = []
    for root in licensed_neuroimaging_roots():
        exists = root.exists()
        print("atlas search root:", root, "exists=", exists)
        if not exists or not root.is_dir():
            continue
        for name in exact_names:
            p = root / name
            if p.is_file() and p.stat().st_size > 10000:
                hits.append(p)
        n_added = 0
        for pat in patterns:
            try:
                for p in root.rglob(pat):
                    try:
                        rel = p.relative_to(root)
                    except ValueError:
                        continue
                    if len(rel.parts) > 5:
                        continue
                    if p.is_file() and p.stat().st_size > 10000:
                        hits.append(p)
                        n_added += 1
                    if n_added >= 20:
                        break
            except OSError as exc:
                print("atlas rglob skip", root, pat, exc)
            if n_added >= 20:
                break
    uniq, seen = [], set()
    for p in hits:
        key = str(p)
        if key not in seen:
            seen.add(key)
            uniq.append(p)

    def _score(p: Path) -> tuple:
        n = p.name.lower()
        gz = 0 if n.endswith(".nii.gz") else 1
        if "bn_atlas_246" in n:
            return (0, gz, n)
        compact = n.replace("_", "").replace("-", "")
        if "bna246" in compact:
            return (1, gz, n)
        if "bna_246" in n or "bna-246" in n:
            return (2, gz, n)
        return (3, gz, n)

    uniq.sort(key=_score)
    if uniq:
        print("licensed BNA nifti candidates:", [str(p) for p in uniq[:8]])
        print("licensed BNA preferred:", uniq[0])
        return uniq[0]
    print("no licensed BNA nifti under neuroimaging roots")
    return None

YEO7_ATLAS_URLS = [
    "https://github.com/ThomasYeoLab/CBIG/raw/master/stable_projects/brain_parcellation/Yeo2011_fcMRI_clustering/1000subjects_reference/Yeo_JNeurophysiol11_MNI152/Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii.gz",
    "https://github.com/ThomasYeoLab/CBIG/raw/master/stable_projects/brain_parcellation/Yeo2011_fcMRI_clustering/1000subjects_reference/Yeo_JNeurophysiol11_SplitLabels/MNI152/Yeo2011_7Networks_MNI152_FreeSurferConformed1mm_LiberalMask.nii.gz",
]
# Documented gyrus→Yeo-7 if the Yeo MNI volume cannot be fetched (FALLBACK_YEO7).
FALLBACK_GYRUS24_YEO = np.array(
    [5, 5, 5, 4, 1, 1, 6, 6, 4, 4, 4, 6, 2, 6, 6, 1, 3, 6, 0, 0, 4, 4, 1, 1], dtype=int
)


def bna_label_to_gyrus48() -> np.ndarray:
    """Map BNA label 1..246 (odd L, even R) → gyrus 0..47. Fan 2016 24 gyri × 2."""
    starts = np.cumsum(np.r_[0, BNA_GYRUS_COUNTS_HEMI[:-1]])
    out = np.zeros(246, dtype=int)
    for lab in range(1, 247):
        sub = (lab - 1) // 2
        g24 = int(np.searchsorted(starts, sub, side="right") - 1)
        hemi = 0 if (lab % 2 == 1) else 1
        out[lab - 1] = g24 + 24 * hemi
    return out


BNA_TO_GYRUS48 = bna_label_to_gyrus48()


def pool_groups(vec: np.ndarray, groups: np.ndarray, n_out: int) -> np.ndarray:
    vec = np.asarray(vec, float).ravel()
    out = np.zeros(n_out, float)
    cnt = np.zeros(n_out, float)
    n = min(vec.size, groups.size)
    for i in range(n):
        g = int(groups[i])
        if 0 <= g < n_out:
            out[g] += vec[i]
            cnt[g] += 1.0
    return out / np.clip(cnt, 1.0, None)


def _http_download(url: str, dest: Path, timeout=90) -> bool:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 1000:
        return True
    try:
        import urllib.request
        print("  download", url)
        req = urllib.request.Request(url, headers={"User-Agent": "yhau-fusion-colab"})
        with urllib.request.urlopen(req, timeout=timeout) as r, open(dest, "wb") as f:
            f.write(r.read())
        return dest.exists() and dest.stat().st_size > 1000
    except Exception as exc:
        print("  download failed", url, exc)
        return False


def load_parcellation_pack(cache_dir: Path | None = None) -> dict:
    """Cache Brainnetome 246 + Yeo-7 MNI. Order: licensed neuroimaging, AFNI/HTTPS, FALLBACK_BNA coords only."""
    cache_dir = Path(cache_dir or "/content/atlas_cache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    pack = {
        "bna_source": "FALLBACK_BNA246",
        "yeo_source": "FALLBACK_YEO7",
        "bna_url": BNA_ATLAS_URLS[0],
        "yeo_url": YEO7_ATLAS_URLS[0],
        "bna_path": None,
        "n_gyrus_official": N_GYRUS_OFFICIAL,
        "n_gyrus_requested": N_GYRUS_REQUESTED,
        "bna_img": None,
        "bna_data": None,
        "bna_to_yeo": None,
    }
    bna_path = cache_dir / "BNA_MPM_thr25_1.25mm.nii.gz"
    yeo_path = cache_dir / "Yeo2011_7Networks_MNI152_1mm.nii.gz"
    bna_ok = False
    ensure_neuroimaging_extracted(Path("/content/neuroimaging"))
    licensed = find_licensed_bna_nifti()
    if licensed is not None:
        bna_path = Path(licensed)
        bna_ok = True
        pack["bna_url"] = str(licensed)
        pack["bna_path"] = str(licensed)
        pack["bna_source"] = "LICENSED_BNA246"
        print("BNA atlas from licensed neuroimaging", bna_path)
    else:
        for u in BNA_ATLAS_URLS:
            if _http_download(u, bna_path):
                pack["bna_url"] = u
                pack["bna_path"] = str(bna_path)
                bna_ok = True
                print("BNA atlas from AFNI/HTTPS", u, "saved", bna_path)
                break
    yeo_ok = False
    for u in YEO7_ATLAS_URLS:
        if _http_download(u, yeo_path):
            pack["yeo_url"] = u
            yeo_ok = True
            break
    try:
        import nibabel as nib
        if bna_ok:
            img = nib.load(str(bna_path))
            data = np.asanyarray(img.dataobj)
            labs = set(np.unique(data).astype(int)) - {0}
            if len(labs) >= 200:
                pack["bna_img"] = img
                pack["bna_data"] = data
                if pack["bna_source"] != "LICENSED_BNA246":
                    pack["bna_source"] = "OFFICIAL_BNA246"
                pack["bna_path"] = str(bna_path)
                print("BNA atlas OFFICIAL", bna_path, "unique labels", len(labs), "shape", data.shape)
            else:
                print("BNA file has too few labels", len(labs), "→ FALLBACK_BNA246")
                pack["bna_source"] = "FALLBACK_BNA246"
                pack["bna_path"] = None
        if yeo_ok:
            yimg = nib.load(str(yeo_path))
            pack["yeo_source"] = "OFFICIAL_YEO7"
            print("Yeo-7 atlas OFFICIAL", yeo_path, "shape", yimg.shape)
            if pack["bna_img"] is not None:
                from nibabel.processing import resample_from_to
                y_in_bna = resample_from_to(yimg, pack["bna_img"], order=0)
                ydat = np.asanyarray(y_in_bna.dataobj)
                bdat = pack["bna_data"]
                b2y = np.zeros(246, dtype=int)
                for r in range(1, 247):
                    m = bdat == r
                    if not np.any(m):
                        g24 = int(BNA_TO_GYRUS48[r - 1] % 24)
                        b2y[r - 1] = int(FALLBACK_GYRUS24_YEO[g24])
                        continue
                    vals = ydat[m]
                    vals = vals[(vals >= 1) & (vals <= 7)]
                    if vals.size == 0:
                        g24 = int(BNA_TO_GYRUS48[r - 1] % 24)
                        b2y[r - 1] = int(FALLBACK_GYRUS24_YEO[g24])
                    else:
                        b2y[r - 1] = int(np.bincount(vals.astype(int)).argmax()) - 1
                pack["bna_to_yeo"] = b2y
                print("BNA→Yeo-7 from MNI overlap (official volumes)")
    except Exception as exc:
        print("atlas load/overlap failed", exc)
    if pack["bna_to_yeo"] is None:
        pack["bna_to_yeo"] = np.array([int(FALLBACK_GYRUS24_YEO[g % 24]) for g in BNA_TO_GYRUS48], dtype=int)
        if pack["yeo_source"] != "OFFICIAL_YEO7":
            pack["yeo_source"] = "FALLBACK_YEO7"
        print("BNA→Yeo-7 using documented gyrus lookup", pack["yeo_source"])
    print("n_gyrus official=48 (24×2 hemispheres); requested=46 — using 48")
    print("parcellation bna=", pack["bna_source"], "yeo=", pack["yeo_source"])
    used = pack.get("bna_path") or pack["bna_source"]
    print("atlas file used:", used)
    if pack["bna_source"] == "FALLBACK_BNA246":
        print("FALLBACK_BNA coords only (no licensed neuroimaging NIfTI and AFNI/HTTPS unavailable)")
    return pack

def fallback_bna246_labels(shape) -> np.ndarray:
    """Deterministic 6×7×6 spatial bins (252), first 246 used. Not official BNA."""
    d0, d1, d2 = shape
    z, y, x = np.ogrid[:d0, :d1, :d2]
    i = (z * 6) // d0
    j = (y * 7) // d1
    k = (x * 6) // d2
    lab = (i * 42 + j * 6 + k).astype(np.int32) + 1
    lab = np.clip(lab, 1, 252)
    lab[lab > 246] = 0
    return lab


def _mean_t_nifti(path: Path):
    import nibabel as nib
    img = nib.load(str(path), mmap=True)
    data = np.asanyarray(img.dataobj, dtype=np.float32)
    if data.ndim == 4:
        data = np.nanmean(data, axis=-1)
    return img, np.asarray(data, np.float32)


def rs_bna246_vector(path: Path, pack: dict) -> np.ndarray | None:
    try:
        img, vol = _mean_t_nifti(path)
    except Exception as exc:
        print("  rs nifti fail", path.name, exc)
        return None
    if vol.ndim != 3:
        return None
    labels = None
    if pack.get("bna_img") is not None:
        try:
            import nibabel as nib
            from nibabel.processing import resample_from_to
            vol_img = nib.Nifti1Image(vol, img.affine, img.header)
            vol_a = resample_from_to(vol_img, pack["bna_img"], order=1)
            vol = np.asanyarray(vol_a.dataobj, dtype=np.float32)
            labels = pack["bna_data"]
        except Exception as exc:
            print("  resample-to-BNA fail", path.name, exc, "→ FALLBACK_BNA246 for this file")
            labels = None
    if labels is None:
        labels = fallback_bna246_labels(vol.shape)
    out = np.zeros(246, float)
    cnt = np.zeros(246, float)
    m = np.isfinite(vol)
    labs = np.asarray(labels, int)
    for r in range(1, 247):
        sel = m & (labs == r)
        if np.any(sel):
            out[r - 1] = float(vol[sel].mean())
            cnt[r - 1] = float(sel.sum())
    return out

def _bna246_roi_means(vol, labels):
    """Mean of finite nonzero voxels in each BNA label 1..246. Empty masks stay 0 (not imputed)."""
    vol = np.asarray(vol, dtype=np.float32)
    labs = np.asarray(labels, dtype=np.int32)
    if vol.shape != labs.shape:
        raise ValueError(f"vol shape {vol.shape} != labels {labs.shape}")
    m = np.isfinite(vol) & (vol != 0) & (labs >= 1) & (labs <= 246)
    out = np.zeros(246, float)
    cnt = np.zeros(246, float)
    if not np.any(m):
        return out, cnt
    labs_m = labs[m]
    vol_m = vol[m].astype(np.float64, copy=False)
    cnt = np.bincount(labs_m, minlength=247)[1:247].astype(float)
    sums = np.bincount(labs_m, weights=vol_m, minlength=247)[1:247]
    nz = cnt > 0
    out[nz] = sums[nz] / cnt[nz]
    return out, cnt


def _print_smri_bna_missing_counts(cnts, atlas_cnt=None, source=""):
    """Print empty-mask counts. Subcortical BNA (labels 211-246) are not filled from cortex."""
    cnts = np.asarray(cnts, float)
    if cnts.ndim == 1:
        cnts = cnts[None, :]
    empty = cnts <= 0
    n_subj = int(empty.shape[0])
    per_subj = empty.sum(1)
    per_reg = empty.sum(0)
    subc = slice(210, 246)
    print(
        "sMRI BNA-246 thickness missing masks: subjects", n_subj, "source", source or "(pack)",
        "empty-ROI per subject min/mean/max",
        int(per_subj.min()) if n_subj else 0,
        float(per_subj.mean()) if n_subj else 0.0,
        int(per_subj.max()) if n_subj else 0,
    )
    print(
        "  subjects with any empty ROI", int((per_subj > 0).sum()) if n_subj else 0,
        "subjects with all 246 empty", int((per_subj == 246).sum()) if n_subj else 0,
    )
    print(
        "  regions empty in ≥1 subject", int((per_reg > 0).sum()) if n_subj else 0,
        "regions empty in all subjects", int((per_reg == n_subj).sum()) if n_subj else 0,
    )
    print(
        "  subcortical labels 211-246 empty per subject mean",
        float(empty[:, subc].sum(1).mean()) if n_subj else 0.0,
        "of 36 (Amyg/Hipp/BG/Tha; zeroed if mask has no overlapping voxels)",
    )
    if atlas_cnt is not None:
        ac = np.asarray(atlas_cnt, float).ravel()
        n_atl = int((ac[:246] <= 0).sum()) if ac.size >= 246 else -1
        print("  atlas volume itself has empty BNA labels", n_atl, "(those nodes stay 0; not invented)")
    always = np.where(per_reg == n_subj)[0] + 1 if n_subj else np.array([], int)
    if always.size:
        print("  labels empty for every subject (zeroed, not imputed):", always.tolist()[:40], ("..." if always.size > 40 else ""))


def smri_bna246_thickness(path: Path, pack: dict):
    """246-D morphological vector: mean thickness in volumetric Brainnetome-246 voxel masks.

    Uses licensed neuroimaging.rar BN_Atlas_246 (then AFNI/HTTPS, then FALLBACK_BNA coords).
    Not AAL. Not FreeSurfer surface BNA. Empty masks are zeroed, never filled from a gyrus neighbour.
    Returns (vec246, voxel_counts246) or None.
    """
    pack = pack or {}
    try:
        import nibabel as nib
        img = nib.load(str(path), mmap=True)
        vol = np.asanyarray(img.dataobj, dtype=np.float32)
    except Exception as exc:
        print("  sMRI thickness nifti fail", Path(path).name, exc)
        return None
    if vol.ndim == 4:
        vol = np.nanmean(vol, axis=-1)
    if vol.ndim != 3:
        print("  sMRI thickness not 3D", Path(path).name, vol.shape)
        return None
    labels = None
    src = pack.get("bna_source") or "FALLBACK_BNA246"
    if pack.get("bna_img") is not None:
        try:
            from nibabel.processing import resample_from_to
            vol_img = nib.Nifti1Image(vol, img.affine, img.header)
            vol_a = resample_from_to(vol_img, pack["bna_img"], order=1)
            vol = np.asanyarray(vol_a.dataobj, dtype=np.float32)
            labels = pack["bna_data"]
            src = pack.get("bna_source") or "LICENSED_BNA246"
        except Exception as exc:
            print("  sMRI resample-to-BNA fail", Path(path).name, exc, "→ FALLBACK_BNA246 for this file")
            labels = None
            src = "FALLBACK_BNA246"
    if labels is None:
        labels = fallback_bna246_labels(vol.shape)
        src = "FALLBACK_BNA246"
    out, cnt = _bna246_roi_means(vol, labels)
    n_empty = int((cnt <= 0).sum())
    n_subc = int((cnt[210:246] <= 0).sum())
    if n_empty == 246:
        print("  sMRI thickness all-246 empty", Path(path).name, "source", src, "(kept as zeros; not imputed)")
    return out, cnt


def smri_nodes_246(Xs):
    """Per-subject 246-D BNA thickness for region-wise gates. Never gyrus-48 broadcast."""
    X = np.asarray(Xs, float)
    one = X.ndim == 1
    if one:
        X = X.reshape(1, -1)
    n, d = X.shape
    out = np.zeros((n, 246), float)
    take = min(int(d), 246)
    if take:
        out[:, :take] = np.nan_to_num(X[:, :take], nan=0.0)
    if d != 246 and not getattr(smri_nodes_246, "_warned", False):
        print(
            "WARN smri_nodes_246: Xs has", d,
            "columns; expected 246 BNA-region thicknesses. "
            "No gyrus-48 copy onto nodes (remaining regions stay 0).",
        )
        smri_nodes_246._warned = True
    return out[0] if one else out

# --- SS-InterGNN novelty tree (Colab-light; status printed, never silently skipped) ---
MODULE_STATUS = {}
SKIP_LOSO = bool(globals().get("SKIP_LOSO", False))
USE_FULL_FC = bool(globals().get("USE_FULL_FC", False))  # 4D Pearson slow; cache /content/rs_fc_cache; else mean-T + anatomical adj
GRAPH_HIDDEN = 8
GRAPH_KNN = 8


def _set_status(name, status, note=""):
    MODULE_STATUS[name] = {"status": status, "note": note}
    print(f"MODULE {name}: {status}" + (f" — {note}" if note else ""))


def anatomical_adj_bna246() -> np.ndarray:
    """Fixed Brainnetome gyrus-block adjacency (same gyrus connected)."""
    A = np.eye(246, dtype=float)
    g = BNA_TO_GYRUS48
    for i in range(246):
        same = np.where(g == g[i])[0]
        A[i, same] = 1.0
    return A


def row_normalize(A: np.ndarray) -> np.ndarray:
    A = np.asarray(A, float)
    d = np.clip(A.sum(axis=1, keepdims=True), 1e-8, None)
    return A / d


def knn_from_features(H: np.ndarray, k: int = 8) -> np.ndarray:
    H = np.asarray(H, float)
    nrm = np.sqrt(np.clip((H * H).sum(1), 1e-8, None))
    sim = (H @ H.T) / nrm[:, None] / nrm[None, :]
    A = np.zeros((H.shape[0], H.shape[0]), float)
    kk = min(k + 1, H.shape[0])
    for i in range(H.shape[0]):
        idx = np.argpartition(-sim[i], kk - 1)[:kk]
        A[i, idx] = 1.0
    return A


def leaky_relu(x, a=0.2):
    return np.where(x > 0, x, a * x)


def spatial_mix_numpy(H, Ahat, rng, fout=GRAPH_HIDDEN):
    """One SpatialMix layer (neighborhood mix). Not Tang GAT."""
    n, f = H.shape
    W = rng.normal(0.0, 1.0 / np.sqrt(f), size=(f, fout))
    al = rng.normal(0.0, 0.1, size=fout)
    ar = rng.normal(0.0, 0.1, size=fout)
    HW = H @ W
    out = np.zeros((n, fout), float)
    for i in range(n):
        js = np.flatnonzero(Ahat[i] > 0)
        if js.size == 0:
            js = np.array([i])
        e = leaky_relu(HW[i] @ al + HW[js] @ ar)
        a = np.exp(e - np.max(e))
        a = a / np.clip(a.sum(), 1e-8, None)
        out[i] = (a[:, None] * HW[js]).sum(0)
    return np.tanh(out)


def spectral_mix_numpy(H, Ahat, rng, K=2, fout=GRAPH_HIDDEN):
    """One SpectralMix layer (Chebyshev polynomial mix). Not Tang ChebConv."""
    n, f = H.shape
    d = np.clip(Ahat.sum(1), 1e-8, None)
    Dm = 1.0 / np.sqrt(d)
    An = (Dm[:, None] * Ahat) * Dm[None, :]
    L = np.eye(n) - An
    T0, T1 = H, L @ H
    feats = [T0, T1]
    Tk_1, Tk = T0, T1
    for _ in range(2, K + 1):
        Tk1 = 2.0 * (L @ Tk) - Tk_1
        feats.append(Tk1)
        Tk_1, Tk = Tk, Tk1
    cat = np.concatenate(feats, axis=1)
    W = rng.normal(0.0, 1.0 / np.sqrt(cat.shape[1]), size=(cat.shape[1], fout))
    return np.tanh(cat @ W)


def spatial_mix_torch(H, Ahat, rng, fout=GRAPH_HIDDEN):
    import torch

    torch.set_num_threads(1)
    H_t = torch.from_numpy(np.asarray(H, np.float32))
    mask = torch.from_numpy((np.asarray(Ahat) > 0).astype(np.float32))
    n, f = H_t.shape
    W = torch.from_numpy(rng.normal(0.0, 1.0 / np.sqrt(f), size=(f, fout)).astype(np.float32))
    al = torch.from_numpy(rng.normal(0.0, 0.1, size=fout).astype(np.float32))
    ar = torch.from_numpy(rng.normal(0.0, 0.1, size=fout).astype(np.float32))
    HW = H_t @ W
    E = HW @ al
    Er = HW @ ar
    scores = E[:, None] + Er[None, :]
    scores = torch.where(scores > 0, scores, 0.2 * scores)
    scores = scores.masked_fill(mask < 0.5, -1e9)
    attn = torch.softmax(scores, dim=1)
    out = torch.tanh(attn @ HW)
    return out.detach().cpu().numpy()


def spectral_mix_torch(H, Ahat, rng, K=2, fout=GRAPH_HIDDEN):
    import torch

    torch.set_num_threads(1)
    H_t = torch.from_numpy(np.asarray(H, np.float32))
    A = torch.from_numpy(np.asarray(Ahat, np.float32))
    n, f = H_t.shape
    d = torch.clamp(A.sum(1), min=1e-8)
    Dm = 1.0 / torch.sqrt(d)
    An = (Dm[:, None] * A) * Dm[None, :]
    L = torch.eye(n) - An
    T0, T1 = H_t, L @ H_t
    feats = [T0, T1]
    Tk_1, Tk = T0, T1
    for _ in range(2, K + 1):
        Tk1 = 2.0 * (L @ Tk) - Tk_1
        feats.append(Tk1)
        Tk_1, Tk = Tk, Tk1
    cat = torch.cat(feats, dim=1)
    W = torch.from_numpy(rng.normal(0.0, 1.0 / np.sqrt(int(cat.shape[1])), size=(int(cat.shape[1]), fout)).astype(np.float32))
    return torch.tanh(cat @ W).detach().cpu().numpy()


def spatial_mix(H, Ahat, rng, fout=GRAPH_HIDDEN):
    try:
        import torch  # noqa: F401

        return spatial_mix_torch(H, Ahat, rng, fout)
    except Exception:
        return spatial_mix_numpy(H, Ahat, rng, fout)


def spectral_mix(H, Ahat, rng, K=2, fout=GRAPH_HIDDEN):
    try:
        import torch  # noqa: F401

        return spectral_mix_torch(H, Ahat, rng, K=K, fout=fout)
    except Exception:
        return spectral_mix_numpy(H, Ahat, rng, K=K, fout=fout)


def coarsen_246_to_48(H246: np.ndarray) -> np.ndarray:
    """Assignment from official BNA region to gyrus map (not learned)."""
    H246 = np.asarray(H246, float)
    fout = H246.shape[1]
    H48 = np.zeros((48, fout), float)
    cnt = np.zeros(48, float)
    for r in range(min(246, H246.shape[0])):
        g = int(BNA_TO_GYRUS48[r])
        H48[g] += H246[r]
        cnt[g] += 1.0
    return H48 / np.clip(cnt[:, None], 1.0, None)


def pool_48_to_yeo7(H48: np.ndarray) -> np.ndarray:
    """Fixed NeuroSubnet-Pool: 48 gyri to Yeo-7."""
    H48 = np.asarray(H48, float)
    fout = H48.shape[1]
    yeo_of = np.array([int(FALLBACK_GYRUS24_YEO[g % 24]) for g in range(48)], dtype=int)
    pack = globals().get("ATLAS_PACK") or {}
    b2y = pack.get("bna_to_yeo")
    if b2y is not None:
        yeo_of = np.zeros(48, dtype=int)
        cnt = np.zeros((48, 7), float)
        for r in range(246):
            g = int(BNA_TO_GYRUS48[r])
            yid = int(b2y[r]) if r < len(b2y) else int(FALLBACK_GYRUS24_YEO[g % 24])
            yid = int(np.clip(yid, 0, 6))
            cnt[g, yid] += 1
        yeo_of = cnt.argmax(1)
    H7 = np.zeros((7, fout), float)
    c7 = np.zeros(7, float)
    for g in range(min(48, H48.shape[0])):
        yid = int(yeo_of[g])
        H7[yid] += H48[g]
        c7[yid] += 1.0
    return H7 / np.clip(c7[:, None], 1.0, None)


def _try_torch_layers():
    try:
        import torch

        _set_status("torch", "IMPLEMENTED_LIGHT", f"cpu torch {torch.__version__}; SpatialMix+SpectralMix (hidden={GRAPH_HIDDEN})")
        return True
    except Exception:
        _set_status("torch", "IMPLEMENTED_LIGHT", "torch missing; numpy SpatialMix + SpectralMix")
        return False


def ss_intergnn_subject(H_nodes, A_anat, rng):
    Aknn = knn_from_features(H_nodes, k=GRAPH_KNN)
    A = row_normalize(((A_anat + Aknn) > 0).astype(float))
    np.fill_diagonal(A, 1.0)
    A = row_normalize(A)
    H_sp = 0.5 * (spatial_mix(H_nodes, A, rng) + spatial_mix(H_nodes, A, rng))  # 2-head SpatialMix, not GAT
    H_spec = spectral_mix(H_nodes, A, rng)  # Chebyshev filters as SpectralMix, not ChebConv module
    H246 = 0.5 * (H_sp + H_spec)
    H48 = coarsen_246_to_48(H246)
    H7 = pool_48_to_yeo7(H48)
    readout = np.concatenate([H246.mean(0), H48.mean(0), H7.mean(0)])
    return readout, A.shape, H246.shape, H48.shape, H7.shape


def pearson_fc_from_roi_ts(ts: np.ndarray) -> np.ndarray:
    """(T, 246) ROI timeseries -> Fisher-z node strength (246,)."""
    ts = np.asarray(ts, float)
    if ts.ndim != 2 or ts.shape[1] < 2:
        return np.zeros(246, float)
    x = ts - ts.mean(0, keepdims=True)
    std = np.clip(x.std(0, keepdims=True), 1e-8, None)
    z = x / std
    C = (z.T @ z) / max(z.shape[0] - 1, 1)
    C = np.clip(C, -0.999999, 0.999999)
    Fz = np.arctanh(C)
    np.fill_diagonal(Fz, 0.0)
    strength = Fz.sum(1)
    if strength.size < 246:
        out = np.zeros(246, float)
        out[: strength.size] = strength
        return out
    return strength[:246]


def _ensure_dir(p, fallback):
    p = Path(p)
    try:
        p.mkdir(parents=True, exist_ok=True)
        return p
    except OSError:
        fb = Path(fallback)
        fb.mkdir(parents=True, exist_ok=True)
        print("NOTE: could not mkdir", p, "— using", fb)
        return fb


def maybe_build_fc_cache(bundle: dict, fc_dir: Path) -> tuple:
    """Return (N,246) node-strength or None. Never silently skip: prints the path."""
    fc_dir = _ensure_dir(fc_dir, Path("/tmp/rs_fc_cache"))
    marker = fc_dir / "FC_PATH.txt"
    strength_p = fc_dir / "node_strength_246.npy"
    if strength_p.exists() and strength_p.stat().st_size > 1000:
        arr = np.load(strength_p)
        marker.write_text("cached Fisher-z node-strength from /content/rs_fc_cache\n", encoding="utf-8")
        return arr, "cached_fisherz_node_strength"
    ts_dir = fc_dir / "roi_ts"
    n = int(np.asarray(bundle.get("X_bna")).shape[0]) if bundle.get("X_bna") is not None else 0
    if USE_FULL_FC and ts_dir.exists():
        ids = list(bundle.get("meta", {}).get("sub_id", [])) if hasattr(bundle.get("meta"), "get") else []
        if "meta" in bundle and hasattr(bundle["meta"], "columns") and "SUB_ID" in bundle["meta"].columns:
            ids = [abide_canon(x) for x in bundle["meta"]["SUB_ID"].astype(str)]
        rows = []
        n_hit = 0
        for i in range(n):
            sid = ids[i] if i < len(ids) else f"{i:07d}"
            p = ts_dir / f"{sid}_bna246_ts.npy"
            if p.exists():
                rows.append(pearson_fc_from_roi_ts(np.load(p)))
                n_hit += 1
            else:
                rows.append(np.zeros(246, float))
        arr = np.vstack(rows) if rows else None
        if arr is not None and n_hit > 0:
            np.save(strength_p, arr)
            marker.write_text(f"Pearson+Fisher-z from cached ROI ts ({n_hit}/{n})\n", encoding="utf-8")
            return arr, f"pearson_from_cached_roi_ts n_hit={n_hit}"
    marker.write_text(
        "mean-T BNA-246 node features + fixed anatomical BNA-246 adjacency "
        "(4D Pearson skipped: too slow unless USE_FULL_FC and roi_ts cache exist)\n",
        encoding="utf-8",
    )
    return None, "meanT_plus_anatomical_adj"


def ss_intergnn_readout(bundle: dict, cache_dir: Path | None = None):
    """Build (N, 24) multiscale readout. Caches npy. Prints which FC path."""
    cache_dir = _ensure_dir(cache_dir or "/content/ss_intergnn_cache", Path("/tmp/ss_intergnn_cache"))
    ck = cache_dir / "readout_ms.npy"
    meta_p = cache_dir / "graph_meta.json"
    Xbna = bundle.get("X_bna")
    Xs = bundle.get("Xs")
    skipped = [
        "FunctionalConnectivity", "MorphologicalFeatures", "MultimodalEncoding", "SubjectAdaptiveGraph",
        "SpatialMix", "SpectralMix", "SS_InterGNN",
        "AdaptiveCoarsening_246to48", "NeuroSubnetPool_48toYeo7",
        "MultiscaleReadout", "SiteRobustComBat", "KAN_classifier", "YHUA_three_experts",
    ]
    if Xbna is None or Xs is None:
        for name in skipped:
            _set_status(name, "SKIPPED", "need X_bna and Xs")
        _set_status("GAT_spatial", "SKIPPED", "not used; SpatialMix is the spatial box")
        _set_status("ChebConv_spectral", "SKIPPED", "not used; SpectralMix is the spectral box")
        return None, {"reason": "missing X_bna/Xs"}
    Xbna = np.asarray(Xbna, float)
    Xs = np.asarray(Xs, float)
    n = Xbna.shape[0]
    if ck.exists() and ck.stat().st_size > 64:
        Xg = np.load(ck)
        meta = json.loads(meta_p.read_text()) if meta_p.exists() else {}
        smri_dim = int(np.asarray(Xs).shape[1]) if np.asarray(Xs).ndim == 2 else -1
        if int(meta.get("smri_dim") or -1) == smri_dim == 246 and int(meta.get("n_regions") or 246) == 246:
            print("SS-InterGNN cache hit", ck, Xg.shape, "smri_dim", smri_dim)
            for k, v in (meta.get("module_status") or {}).items():
                MODULE_STATUS[k] = v
                print(f"MODULE {k}: {v.get('status')} — {v.get('note','')} (cached)")
            _force_skip_gat_cheb()
            return Xg, meta
        print("SS-InterGNN cache stale (need 246-D BNA thickness fusion); recomputing", ck, "cached smri_dim", meta.get("smri_dim"), "now", smri_dim)

    fc_strength, fc_path = maybe_build_fc_cache(bundle, Path("/content/rs_fc_cache"))
    if fc_path.startswith("pearson") or fc_path.startswith("cached"):
        _set_status(
            "FunctionalConnectivity",
            "IMPLEMENTED_LIGHT",
            f"{fc_path}; Fisher-z node strength; cache /content/rs_fc_cache",
        )
    else:
        _set_status(
            "FunctionalConnectivity",
            "IMPLEMENTED_LIGHT",
            "mean-T BNA-246 node features + fixed anatomical gyrus adjacency "
            "(not 4D Pearson FC; too slow for N=1007 Colab). Cache /content/rs_fc_cache",
        )

    _set_status("MorphologicalFeatures", "IMPLEMENTED_LIGHT", "s-MRI BNA-246 mean thickness per volumetric atlas mask (not gyrus-48 broadcast)")
    _set_status("MultimodalEncoding", "IMPLEMENTED_LIGHT", "adaptive gated fusion per BNA-246 region (rs vs that region's thickness)")
    _set_status("SubjectAdaptiveGraph", "IMPLEMENTED_LIGHT", "per-subject kNN(k=8) on node feats + anatomical gyrus edges, row-normalized")
    torch_ok = _try_torch_layers()
    backend = "torch" if torch_ok else "numpy"
    _set_status("SpatialMix", "IMPLEMENTED_LIGHT", f"2-head spatial mix hidden=8 ({backend}); GAT_spatial SKIPPED")
    _set_status("SpectralMix", "IMPLEMENTED_LIGHT", f"one-layer spectral polynomial mix K=2 hidden=8 ({backend})")
    _set_status("SS_InterGNN", "IMPLEMENTED_LIGHT", "0.5*(SpatialMix multi-head + SpectralMix Chebyshev)")
    _force_skip_gat_cheb()
    _set_status("AdaptiveCoarsening_246to48", "IMPLEMENTED_FAITHFUL", "fixed Brainnetome region-to-gyrus assignment (Fan 2016), not learned")
    yeo_src = (globals().get("ATLAS_PACK") or {}).get("yeo_source", "FALLBACK_YEO7")
    _set_status(
        "NeuroSubnetPool_48toYeo7",
        "IMPLEMENTED_FAITHFUL" if "OFFICIAL" in str(yeo_src) else "IMPLEMENTED_LIGHT",
        f"48 to Yeo-7 pool ({yeo_src})",
    )
    _set_status("MultiscaleReadout", "IMPLEMENTED_LIGHT", "concat mean-pool 246, 48, 7 -> 24-D")
    _set_status("SiteRobustComBat", "IMPLEMENTED_FAITHFUL", "train-only ComBat on readout in expert transform; site not a feature; no ADOS")
    _set_status("YHUA_three_experts", "IMPLEMENTED_FAITHFUL", "elastic-net rs/sMRI/pheno + inv-var fusion remains as fallback")
    _set_status("KAN_classifier", "IMPLEMENTED_LIGHT", "RBF-KAN on fused/expert logits (locked on val)")

    A_anat = anatomical_adj_bna246()
    rng = np.random.default_rng(SEED)
    Xg = np.zeros((n, 3 * GRAPH_HIDDEN), float)
    last_shapes = None
    sm_all = smri_nodes_246(Xs)
    for i in range(n):
        sm = np.asarray(sm_all[i], float).ravel()[:246]
        rs = np.asarray(Xbna[i], float).ravel()[:246]
        if rs.size < 246:
            rs = np.pad(rs, (0, 246 - rs.size))
        gate = clipped_sigmoid_gate(rs, sm)  # adaptive gated fusion per BNA region
        fused = gate * rs + (1.0 - gate) * sm
        cols = [fused, rs, sm]
        if fc_strength is not None and i < len(fc_strength):
            cols.append(np.asarray(fc_strength[i], float).ravel()[:246])
        H = np.column_stack(cols)
        H = (H - H.mean(0)) / np.clip(H.std(0), 1e-6, None)
        vec, *shp = ss_intergnn_subject(H, A_anat, rng)
        Xg[i] = vec
        last_shapes = shp
        if (i + 1) % 100 == 0 or i == 0:
            print("  SS-InterGNN subject", i + 1, "/", n, "readout", vec.shape, "shapes", shp)
    np.save(ck, Xg)
    meta = {
        "Xg_shape": list(Xg.shape),
        "last_shapes": [list(s) for s in last_shapes] if last_shapes else None,
        "fc_path": fc_path,
        "module_status": MODULE_STATUS,
        "hidden": GRAPH_HIDDEN,
        "knn": GRAPH_KNN,
        "torch_layers": bool(torch_ok),
        "smri_dim": int(np.asarray(Xs).shape[1]) if np.asarray(Xs).ndim == 2 else -1,
        "n_regions": 246,
        "fusion": "region_gate_bna246_thickness",
    }
    meta_p.write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")
    print("SS-InterGNN readout", Xg.shape, "cached", ck, "fc_path", fc_path)
    return Xg, meta


def _force_skip_gat_cheb():
    _set_status("GAT_spatial", "SKIPPED", "not used; SpatialMix is the spatial box (Tang GAT not claimed)")
    _set_status("ChebConv_spectral", "SKIPPED", "not used; SpectralMix is the spectral box (Tang ChebConv not claimed)")


def print_novelty_tree():
    print(
        """
===== NOVELTY TREE (Sai; SpatialMix | SpectralMix — not Tang GAT/ChebConv) =====
rs-fMRI + sMRI
├─ 1 Functional connectivity     BNA-246 Pearson/Fisher-z or mean-T + anatomical adj
├─ 2 Multimodal encoding         adaptive gated fusion per BNA region (rs vs sMRI)
├─ 3 Subject-adaptive graph      kNN(k=8) + anatomical edges, row-normalized
│     ├─ SpatialMix              2-head spatial mix (not GAT)
│     └─ SpectralMix             Chebyshev filters (not ChebConv)
│           └─ SS-InterGNN = 0.5*(SpatialMix multi-head + SpectralMix Chebyshev)
├─ 4 Adaptive coarsening         246 → 48 gyrus (Brainnetome map)
├─ 5 NeuroSubnet-Pool            48 → Yeo-7 (fixed assignment)
├─ 6 Multiscale readout          concat(246-pool, 48-pool, 7-pool)
├─ 7 Site-robust ComBat          train-only; batch=site; no site ID / ADOS
└─ 8 KAN classifier              YHUA three experts as fallback
     GAT_spatial SKIPPED | ChebConv_spectral SKIPPED
"""
    )


def _novelty_ok(name, skipped_means_true=False):
    st = str((MODULE_STATUS.get(name) or {}).get("status") or "SKIPPED")
    if skipped_means_true:
        return st == "SKIPPED"
    return st.startswith("IMPLEMENTED")


def print_module_checklist():
    print_novelty_tree()
    print("N_REGIONS", int(globals().get("N_REGIONS", 246)), "atlas", globals().get("ATLAS_NAME", "Brainnetome"))
    bna_src = (globals().get("ATLAS_PACK") or {}).get("bna_source", "not loaded")
    print("parcellation", bna_src, "| FALLBACK_BNA only if download fails; never AAL")
    print("===== MODULE CHECKLIST True/False (Sai tree; working atlas=Brainnetome-246) =====")
    items = [
        ("rs-fMRI FC", "FunctionalConnectivity", False),
        ("s-MRI morphological features", "MorphologicalFeatures", False),
        ("Multimodal encoding: adaptive gated fusion per region", "MultimodalEncoding", False),
        ("Subject-adaptive graph, 246 Brainnetome regions", "SubjectAdaptiveGraph", False),
        ("SS-InterGNN 0.5*(SpatialMix multi-head + SpectralMix Chebyshev)", "SS_InterGNN", False),
        ("GAT_spatial SKIPPED", "GAT_spatial", True),
        ("ChebConv_spectral SKIPPED", "ChebConv_spectral", True),
        ("Adaptive graph coarsening 48 gyrus nodes", "AdaptiveCoarsening_246to48", False),
        ("Neuro-Subnet pool 7 Yeo networks", "NeuroSubnetPool_48toYeo7", False),
        ("Multiscale readout regions + gyrus + networks", "MultiscaleReadout", False),
        ("Site-robust ComBat fitted on training folds only", "SiteRobustComBat", False),
        ("KAN classifier ASD/TC", "KAN_classifier", False),
    ]
    atlas_true = int(globals().get("N_REGIONS", 246)) == 246
    bun = globals().get("bundle") or {}
    atlas_name = str(bun.get("atlas") or globals().get("ATLAS_NAME") or "Brainnetome")
    atlas_true = atlas_true and ("AAL" not in atlas_name.upper()) and ("SCHAEFER" not in atlas_name.upper())
    print(f"  {'atlas Brainnetome 246 (not AAL)':56s} {atlas_true}")
    for label, key, skip_ok in items:
        ok = _novelty_ok(key, skipped_means_true=skip_ok)
        st = MODULE_STATUS.get(key) or {"status": "SKIPPED", "note": "not run yet"}
        print(f"  {label:56s} {ok}    [{st.get('status')}] {st.get('note','')}")
    print("===== MODULE STATUS (raw) =====")
    order = [
        "FunctionalConnectivity", "MorphologicalFeatures", "MultimodalEncoding", "SubjectAdaptiveGraph",
        "SpatialMix", "SpectralMix", "GAT_spatial", "ChebConv_spectral", "SS_InterGNN",
        "AdaptiveCoarsening_246to48", "NeuroSubnetPool_48toYeo7",
        "MultiscaleReadout", "SiteRobustComBat", "YHUA_three_experts",
        "KAN_classifier", "torch",
    ]
    for name in order:
        st = MODULE_STATUS.get(name) or {"status": "SKIPPED", "note": "not run yet"}
        print(f"  {name:32s} {st.get('status'):20s} {st.get('note','')}")


class TrainCombat:
    """Parametric ComBat-lite: location/scale fit on training subjects only."""

    def fit(self, X, batch, covars=None):
        X = np.asarray(X, float)
        batch = np.asarray(batch).ravel()
        n, p = X.shape
        if covars is None:
            C = np.ones((n, 1))
        else:
            covars = np.asarray(covars, float)
            if covars.ndim == 1:
                covars = covars[:, None]
            C = np.column_stack([np.ones(n), covars])
        self.beta_ = np.linalg.pinv(C) @ X
        R = X - C @ self.beta_
        self.grand_mean_ = R.mean(0)
        R0 = R - self.grand_mean_
        self.gamma_ = {}
        self.delta_ = {}
        for b in np.unique(batch):
            m = batch == b
            self.gamma_[b] = R0[m].mean(0)
            self.delta_[b] = np.clip(R0[m].std(0), 1e-6, None)
        self.pooled_std_ = np.clip(R0.std(0), 1e-6, None)
        self.n_cov_ = C.shape[1]
        self.train_only_ = True
        return self

    def transform(self, X, batch, covars=None):
        X = np.asarray(X, float)
        batch = np.asarray(batch).ravel()
        n = X.shape[0]
        if covars is None:
            C = np.ones((n, 1))
        else:
            covars = np.asarray(covars, float)
            if covars.ndim == 1:
                covars = covars[:, None]
            C = np.column_stack([np.ones(n), covars])
        q = int(self.n_cov_)
        if C.shape[1] != q:
            C2 = np.ones((n, q))
            C2[:, : min(q, C.shape[1])] = C[:, : min(q, C.shape[1])]
            C = C2
        R0 = X - C @ self.beta_ - self.grand_mean_
        out = np.zeros_like(R0)
        for i in range(n):
            b = batch[i]
            if b in self.gamma_:
                g, d = self.gamma_[b], self.delta_[b]
            else:
                g, d = 0.0, self.pooled_std_
            out[i] = (R0[i] - g) / d * self.pooled_std_ + self.grand_mean_
        return out + C @ self.beta_


def fisher_z(C):
    C = np.clip(np.asarray(C, float), -0.999999, 0.999999)
    Z = np.arctanh(C)
    np.fill_diagonal(Z, 0.0)
    return Z


def fc_from_timeseries(ts):
    ts = np.asarray(ts, float)
    if ts.ndim == 1:
        ts = ts[:, None]
    x = ts - ts.mean(0, keepdims=True)
    std = np.clip(x.std(0, keepdims=True), 1e-8, None)
    z = x / std
    C = (z.T @ z) / max(z.shape[0] - 1, 1)
    return fisher_z(C)


def _novelty_dirs():
    dirs = []
    out0 = globals().get("OUTPUT_DIR") or Path("/content/yeo_neuro_outputs")
    for d in (
        Path(out0) / "novelty_outputs",
        Path("/content/yeo_neuro_outputs") / "novelty_outputs",
        Path("/content/drive/MyDrive/Project/outputs") / "novelty_outputs",
    ):
        try:
            d.mkdir(parents=True, exist_ok=True)
            dirs.append(d)
        except OSError:
            continue
    if not dirs:
        d = Path.cwd() / "novelty_outputs"
        d.mkdir(parents=True, exist_ok=True)
        dirs.append(d)
    # unique
    seen, uniq = set(), []
    for d in dirs:
        s = str(d)
        if s not in seen:
            seen.add(s)
            uniq.append(d)
    return uniq


def _write_all(dirs, name, data=None, png_fn=None, csv_df=None):
    paths = []
    for d in dirs:
        if data is not None:
            p = d / name
            if str(name).endswith(".npz"):
                np.savez_compressed(p, **data)
            elif str(name).endswith(".npy"):
                np.save(p, data)
            elif str(name).endswith(".json"):
                p.write_text(json.dumps(data, indent=2, default=str), encoding="utf-8")
            else:
                p.write_text(str(data), encoding="utf-8")
            paths.append(p)
            print("wrote", p)
        if csv_df is not None:
            p = d / (name if str(name).endswith(".csv") else str(name) + ".csv")
            csv_df.to_csv(p, index=False)
            paths.append(p)
            print("wrote", p)
        if png_fn is not None and str(name).endswith(".png"):
            p = d / name
            png_fn(p)
            paths.append(p)
            print("wrote", p)
    return paths


# Fan 2016 left-hemisphere MNI centroids (mm) for 24 gyri; R = (-x,y,z). FALLBACK_BNA coords only.
BNA_GYRUS24_MNI_L = np.array(
    [
        [-16.0, 40.0, 44.0],
        [-32.0, 28.0, 38.0],
        [-44.0, 22.0, 12.0],
        [-26.0, 36.0, -16.0],
        [-38.0, -14.0, 52.0],
        [-8.0, -28.0, 64.0],
        [-52.0, -10.0, 4.0],
        [-56.0, -28.0, -8.0],
        [-50.0, -32.0, -22.0],
        [-32.0, -40.0, -18.0],
        [-24.0, -20.0, -22.0],
        [-50.0, -42.0, 10.0],
        [-24.0, -60.0, 54.0],
        [-42.0, -50.0, 40.0],
        [-8.0, -64.0, 40.0],
        [-42.0, -28.0, 50.0],
        [-36.0, 6.0, 4.0],
        [-6.0, 10.0, 34.0],
        [-10.0, -78.0, 8.0],
        [-32.0, -84.0, 6.0],
        [-24.0, -4.0, -18.0],
        [-26.0, -20.0, -14.0],
        [-18.0, 8.0, 2.0],
        [-10.0, -18.0, 8.0],
    ],
    dtype=float,
)


def fallback_bna_mni_coords() -> np.ndarray:
    """Deterministic 246×3 MNI mm from Fan gyrus centroids. Never AAL, never oval layout."""
    counts = np.asarray(globals().get("BNA_GYRUS_COUNTS_HEMI"), int)
    xyz = np.zeros((246, 3), float)
    starts = np.cumsum(np.r_[0, counts[:-1]])
    rng = np.random.default_rng(2016)
    for g, n in enumerate(counts):
        cL = BNA_GYRUS24_MNI_L[g]
        for k in range(int(n)):
            sub = int(starts[g] + k)
            jitter = rng.normal(0.0, 5.5, size=3)
            pL = cL + jitter
            pL[0] = -abs(float(pL[0]))
            xyz[2 * sub] = pL
            xyz[2 * sub + 1] = [-pL[0], pL[1], pL[2]]
    return xyz


def bna_mni_coords_from_pack(pack) -> np.ndarray | None:
    pack = pack or {}
    data = pack.get("bna_data")
    img = pack.get("bna_img")
    if data is None or img is None:
        return None
    try:
        import nibabel as nib

        aff = img.affine
        xyz = np.zeros((246, 3), float)
        ok = 0
        for r in range(1, 247):
            m = np.asarray(data) == r
            if not np.any(m):
                continue
            ijk = np.column_stack(np.nonzero(m))
            com = ijk.mean(0)
            xyz[r - 1] = nib.affines.apply_affine(aff, com)
            ok += 1
        if ok < 200:
            return None
        return xyz
    except Exception as exc:
        print("BNA NIfTI centroid failed", exc)
        return None


def get_bna_mni_coords():
    pack = globals().get("ATLAS_PACK") or {}
    xyz = bna_mni_coords_from_pack(pack)
    if xyz is not None:
        used = pack.get("bna_path") or pack.get("bna_source")
        print("atlas file used:", used)
        print("BNA MNI coords: NIfTI centers-of-mass", pack.get("bna_source"), "shape", xyz.shape)
        return np.asarray(xyz, float), "OFFICIAL_BNA_NIFTI_COM"
    finder = globals().get("find_licensed_bna_nifti")
    if callable(finder):
        loc = finder()
        if loc is not None:
            try:
                import nibabel as nib
                img = nib.load(str(loc))
                data = np.asanyarray(img.dataobj)
                tmp = {"bna_img": img, "bna_data": data}
                xyz = bna_mni_coords_from_pack(tmp)
                if xyz is not None:
                    print("atlas file used:", loc)
                    print("BNA MNI coords: licensed NIfTI centers-of-mass", loc, "shape", xyz.shape)
                    return np.asarray(xyz, float), f"LICENSED_BNA_NIFTI_COM:{loc}"
            except Exception as exc:
                print("licensed BNA coords failed", loc, exc)
    xyz = fallback_bna_mni_coords()
    print("atlas file used: FALLBACK_BNA coords only")
    print("BNA MNI coords: FALLBACK_BNA Fan-2016 gyrus centroids (mm); never AAL, never oval schematic")
    return xyz, "FALLBACK_BNA_MNI"


def gyrus48_mni_coords(xyz246, g48) -> np.ndarray:
    xyz246 = np.asarray(xyz246, float)
    g48 = np.asarray(g48, int)
    out = np.zeros((48, 3), float)
    for g in range(48):
        m = g48[: xyz246.shape[0]] == g
        if np.any(m):
            out[g] = xyz246[m].mean(0)
    return out


def yeo7_mni_coords(xyz246, yeo_ids) -> np.ndarray:
    xyz246 = np.asarray(xyz246, float)
    yeo_ids = np.asarray(yeo_ids, int).ravel()
    out = np.zeros((7, 3), float)
    for y in range(7):
        m = yeo_ids[: xyz246.shape[0]] == y
        if np.any(m):
            out[y] = xyz246[m].mean(0)
    return out


def sparsify_adj(W, edge_keep=80):
    W = np.asarray(W, float).copy()
    np.fill_diagonal(W, 0.0)
    W = 0.5 * (W + W.T)
    iu = np.triu_indices(W.shape[0], 1)
    vals = np.abs(W[iu])
    if vals.size == 0:
        return W
    k = int(min(max(edge_keep, 1), vals.size))
    thr = np.partition(vals, -k)[-k]
    out = np.where(np.abs(W) >= max(thr, 1e-12), W, 0.0)
    np.fill_diagonal(out, 0.0)
    return out


BNA_NIFTI_MIRRORS = [
    "https://afni.nimh.nih.gov/pub/dist/atlases/afni_atlases_dist/BN_Atlas_246_1mm.nii.gz",
    "https://afni-dev.nimh.nih.gov/pub/dist/atlases/afni_atlases_dist/BN_Atlas_246_1mm.nii.gz",
]


def _mni152_t1(resolution=1):
    from nilearn.datasets import load_mni152_template

    return load_mni152_template(resolution=resolution)


def fetch_bna_label_img():
    """Return (nibabel img of BNA-246 labels, source). Licensed neuroimaging first; then AFNI/HTTPS; then FALLBACK_BNA."""
    import nibabel as nib

    pack = globals().get("ATLAS_PACK") or {}
    if pack.get("bna_img") is not None and pack.get("bna_data") is not None:
        labs = set(np.unique(np.asarray(pack["bna_data"]).astype(int))) - {0}
        if len(labs) >= 200:
            used = pack.get("bna_path") or pack.get("bna_source")
            print("atlas file used:", used)
            print("BNA NIfTI from ATLAS_PACK", pack.get("bna_source"))
            return pack["bna_img"], str(pack.get("bna_source") or "OFFICIAL_BNA")
    finder = globals().get("find_licensed_bna_nifti")
    if callable(finder):
        loc = finder()
        if loc is not None:
            try:
                img = nib.load(str(loc))
                data = np.asanyarray(img.dataobj)
                if len(set(np.unique(data.astype(int))) - {0}) >= 200:
                    print("atlas file used:", loc)
                    return img, f"LICENSED:{loc}"
            except Exception as exc:
                print("licensed BNA unreadable", loc, exc)
    dests = [
        Path("/tmp/bna/BN_Atlas_246_1mm.nii.gz"),
        Path("/content/atlas_cache/BN_Atlas_246_1mm.nii.gz"),
        Path("/content/atlas_cache/BNA_MPM_thr25_1.25mm.nii.gz"),
    ]
    urls = list(BNA_NIFTI_MIRRORS) + list(globals().get("BNA_ATLAS_URLS") or [])
    for d in dests:
        if d.exists() and d.stat().st_size > 10000:
            try:
                img = nib.load(str(d))
                data = np.asanyarray(img.dataobj)
                if len(set(np.unique(data.astype(int))) - {0}) >= 200:
                    print("BNA NIfTI cache", d, "shape", data.shape)
                    return img, f"cached:{d}"
            except Exception as exc:
                print("BNA cache unreadable", d, exc)
    cache = Path("/content/atlas_cache")
    try:
        cache.mkdir(parents=True, exist_ok=True)
        dest = cache / "BN_Atlas_246_1mm.nii.gz"
    except OSError:
        dest = Path("/tmp/bna")
        dest.mkdir(parents=True, exist_ok=True)
        dest = dest / "BN_Atlas_246_1mm.nii.gz"
    dl = globals().get("_http_download")
    for u in urls:
        ok = False
        if callable(dl):
            ok = bool(dl(u, dest))
        else:
            try:
                import urllib.request

                req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=90) as r:
                    dest.write_bytes(r.read())
                ok = dest.exists() and dest.stat().st_size > 10000
            except Exception as exc:
                print("BNA download fail", u, exc)
                ok = False
        if not ok:
            continue
        try:
            img = nib.load(str(dest))
            data = np.asanyarray(img.dataobj)
            if len(set(np.unique(data.astype(int))) - {0}) >= 200:
                    print("atlas file used:", dest, "from", u)
                    print("BNA NIfTI downloaded", u, "shape", data.shape)
                    return img, u
        except Exception as exc:
            print("BNA nifti load fail", dest, exc)
    print("atlas file used: FALLBACK_BNA (no licensed neuroimaging NIfTI and AFNI/HTTPS download failed)")
    print("FALLBACK: nearest-centroid BNA labels stamped onto MNI152 T1 (underlay is still real MRI)")
    return _fallback_bna_labels_on_mni(), "FALLBACK_BNA_ON_MNI152"


def _fallback_bna_labels_on_mni():
    import nibabel as nib

    bg = _mni152_t1(resolution=2)
    xyz = fallback_bna_mni_coords()
    data = np.asanyarray(bg.dataobj)
    mask = np.asarray(data) > np.percentile(np.asarray(data)[np.asarray(data) > 0], 15)
    ijk = np.column_stack(np.nonzero(mask))
    world = nib.affines.apply_affine(bg.affine, ijk)
    labels = np.zeros(bg.shape, np.int16)
    # chunked nearest centroid
    step = 40000
    for s in range(0, world.shape[0], step):
        w = world[s : s + step]
        d2 = ((w[:, None, :] - xyz[None, :, :]) ** 2).sum(-1)
        lab = d2.argmin(1) + 1
        ii = ijk[s : s + step]
        labels[ii[:, 0], ii[:, 1], ii[:, 2]] = lab
    return nib.Nifti1Image(labels, bg.affine)


def remap_label_img(bna_img, mapping_246, n_out):
    import nibabel as nib

    data = np.asanyarray(bna_img.dataobj).astype(np.int32)
    mapping_246 = np.asarray(mapping_246, int)
    out = np.zeros(data.shape, np.int16)
    for r in range(1, 247):
        g = int(mapping_246[r - 1]) if r - 1 < len(mapping_246) else 0
        if 0 <= g < n_out:
            out[data == r] = g + 1
    return nib.Nifti1Image(out, bna_img.affine, getattr(bna_img, "header", None))


def nilearn_plot_roi_on_t1(path, roi_img, title, cmap="nipy_spectral"):
    """Paint integer parcels on gray MNI152 T1 (plot_roi). Never glass-brain outlines."""
    from nilearn.plotting import plot_roi
    from nilearn.image import resample_to_img

    bg = _mni152_t1(resolution=1)
    roi = resample_to_img(roi_img, bg, interpolation="nearest", copy_header=True)
    plot_roi(
        roi,
        bg_img=bg,
        display_mode="tiled",
        title=title,
        output_file=str(path),
        colorbar=True,
        alpha=0.55,
        cmap=cmap,
        draw_cross=False,
        black_bg=False,
        view_type="continuous",
        annotate=True,
    )
    print("nilearn.plotting.plot_roi on load_mni152_template", path, "display_mode=tiled underlay=MNI152 T1")


def nilearn_plot_connectome_file(path, adj, coords, title, node_color=None, node_size=12, edge_keep=80):
    nilearn_plot_nodes_three_views(path, coords, title, node_color=node_color or "#2cb5b8", node_size=node_size)


def nilearn_plot_nodes_three_views(path, coords, title, node_color="#2cb5b8", node_size=12):
    """Teal nodes on 3-view MNI glass brain (sagittal/coronal/axial). Zero adjacency: no edges."""
    from nilearn.plotting import plot_connectome
    import matplotlib.pyplot as plt

    coords = np.asarray(coords, float)
    n = int(coords.shape[0])
    adj = np.zeros((n, n), float)
    fig = plt.figure(figsize=(14.2, 4.15), facecolor="white")
    fig.text(
        0.012,
        0.93,
        title,
        fontsize=9,
        color="white",
        fontweight="bold",
        bbox={"facecolor": "#111111", "edgecolor": "none", "pad": 4.5},
        ha="left",
        va="top",
    )
    for i, mode in enumerate(("x", "y", "z")):
        ax = fig.add_axes([0.01 + i * 0.33, 0.02, 0.32, 0.82])
        plot_connectome(
            adj,
            coords,
            node_color=node_color,
            node_size=node_size,
            display_mode=mode,
            axes=ax,
            colorbar=False,
            title=None,
            annotate=True,
            black_bg=False,
        )
    fig.savefig(str(path), dpi=160, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("nilearn.plotting.plot_connectome nodes-only x,y,z", path, "N=", n, "teal glass brain")


def nilearn_plot_yeo_glass_file(path, coords7, values7, title):
    """Yeo-7 on MNI glass brain: Gaussian blobs + 7-node plot_connectome overlay file."""
    from nilearn.plotting import plot_glass_brain, plot_connectome
    from nilearn.datasets import load_mni152_template
    import nibabel as nib

    coords7 = np.asarray(coords7, float)
    values7 = np.asarray(values7, float)
    values7 = values7 / np.clip(np.max(np.abs(values7)), 1e-6, None)
    tmpl = load_mni152_template(resolution=2)
    data = np.zeros(tmpl.shape, np.float32)
    aff = tmpl.affine
    vx = np.sqrt((aff[:3, :3] ** 2).sum(0))
    sig = np.maximum(10.0 / np.maximum(vx, 1e-3), 1.0)
    inv = np.linalg.inv(aff)
    for y in range(min(7, len(coords7))):
        ijk = nib.affines.apply_affine(inv, coords7[y])
        rad = np.ceil(3 * sig).astype(int)
        i0 = np.round(ijk).astype(int)
        for i in range(max(0, i0[0] - rad[0]), min(data.shape[0], i0[0] + rad[0] + 1)):
            for j in range(max(0, i0[1] - rad[1]), min(data.shape[1], i0[1] + rad[1] + 1)):
                for k in range(max(0, i0[2] - rad[2]), min(data.shape[2], i0[2] + rad[2] + 1)):
                    d2 = ((i - ijk[0]) / sig[0]) ** 2 + ((j - ijk[1]) / sig[1]) ** 2 + ((k - ijk[2]) / sig[2]) ** 2
                    data[i, j, k] += float(values7[y]) * np.exp(-0.5 * d2)
    img = nib.Nifti1Image(data, aff)
    plot_glass_brain(
        img,
        display_mode="lyrz",
        colorbar=True,
        title=title,
        plot_abs=False,
        output_file=str(path),
        cmap="plasma",
    )
    print("nilearn.plotting.plot_glass_brain", path, "display_mode=lyrz Yeo-7 Gaussians on MNI152 (not AAL)")


def dump_novelty_outputs():
    """One artifact per novelty + explicit FC / BNA-246 / gyrus-48 / Yeo-7 / KAN visuals. Never AAL."""
    import matplotlib.pyplot as plt

    dirs = _novelty_dirs()
    print("===== NOVELTY OUTPUT DUMP (Brainnetome-246; never AAL) =====")
    print("N_REGIONS", int(globals().get("N_REGIONS", 246)), "atlas", globals().get("ATLAS_NAME", "Brainnetome"))
    bundle = globals().get("bundle") or {}
    Xbna = np.asarray(bundle.get("X_bna") if bundle.get("X_bna") is not None else globals().get("Xbna"), float) if (bundle.get("X_bna") is not None or globals().get("Xbna") is not None) else None
    if Xbna is None:
        Xbna = np.asarray(globals().get("Xf"), float) if globals().get("Xf") is not None else None
    Xs = np.asarray(bundle.get("Xs") if bundle.get("Xs") is not None else globals().get("Xs"), float) if (bundle.get("Xs") is not None or globals().get("Xs") is not None) else None
    Xgy = bundle.get("X_gyrus")
    Xye = bundle.get("X_yeo")
    Xg = globals().get("Xg")
    y = globals().get("y")
    site = globals().get("site")
    i_tr = globals().get("i_tr")
    n_reg = int(globals().get("N_REGIONS", 246))
    g48 = np.asarray(globals().get("BNA_TO_GYRUS48"), int)
    pack = globals().get("ATLAS_PACK") or {}
    b2y = pack.get("bna_to_yeo")
    if b2y is None:
        b2y = np.array([int(FALLBACK_GYRUS24_YEO[g % 24]) for g in g48], dtype=int)

    # 0 atlas
    atlas_info = {
        "N_REGIONS": n_reg,
        "atlas": "Brainnetome",
        "bna_source": pack.get("bna_source"),
        "yeo_source": pack.get("yeo_source"),
        "FALLBACK_BNA_only_if_download_fails": True,
        "working_atlas_is_AAL": False,
        "bundle_atlas": bundle.get("atlas"),
        "source": bundle.get("source"),
        "note": "SYNTHETIC_DEMO is not a Drive result",
        "model_selection": "best_val_metric",
    }
    _write_all(dirs, "00_atlas_brainnetome.json", data=atlas_info)

    # 1 rs-fMRI FC heatmap (246x246 from node-feature correlations; Fisher-z analogue)
    if Xbna is not None and Xbna.ndim == 2 and Xbna.shape[1] >= 2:
        Xn = Xbna[:, :n_reg]
        if Xn.shape[1] < n_reg:
            pad = np.zeros((Xn.shape[0], n_reg - Xn.shape[1]))
            Xn = np.hstack([Xn, pad])
        Z = Xn - Xn.mean(0, keepdims=True)
        sd = np.clip(Z.std(0, keepdims=True), 1e-8, None)
        Zn = Z / sd
        FC = (Zn.T @ Zn) / max(Zn.shape[0] - 1, 1)
        FC = np.clip(FC, -1.0, 1.0)
        np.fill_diagonal(FC, 0.0)
        _write_all(dirs, "01_rs_fmri_fc.npz", data={"FC": FC, "X_bna": Xn, "n_regions": n_reg, "atlas": "Brainnetome"})
        pd.DataFrame({"region": np.arange(1, n_reg + 1), "node_strength": FC.sum(1)}).to_csv(dirs[0] / "01_rs_fmri_fc_strength.csv", index=False)
        print("wrote", dirs[0] / "01_rs_fmri_fc_strength.csv")
        for d in dirs[1:]:
            (d / "01_rs_fmri_fc_strength.csv").write_text((dirs[0] / "01_rs_fmri_fc_strength.csv").read_text())
            print("wrote", d / "01_rs_fmri_fc_strength.csv")

        def _png_fc(p):
            fig, ax = plt.subplots(figsize=(6.2, 5.4))
            im = ax.imshow(FC, cmap="coolwarm", vmin=-0.6, vmax=0.6, interpolation="nearest")
            ax.set_title("Functional connectivity (Brainnetome-246)")
            ax.set_xlabel("BNA region")
            ax.set_ylabel("BNA region")
            fig.colorbar(im, ax=ax, fraction=0.046)
            fig.tight_layout()
            fig.savefig(p, dpi=140, bbox_inches="tight")
            plt.close(fig)

        _write_all(dirs, "01_rs_fmri_fc.png", png_fn=_png_fc)
        _set_status("FunctionalConnectivity", "IMPLEMENTED_LIGHT", f"BNA-{n_reg} FC heatmap dumped; not AAL")
    else:
        print("FC dump skipped: no X_bna")

    # 2 sMRI
    if Xs is not None:
        _write_all(dirs, "02_smri_morphology.npz", data={"Xs": np.asarray(Xs, float), "n_regions": int(np.asarray(Xs).shape[1]), "atlas": "Brainnetome-246"})
        mu = np.asarray(Xs, float).mean(0)
        pd.DataFrame({"bna_region": np.arange(1, mu.size + 1), "mean_thickness": mu}).to_csv(dirs[0] / "02_smri_morphology.csv", index=False)
        print("wrote", dirs[0] / "02_smri_morphology.csv")
        for d in dirs[1:]:
            (d / "02_smri_morphology.csv").write_text((dirs[0] / "02_smri_morphology.csv").read_text())
            print("wrote", d / "02_smri_morphology.csv")

        def _png_sm(p):
            fig, ax = plt.subplots(figsize=(7.2, 3.2))
            ax.bar(np.arange(mu.size), mu, color="#4c78a8")
            ax.set_title("s-MRI morphological features (BNA-246 mean thickness)")
            ax.set_xlabel("BNA region")
            fig.tight_layout()
            fig.savefig(p, dpi=140, bbox_inches="tight")
            plt.close(fig)

        _write_all(dirs, "02_smri_morphology.png", png_fn=_png_sm)
        _set_status("MorphologicalFeatures", "IMPLEMENTED_LIGHT", "s-MRI BNA-246 volumetric mask mean thickness")

    # 3 gated fusion
    if Xbna is not None and Xs is not None:
        rs = np.asarray(Xbna[0], float).ravel()
        if rs.size < n_reg:
            rs = np.pad(rs, (0, n_reg - rs.size))
        rs = rs[:n_reg]
        sm = np.asarray(smri_nodes_246(Xs[0]), float).ravel()[:n_reg]
        if sm.size < n_reg:
            sm = np.pad(sm, (0, n_reg - sm.size))
        gate = clipped_sigmoid_gate(rs, sm)
        fused = gate * rs + (1.0 - gate) * sm
        _write_all(dirs, "03_gated_fusion.npz", data={"gate": gate, "fused": fused, "rs": rs, "smri": sm})

        def _png_gate(p):
            fig, ax = plt.subplots(figsize=(7.2, 3.2))
            ax.plot(gate, lw=0.8)
            ax.set_ylim(0, 1)
            ax.set_title("Adaptive gated fusion per BNA region (subject 0)")
            ax.set_xlabel("BNA-246 region")
            fig.tight_layout()
            fig.savefig(p, dpi=140, bbox_inches="tight")
            plt.close(fig)

        _write_all(dirs, "03_gated_fusion.png", png_fn=_png_gate)
        _set_status("MultimodalEncoding", "IMPLEMENTED_LIGHT", "adaptive gated fusion per region dumped")

    # 4 BNA-246 node map/count  + subject-adaptive graph
    counts = np.bincount(g48, minlength=48)
    bna_csv = pd.DataFrame({
        "bna_region": np.arange(1, n_reg + 1),
        "gyrus48": g48[:n_reg],
        "yeo7": np.asarray(b2y, int)[:n_reg],
    })
    for d in dirs:
        p = d / "04_bna246_nodes.csv"
        bna_csv.to_csv(p, index=False)
        print("wrote", p)
    _write_all(dirs, "04_bna246_nodes.npz", data={"BNA_TO_GYRUS48": g48, "bna_to_yeo": np.asarray(b2y, int), "n_regions": n_reg})

    def _png_bna(p):
        fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4))
        axes[0].hist(g48, bins=np.arange(49) - 0.5, color="#59a14f")
        axes[0].set_title("BNA-246 regions per gyrus-48")
        axes[0].set_xlabel("gyrus")
        yeo_c = np.bincount(np.asarray(b2y, int)[:n_reg], minlength=7)
        axes[1].bar(np.arange(7), yeo_c, color="#e15759")
        axes[1].set_title("BNA-246 count per Yeo-7")
        axes[1].set_xticks(range(7))
        axes[1].set_xticklabels(list(globals().get("YEO_NAMES") or list(range(7))), rotation=45, ha="right", fontsize=8)
        fig.suptitle("Brainnetome-246 node map (not AAL)")
        fig.tight_layout()
        fig.savefig(p, dpi=140, bbox_inches="tight")
        plt.close(fig)

    _write_all(dirs, "04_bna246_nodes.png", png_fn=_png_bna)

    A246 = None
    if Xbna is not None and Xs is not None:
        n_use = min(24, Xbna.shape[0])
        A_anat = anatomical_adj_bna246()
        acc = np.zeros((246, 246), float)
        for i in range(n_use):
            xb = np.zeros(246, float)
            xb[: min(246, Xbna.shape[1])] = np.asarray(Xbna[i], float).ravel()[:246]
            sm = np.asarray(smri_nodes_246(Xs[i]), float).ravel()[:246]
            H = np.column_stack([xb, sm])
            H = (H - H.mean(0)) / np.clip(H.std(0), 1e-6, None)
            Aknn = knn_from_features(H, k=int(globals().get("GRAPH_KNN", 8)))
            A = ((A_anat + Aknn) > 0).astype(float)
            np.fill_diagonal(A, 0.0)
            acc += A
        A246 = acc / max(n_use, 1)
        _write_all(dirs, "04_subject_adaptive_graph.npz", data={"A246": A246, "n_subjects_mean": n_use, "atlas": "Brainnetome-246"})
        _set_status("SubjectAdaptiveGraph", "IMPLEMENTED_LIGHT", "BNA-246 kNN+anatomical graph dumped")

        def _png_g(p):
            fig, ax = plt.subplots(figsize=(5.6, 5.0))
            ax.imshow(A246, cmap="viridis", interpolation="nearest")
            ax.set_title("Subject-adaptive graph (mean adj, BNA-246)")
            fig.tight_layout()
            fig.savefig(p, dpi=140, bbox_inches="tight")
            plt.close(fig)

        _write_all(dirs, "04_subject_adaptive_graph.png", png_fn=_png_g)

    # 5 SS-InterGNN one subject
    H246 = H48 = H7 = readout = None
    if Xbna is not None and Xs is not None:
        rng = np.random.default_rng(int(globals().get("SEED", 42)))
        rs = np.asarray(Xbna[0], float).ravel()
        if rs.size < 246:
            rs = np.pad(rs, (0, 246 - rs.size))
        rs = rs[:246]
        sm = np.asarray(smri_nodes_246(Xs[0]), float).ravel()[:246]
        gate = clipped_sigmoid_gate(rs, sm)
        fused = gate * rs + (1.0 - gate) * sm
        H = np.column_stack([fused, rs, sm])
        H = (H - H.mean(0)) / np.clip(H.std(0), 1e-6, None)
        A_anat = anatomical_adj_bna246()
        readout, shpA, shp246, shp48, shp7 = ss_intergnn_subject(H, A_anat, rng)
        # reconstruct H246/H48/H7 via same path internals
        Aknn = knn_from_features(H, k=int(globals().get("GRAPH_KNN", 8)))
        A = row_normalize(((A_anat + Aknn) > 0).astype(float))
        np.fill_diagonal(A, 1.0)
        A = row_normalize(A)
        H_sp = 0.5 * (spatial_mix(H, A, rng) + spatial_mix(H, A, rng))
        H_spec = spectral_mix(H, A, rng)
        H246 = 0.5 * (H_sp + H_spec)
        H48 = coarsen_246_to_48(H246)
        H7 = pool_48_to_yeo7(H48)
        _write_all(
            dirs,
            "05_ss_intergnn.npz",
            data={
                "readout": readout,
                "H246": H246,
                "H48": H48,
                "H7": H7,
                "shapes": [list(shpA), list(shp246), list(shp48), list(shp7)],
                "GAT_spatial": "SKIPPED",
                "ChebConv_spectral": "SKIPPED",
            },
        )
        _force_skip_gat_cheb()
        _set_status("SS_InterGNN", "IMPLEMENTED_LIGHT", "0.5*(SpatialMix 2-head + SpectralMix Chebyshev)")
        _set_status("SpatialMix", "IMPLEMENTED_LIGHT", "2-head spatial mix; GAT SKIPPED")
        _set_status("SpectralMix", "IMPLEMENTED_LIGHT", "Chebyshev K=2; ChebConv SKIPPED")

        def _png_ss(p):
            fig, ax = plt.subplots(figsize=(6.4, 3.2))
            ax.plot(readout, marker="o", ms=3)
            ax.set_title("SS-InterGNN multiscale readout (24-D, subject 0)")
            fig.tight_layout()
            fig.savefig(p, dpi=140, bbox_inches="tight")
            plt.close(fig)

        _write_all(dirs, "05_ss_intergnn.png", png_fn=_png_ss)

    # 6 gyrus-48 coarsened graph
    if A246 is None and Xbna is not None:
        A246 = anatomical_adj_bna246()
    if A246 is not None:
        A48 = coarsen_246_to_48_mat_local(A246, g48)
        _write_all(dirs, "06_gyrus48_coarsened.npz", data={"A48": A48, "n_gyrus": 48, "atlas": "Brainnetome gyrus-48"})
        names = list(globals().get("BNA_GYRUS_NAMES_HEMI") or [f"g{i}" for i in range(24)])
        names48 = [f"L-{n}" for n in names] + [f"R-{n}" for n in names]
        pd.DataFrame({"gyrus": names48[: A48.shape[0]], "degree": A48.sum(1)}).to_csv(dirs[0] / "06_gyrus48_coarsened.csv", index=False)
        print("wrote", dirs[0] / "06_gyrus48_coarsened.csv")
        for d in dirs[1:]:
            (d / "06_gyrus48_coarsened.csv").write_text((dirs[0] / "06_gyrus48_coarsened.csv").read_text())
            print("wrote", d / "06_gyrus48_coarsened.csv")

        def _png_48(p):
            fig, ax = plt.subplots(figsize=(5.8, 5.2))
            ax.imshow(A48, cmap="magma", interpolation="nearest")
            ax.set_title("Adaptive coarsening: gyrus-48 graph")
            fig.tight_layout()
            fig.savefig(p, dpi=140, bbox_inches="tight")
            plt.close(fig)

        _write_all(dirs, "06_gyrus48_coarsened.png", png_fn=_png_48)
        _set_status("AdaptiveCoarsening_246to48", "IMPLEMENTED_FAITHFUL", "Fan 2016 24 gyri x 2 = 48")

    # 7 Yeo-7 pool
    if Xye is not None:
        Xye_a = np.asarray(Xye, float)
    elif Xbna is not None:
        Xye_a = np.vstack([pool_groups(np.asarray(Xbna[i], float), np.asarray(b2y, int), 7) for i in range(min(len(Xbna), 200))])
    else:
        Xye_a = None
    if H7 is not None or Xye_a is not None:
        yeo_mean = np.asarray(H7).mean(1) if H7 is not None else np.asarray(Xye_a).mean(0)
        _write_all(dirs, "07_yeo7_pool.npz", data={"yeo_mean": yeo_mean, "H7": H7 if H7 is not None else yeo_mean, "n_networks": 7})
        yeo_names = list(globals().get("YEO_NAMES") or [f"Yeo{i}" for i in range(7)])
        pd.DataFrame({"network": yeo_names[: yeo_mean.size], "mean": yeo_mean}).to_csv(dirs[0] / "07_yeo7_pool.csv", index=False)
        print("wrote", dirs[0] / "07_yeo7_pool.csv")
        for d in dirs[1:]:
            (d / "07_yeo7_pool.csv").write_text((dirs[0] / "07_yeo7_pool.csv").read_text())
            print("wrote", d / "07_yeo7_pool.csv")

        def _png_yeo(p):
            fig, ax = plt.subplots(figsize=(6.2, 3.4))
            ax.bar(np.arange(yeo_mean.size), yeo_mean, color="#f28e2b")
            ax.set_xticks(range(yeo_mean.size))
            ax.set_xticklabels(yeo_names[: yeo_mean.size], rotation=30, ha="right")
            ax.set_title("Neuro-Subnet pool Yeo-7")
            fig.tight_layout()
            fig.savefig(p, dpi=140, bbox_inches="tight")
            plt.close(fig)

        _write_all(dirs, "07_yeo7_pool.png", png_fn=_png_yeo)
        _set_status("NeuroSubnetPool_48toYeo7", "IMPLEMENTED_LIGHT", "Yeo-7 pool dumped")

    # 8 multiscale
    if Xg is not None:
        Xg_a = np.asarray(Xg, float)
        _write_all(dirs, "08_multiscale_readout.npz", data={"Xg": Xg_a})

        def _png_ms(p):
            fig, ax = plt.subplots(figsize=(6.4, 3.2))
            ax.bar(np.arange(Xg_a.shape[1]), Xg_a.mean(0), color="#76b7b2")
            ax.set_title("Multiscale readout mean (regions+gyrus+networks)")
            fig.tight_layout()
            fig.savefig(p, dpi=140, bbox_inches="tight")
            plt.close(fig)

        _write_all(dirs, "08_multiscale_readout.png", png_fn=_png_ms)
        _set_status("MultiscaleReadout", "IMPLEMENTED_LIGHT", "concat 246+48+7 pool")
    elif readout is not None:
        _write_all(dirs, "08_multiscale_readout.npz", data={"readout_subject0": readout})

    # 9 ComBat train-only
    combat_meta = {"train_only": True, "batch": "site", "covariates_fit": ["diagnosis", "age", "sex"], "covariates_val_test": ["diagnosis_dummy_train_mean", "age", "sex"], "val_test_uses_true_y": False, "site_as_predictor": False, "ados_as_predictor": False}
    if i_tr is not None:
        combat_meta["n_train"] = int(len(i_tr))
    harm = globals().get("harm")
    if harm is not None:
        combat_meta["X_harm"] = harm
    _write_all(dirs, "09_combat_train_only.json", data=combat_meta)
    _set_status("SiteRobustComBat", "IMPLEMENTED_FAITHFUL", "train-only ComBat; site not a feature")

    # 10 KAN
    chosen = globals().get("chosen") or {}
    z_te = globals().get("z_te")
    pred_te = globals().get("pred_te")
    kan = globals().get("kan_head") or globals().get("kan")
    kan_pack = {
        "classifier": "KAN",
        "n_pca": chosen.get("n_pca"),
        "C": chosen.get("C"),
        "t": chosen.get("t"),
        "kan_C": chosen.get("kan_C"),
        "T": chosen.get("T"),
        "bias": chosen.get("bias"),
        "val_acc": chosen.get("val_acc"),
        "model_selection": "best_val_metric",
        "hardcoded_n_correct": False,
    }
    _write_all(dirs, "10_kan_classifier.json", data=kan_pack)
    logit = None
    if z_te is not None:
        logit = np.asarray(z_te, float).ravel()
    elif globals().get("kan_logits_te") is not None:
        logit = np.asarray(globals().get("kan_logits_te"), float).ravel()
    weights = None
    for obj in (kan, chosen):
        if obj is None:
            continue
        clf = getattr(obj, "clf_", None) if not isinstance(obj, dict) else None
        if clf is not None and hasattr(clf, "coef_"):
            weights = np.asarray(clf.coef_, float).ravel()
            break
    extra = {}
    if logit is not None:
        extra["logits"] = logit
    if weights is not None:
        extra["rbf_or_linear_weights"] = weights
    if extra:
        _write_all(dirs, "10_kan_classifier.npz", data=extra)

    def _png_kan(p):
        fig, axes = plt.subplots(1, 2 if (logit is not None and weights is not None) else 1, figsize=(8.4, 3.3))
        if not hasattr(axes, "__len__"):
            axes = [axes]
        ai = 0
        if logit is not None:
            axes[ai].hist(logit, bins=24, color="#b07aa1", edgecolor="white")
            axes[ai].set_title("KAN logits (test)")
            ai += 1
        if weights is not None and ai < len(axes):
            axes[ai].plot(weights, marker="o", ms=3, color="#4e79a7")
            axes[ai].set_title("KAN RBF-head weights")
        elif logit is None:
            axes[0].text(0.5, 0.5, "KAN locked\nclassifier=KAN\nbest val metric", ha="center", va="center")
            axes[0].set_axis_off()
        fig.suptitle("KAN classifier ASD/TC")
        fig.tight_layout()
        fig.savefig(p, dpi=140, bbox_inches="tight")
        plt.close(fig)

    _write_all(dirs, "10_kan_classifier.png", png_fn=_png_kan)
    _set_status("KAN_classifier", "IMPLEMENTED_LIGHT", "RBF-KAN head dumped")

    xyz246, coord_src = get_bna_mni_coords()
    yid = np.clip(np.asarray(b2y, int).ravel()[:246], 0, 6)
    A48v = coarsen_246_to_48_mat_local(A246, g48) if A246 is not None else None
    xyz48 = gyrus48_mni_coords(xyz246, g48)
    xyz7 = yeo7_mni_coords(xyz246, yid)
    _write_all(
        dirs,
        "11_brain_bna246.npz",
        data={"xyz_mni": xyz246, "coord_src": coord_src, "atlas": "Brainnetome-246", "nilearn": "plot_connectome nodes-only", "style": "teal glass brain 3-view"},
    )

    def _png_brain_bna(p):
        nilearn_plot_nodes_three_views(
            p,
            xyz246,
            "Brainnetome Atlas (BNA-246) Parcellation: 246 Regions across 7 Anatomical Lobes",
            node_color="#2cb5b8",
            node_size=10,
        )

    try:
        _write_all(dirs, "11_brain_bna246.png", png_fn=_png_brain_bna)
    except Exception as exc:
        print("BNA glass-brain nodes failed", exc)

    _write_all(
        dirs,
        "12_brain_gyrus48.npz",
        data={"xyz_mni": xyz48, "A48": A48v if A48v is not None else np.eye(48), "atlas": "gyrus-48", "nilearn": "plot_connectome nodes-only"},
    )

    def _png_brain_g48(p):
        nilearn_plot_nodes_three_views(
            p,
            xyz48,
            "Brainnetome gyrus-48 coarsening (48 nodes on MNI glass brain)",
            node_color="#2cb5b8",
            node_size=22,
        )

    try:
        _write_all(dirs, "12_brain_gyrus48.png", png_fn=_png_brain_g48)
    except Exception as exc:
        print("gyrus-48 glass-brain nodes failed", exc)

    yeo_val = np.ones(7)
    if Xye is not None:
        yeo_val = np.abs(np.asarray(Xye, float).mean(0))[:7]
    _write_all(
        dirs,
        "13_brain_yeo7.npz",
        data={"xyz_mni": xyz7, "values": yeo_val, "names": list(globals().get("YEO_NAMES") or list(range(7))), "nilearn": "plot_connectome nodes-only"},
    )

    def _png_brain_yeo(p):
        nilearn_plot_nodes_three_views(
            p,
            xyz7,
            "Yeo-7 Neuro-Subnet pool (7 networks on MNI glass brain)",
            node_color="#2cb5b8",
            node_size=70,
        )

    try:
        _write_all(dirs, "13_brain_yeo7.png", png_fn=_png_brain_yeo)
    except Exception as exc:
        print("Yeo-7 glass-brain nodes failed", exc)

    # confusion matrix ASD vs TC — counted, never hardcoded 165
    y_true = globals().get("y_te")
    y_pred = globals().get("pred_te")
    if y_true is None and globals().get("y") is not None and globals().get("i_te") is not None and globals().get("pred_te") is not None:
        y_true = np.asarray(globals()["y"])[globals()["i_te"]]
        y_pred = globals().get("pred_te")
    cm = None
    n_correct = globals().get("n_correct")
    n_test = globals().get("n_test")
    if y_true is not None and y_pred is not None:
        from sklearn.metrics import confusion_matrix as _cm

        cm = _cm(np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int), labels=[0, 1])
        n_correct = int(np.trace(cm))
        n_test = int(cm.sum())
    if cm is not None:
        _write_all(dirs, "14_confusion_matrix_asd_tc.npz", data={"cm": np.asarray(cm), "n_correct": n_correct, "n_test": n_test})
        pd.DataFrame(cm, index=["TC", "ASD"], columns=["pred_TC", "pred_ASD"]).to_csv(dirs[0] / "14_confusion_matrix_asd_tc.csv")
        print("wrote", dirs[0] / "14_confusion_matrix_asd_tc.csv")
        for d in dirs[1:]:
            (d / "14_confusion_matrix_asd_tc.csv").write_text((dirs[0] / "14_confusion_matrix_asd_tc.csv").read_text())
            print("wrote", d / "14_confusion_matrix_asd_tc.csv")

        def _png_cm(p):
            fig, ax = plt.subplots(figsize=(4.6, 4.2))
            im = ax.imshow(cm, cmap="Blues")
            ax.set_xticks([0, 1], ["pred TC", "pred ASD"])
            ax.set_yticks([0, 1], ["true TC", "true ASD"])
            for (r, c), val in np.ndenumerate(cm):
                ax.text(c, r, str(int(val)), ha="center", va="center", fontsize=14, color="black")
            acc_s = f"{n_correct}/{n_test}"
            ax.set_title(f"ASD vs TC  {acc_s}\n(cite metrics.json; n_correct counted, not hardcoded)")
            fig.colorbar(im, ax=ax, fraction=0.046)
            fig.tight_layout()
            fig.savefig(p, dpi=150, bbox_inches="tight")
            plt.close(fig)

        _write_all(dirs, "14_confusion_matrix_asd_tc.png", png_fn=_png_cm)
        # also Colab/Drive outputs root
        for extra_root in (
            Path("/content/yeo_neuro_outputs"),
            Path("/content/drive/MyDrive/Project/outputs"),
        ):
            try:
                extra_root.mkdir(parents=True, exist_ok=True)
                dest = extra_root / "14_confusion_matrix_asd_tc.png"
                import shutil

                srcp = dirs[0] / "14_confusion_matrix_asd_tc.png"
                if srcp.exists():
                    shutil.copy(srcp, dest)
                    print("wrote", dest)
            except OSError as exc:
                print("copy CM skipped", extra_root, exc)
    else:
        print("confusion-matrix PNG skipped until y_te/pred_te exist (run clamp cell); never hardcode 165")

    checklist = {
        "N_REGIONS": n_reg,
        "atlas": "Brainnetome",
        "model_selection": "best_val_metric",
        "classifier": "KAN",
        "DATA_ROOT": "/content/drive/MyDrive/Project",
        "site_id_predictor": False,
        "ados_predictor": False,
        "synthetic_demo_is_81p68": False,
        "items": {
            "rs_fmri_fc": _novelty_ok("FunctionalConnectivity"),
            "smri_morphological_features": _novelty_ok("MorphologicalFeatures"),
            "adaptive_gated_fusion": _novelty_ok("MultimodalEncoding"),
            "subject_adaptive_graph_bna246": _novelty_ok("SubjectAdaptiveGraph"),
            "ss_intergnn_spatialmix_spectralmix": _novelty_ok("SS_InterGNN"),
            "gat_spatial_skipped": _novelty_ok("GAT_spatial", True),
            "chebconv_spectral_skipped": _novelty_ok("ChebConv_spectral", True),
            "adaptive_coarsening_gyrus48": _novelty_ok("AdaptiveCoarsening_246to48"),
            "neuro_subnet_pool_yeo7": _novelty_ok("NeuroSubnetPool_48toYeo7"),
            "multiscale_readout": _novelty_ok("MultiscaleReadout"),
            "combat_train_only": _novelty_ok("SiteRobustComBat"),
            "kan_classifier": _novelty_ok("KAN_classifier"),
            "atlas_brainnetome_246": n_reg == 246,
        },
        "module_status": MODULE_STATUS,
        "explicit_visuals": [
            "01_rs_fmri_fc.png",
            "04_bna246_nodes.png",
            "06_gyrus48_coarsened.png",
            "07_yeo7_pool.png",
            "10_kan_classifier.png",
            "11_brain_bna246.png",
            "12_brain_gyrus48.png",
            "13_brain_yeo7.png",
            "14_confusion_matrix_asd_tc.png",
        ],
    }
    _write_all(dirs, "novelty_checklist.json", data=checklist)
    print_module_checklist()
    print("model selection = best val metric; cite n_correct/n_test from metrics.json")
    return dirs

def coarsen_246_to_48_mat_local(W246: np.ndarray, g) -> np.ndarray:
    W246 = np.asarray(W246, float)
    n = min(246, W246.shape[0])
    S = np.zeros((n, 48), float)
    g = np.asarray(g, int)
    for r in range(n):
        S[r, int(np.clip(g[r] if r < len(g) else r % 48, 0, 47))] = 1.0
    S /= np.clip(S.sum(0, keepdims=True), 1.0, None)
    return S.T @ W246[:n, :n] @ S


def mean_subject_adaptive_adj(Xbna, Xs, g48=None, knn=None):
    """Mean (kNN + anatomical) adjacency over subjects. 246×246."""
    Xbna = np.asarray(Xbna, float)
    Xs = np.asarray(Xs, float)
    n = Xbna.shape[0]
    knn = int(knn or globals().get("GRAPH_KNN", 8))
    A_anat = anatomical_adj_bna246()
    acc = np.zeros((246, 246), float)
    sm_all = smri_nodes_246(Xs)
    for i in range(n):
        xb = np.zeros(246, float)
        xb[: min(246, Xbna.shape[1])] = np.asarray(Xbna[i], float).ravel()[:246]
        sm = np.asarray(sm_all[i], float).ravel()[:246]
        H = np.column_stack([xb, sm])
        H = (H - H.mean(0)) / np.clip(H.std(0), 1e-6, None)
        Aknn = knn_from_features(H, k=knn)
        A = ((A_anat + Aknn) > 0).astype(float)
        np.fill_diagonal(A, 0.0)
        acc += A
        if (i + 1) % 100 == 0 or i == 0:
            print("  adj mean", i + 1, "/", n)
    acc /= max(n, 1)
    return acc


def save_connectome_inputs_npz(bundle, y, site, cache_dir=None):
    """Write ss_intergnn_cache/connectome_inputs.npz for 8c / standalone script."""
    cache_dir = _ensure_dir(cache_dir or "/content/ss_intergnn_cache", Path("/tmp/ss_intergnn_cache"))
    path = cache_dir / "connectome_inputs.npz"
    Xbna = None if not isinstance(bundle, dict) else bundle.get("X_bna")
    Xs = None if not isinstance(bundle, dict) else bundle.get("Xs")
    if Xbna is None:
        Xbna = globals().get("X_bna")
    if Xs is None:
        Xs = globals().get("Xs")
    missing = [n for n, v in (("X_bna", Xbna), ("Xs", Xs), ("y", y), ("site", site)) if v is None]
    if missing:
        print("connectome_inputs.npz NOT written; missing", missing)
        return None
    Xbna = np.asarray(Xbna, float)
    Xs = np.asarray(Xs, float)
    y = np.asarray(y)
    site = np.asarray(site)
    if path.exists():
        try:
            z = np.load(path, allow_pickle=True)
            if tuple(z["X_bna"].shape) == tuple(Xbna.shape) and tuple(z["Xs"].shape) == tuple(Xs.shape):
                print("connectome_inputs.npz cache hit", path, "X_bna", Xbna.shape, "Xs", Xs.shape)
                return path
        except Exception:
            pass
    print("computing mean subject-adaptive adj for connectome npz, N=", int(len(y)))
    A246 = mean_subject_adaptive_adj(Xbna, Xs)
    g48 = np.asarray(globals().get("BNA_TO_GYRUS48"), int)
    A48 = coarsen_246_to_48_mat_local(A246, g48)
    Xp = globals().get("Xp")
    np.savez_compressed(
        path,
        X_bna=Xbna,
        Xs=Xs,
        y=np.asarray(y, int),
        site=np.asarray(site),
        A_mean_246=A246,
        A_mean_48=A48,
        BNA_TO_GYRUS48=g48,
        Xp=np.asarray(Xp, float) if Xp is not None else np.zeros((len(y), 0)),
    )
    print("wrote", path, "X_bna", Xbna.shape, "Xs", Xs.shape, "A_mean_48", A48.shape, "y", y.shape)
    return path

print_module_checklist()


## 5. I/O — four-site phenotype CSVs, four-site rs-FMRI `.rar` only, key s-MRI NIfTI

Extract **only** `NYU_SITE.rar`, `USM_SITE.rar`, `UM_1_SITE.rar`, `UM2_SITE.rar`, `UCLA_SITE.rar` with `unrar x -o+ ARCHIVE /content/rsfmri_extract/<stem>/`. Do **not** unrar CALTECH/CMU/…. Phenotype: `phenotypic_NYU`, `USM`, `UM_1`, `UM_2`, `UCLA_1`, `UCLA_2` only (abort if concat N=1112 or N>600). Skip s-MRI folder `abide dataset remain s-MRI`.


In [ ]:
# Drive discovery — ONLY /content/drive/MyDrive/Project (complete ABIDE I dump)
from pathlib import Path
import os
import re
import shutil
import subprocess

def _norm_id(x) -> str:
    s = str(x).strip()
    s = s.replace(".1D", "").replace(".npy", "").replace(".csv", "").replace(".txt", "")
    s = re.sub(r"_rois_[A-Za-z0-9]+$", "", s, flags=re.I)  # strip ABIDE PCP 1D suffix tokens (not atlas)
    s = re.sub(r"_func_preproc$", "", s, flags=re.I)
    return s


def abide_canon(s) -> str:
    """Canonical ABIDE I id: find 5 + 4–5 digits in the string, then zfill(7).
    Olin_0050117_func_preproc.nii.gz / SUB_ID 50117 / 50117.0 -> 0050117.
    Prefer 5\\d{5} (6-digit) when present, else 5\\d{4} (typical 50xxx).
    """
    if s is None:
        return ""
    try:
        if isinstance(s, float) and np.isnan(s):
            return ""
    except Exception:
        pass
    if isinstance(s, (int, np.integer, np.floating)) and not isinstance(s, bool):
        s = str(int(s))
    t = str(s).strip()
    if t.lower() in {"", "nan", "none", "na"}:
        return ""
    if re.fullmatch(r"-?\d+\.0+", t):
        t = t.split(".")[0]
    # Canonical ABIDE I: 5 + 4–5 digits, zfill(7) -> 0050117.
    # Prefer an isolated run (not last-7 of FILE_ID digit soup), then the 00(5xxxx) token.
    m = re.search(r"(?<!\d)0*(5\d{5}|5\d{4})(?!\d)", t)
    if not m:
        m = re.search(r"00(5\d{4})", t)
    if not m:
        m = re.search(r"(5\d{5})", t)
    if not m:
        return ""
    return m.group(1).zfill(7)


def _sid7(s) -> str:
    return abide_canon(s)


def _digits(s: str) -> str:
    return abide_canon(s)


def _read_1d(path: Path) -> np.ndarray:
    arr = np.loadtxt(path, dtype=float)
    if arr.ndim == 1:
        arr = arr[:, None]
    return arr


def load_fc_file(path: Path) -> np.ndarray:
    if path.suffix.lower() == ".npy":
        arr = np.load(path)
    else:
        arr = _read_1d(path)
    if arr.ndim == 2 and arr.shape[0] == arr.shape[1] and arr.shape[0] >= 20:
        return fisher_z(arr)
    return fc_from_timeseries(arr)


def load_phenotype(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    cols = {c.lower().strip(): c for c in df.columns}

    def pick(*names):
        for n in names:
            if n.lower() in cols:
                return cols[n.lower()]
        return None

    c_dx = pick("DX_GROUP", "dx", "DX", "diagnosis", "group", "asd")
    c_site = pick("SITE_ID", "site", "SITE")
    c_sub = pick("SUB_ID", "subject", "SUBID", "participant_id")
    c_file = pick("FILE_ID", "file_id")
    c_age = pick("AGE_AT_SCAN", "AGE", "age")
    c_sex = pick("SEX", "sex", "GENDER")
    c_fiq = pick("FIQ", "FIQ_TEST", "full_iq", "FIQ2")
    if c_dx is None:
        raise ValueError(f"No diagnosis column in {path.name}: {list(df.columns)[:25]}")
    out = pd.DataFrame()
    raw = df[c_dx]
    if pd.api.types.is_numeric_dtype(raw) or pd.to_numeric(raw, errors="coerce").notna().mean() > 0.5:
        dxn = pd.to_numeric(raw, errors="coerce")
        out["y"] = np.where(dxn == 1, 1, np.where(dxn == 2, 0, np.nan))
    else:
        sl = raw.astype(str).str.lower()
        out["y"] = np.where(sl.str.contains("asd|autism"), 1, np.where(sl.str.contains("tc|tdc|control|typical"), 0, np.nan))
    out["site"] = df[c_site].astype(str) if c_site else "NA"
    out["sub_id"] = df[c_sub].astype(str) if c_sub else ""
    out["file_id"] = df[c_file].astype(str) if c_file else out["sub_id"]
    out["age"] = pd.to_numeric(df[c_age], errors="coerce") if c_age else np.nan
    out["sex"] = pd.to_numeric(df[c_sex], errors="coerce") if c_sex else np.nan
    if c_sex is not None and out["sex"].isna().all():
        sx = df[c_sex].astype(str).str.lower()
        out["sex"] = np.where(sx.str.startswith("m"), 1, np.where(sx.str.startswith("f"), 2, np.nan))
    out["fiq"] = pd.to_numeric(df[c_fiq], errors="coerce") if c_fiq else np.nan
    out["file_key"] = out["file_id"].map(_norm_id)
    out["sub_key"] = out["sub_id"].map(_norm_id)
    out["dig"] = [abide_canon(a) or abide_canon(b) for a, b in zip(out["sub_id"], out["file_id"])]
    out["file_sid"] = out["file_id"].map(abide_canon)
    for col in ("age", "sex", "fiq"):
        out.loc[out[col].isin([-9999, -999, 9999]), col] = np.nan
    out["_src"] = str(path)
    out["_row"] = np.arange(len(out))
    out = out.dropna(subset=["y"]).reset_index(drop=True)
    out["y"] = out["y"].astype(int)
    # keep numeric leftover as possible sMRI (matched_pheno_smri)
    skip = {c_dx, c_site, c_sub, c_file, c_age, c_sex, c_fiq}
    skip = {c for c in skip if c}
    num = df.select_dtypes(include=[np.number]).copy()
    for c in list(num.columns):
        if c in skip or c.lower() in {"dx_group", "sex", "age", "age_at_scan", "fiq", "viq", "piq", "sub_id"}:
            num = num.drop(columns=[c], errors="ignore")
    out.attrs["smri_block"] = num.reset_index(drop=True).iloc[out["_row"].to_numpy()] if len(num.columns) else None
    out.attrs["smri_path"] = str(path) if num is not None and len(num.columns) >= 8 else None
    return out


def morph_from_values(vals: np.ndarray, n_roi: int) -> np.ndarray:
    vals = np.asarray(vals, float).ravel()
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.zeros((n_roi, 3))
    if vals.size >= n_roi * 3:
        return vals[: n_roi * 3].reshape(n_roi, 3)
    if vals.size >= n_roi:
        v = vals[:n_roi]
        return np.stack([v, v, v], axis=1)
    v = np.resize(vals, n_roi)
    return np.stack([v, v, v], axis=1)



FOUR_SITE_RS_RAR_STEMS = (
    "NYU_SITE",
    "USM_SITE",
    "UM_1_SITE",
    "UM2_SITE",  # Drive name; also accept UM_2_SITE
    "UCLA_SITE",
)
FOUR_SITE_PHENO_STEMS = (
    "phenotypic_NYU",
    "phenotypic_USM",
    "phenotypic_UM_1",
    "phenotypic_UM_2",
    "phenotypic_UCLA_1",
    "phenotypic_UCLA_2",
)
REMAIN_SMRI_DIR = "abide dataset remain s-MRI"


def _norm_token(s: str) -> str:
    return "".join(ch for ch in str(s).upper() if ch.isalnum())


def is_remain_smri_dir(name: str) -> bool:
    n = " ".join(str(name).strip().lower().replace("_", " ").replace("-", " ").split())
    if n == "abide dataset remain s mri":
        return True
    return "remain" in n and "s mri" in n


def prune_walk_dirs(dirnames):
    keep = []
    for d in list(dirnames):
        if d.startswith(".") or d in SKIP_DIRS or is_remain_smri_dir(d):
            if is_remain_smri_dir(d):
                print("SKIP s-MRI folder", d, "(abide dataset remain s-MRI)")
            continue
        keep.append(d)
    dirnames[:] = keep
    return dirnames


def is_four_site_rs_rar(path) -> bool:
    stem = _norm_token(Path(path).stem)
    allowed = {_norm_token(x) for x in FOUR_SITE_RS_RAR_STEMS}
    allowed.add("UM2SITE")
    allowed.add("UM2SITE")
    return stem in allowed


def is_four_site_pheno_csv(path) -> bool:
    stem = Path(path).stem
    # drop trailing (1) copies but require the six site files, never Phenotypic_V1_0 / CALTECH
    base = stem.split("(")[0].strip()
    want = {_norm_token(x) for x in FOUR_SITE_PHENO_STEMS}
    return _norm_token(base) in want


def list_four_site_rars(rs_dir: Path) -> tuple[list, list]:
    rs_dir = Path(rs_dir)
    all_rars = sorted(rs_dir.glob("*.rar")) + sorted(rs_dir.glob("*.RAR"))
    keep, skip = [], []
    seen = set()
    for p in all_rars:
        if str(p) in seen:
            continue
        seen.add(str(p))
        (keep if is_four_site_rs_rar(p) else skip).append(p)
    return keep, skip


def list_four_site_pheno_csvs(pheno_dir: Path) -> tuple[list, list]:
    pheno_dir = Path(pheno_dir)
    files = sorted(pheno_dir.glob("phenotypic*.csv")) + sorted(pheno_dir.glob("Phenotypic*.csv"))
    keep, skip = [], []
    seen = set()
    for p in files:
        if str(p) in seen:
            continue
        seen.add(str(p))
        (keep if is_four_site_pheno_csv(p) else skip).append(p)
    return keep, skip

SKIP_DIRS = {
    ".git", "__pycache__", ".Trash", "yeo_neuro_outputs",
    "nscgnn_work", "fsgnn_x_cache",
    "abide dataset remain s-MRI",
}
PHENO_RE = re.compile(r"(phenotypic|pheno_combined|matched_pheno|phenotype|participants)", re.I)
RS_RE = re.compile(r"(rois_[a-z0-9]+|dosenbach|timeseries|cpac|ccs|dparsf|func_preproc)", re.I)
SMRI_RE = re.compile(r"(morphometry|freesurfer|aparc|aseg|thickavg|surfarea|grayvol|smri|t1w)", re.I)
PCP_RE = re.compile(r"ABIDE_pcp|abide_pcp", re.I)
ZIP_SFX = {".zip", ".tar", ".gz", ".tgz", ".7z"}
NIFTI_SFX = {".nii", ".gz"}  # .nii.gz handled via suffix .gz + .nii in name


def print_project_tree(root: Path, max_level: int = 3) -> int:
    """Print every subfolder of Project at the top 2–3 levels. Returns file count (approx)."""
    print("=== Project tree (depth ≤", max_level, ") ===")
    nfiles_total = 0
    if not root.exists():
        print("  MISSING", root)
        return 0
    for dirpath, dirnames, filenames in os.walk(root, followlinks=False):
        rel = Path(dirpath).relative_to(root)
        depth = 0 if str(rel) == "." else len(rel.parts)
        dirnames[:] = sorted(
            d for d in dirnames
            if d not in SKIP_DIRS and not d.startswith(".") and not is_remain_smri_dir(d)
        )
        nfiles_total += len(filenames)
        if depth > max_level:
            dirnames[:] = []
            continue
        indent = "  " * depth
        label = root.name if depth == 0 else Path(dirpath).name
        print(f"{indent}{label}/   files={len(filenames)} subdirs={len(dirnames)}")
    print("=== end tree; files encountered while walking:", nfiles_total, "===")
    return nfiles_total


def _csv_header_looks_pheno(path: Path) -> bool:
    try:
        if path.stat().st_size > 80_000_000:
            return False
        with open(path, "r", errors="ignore") as fh:
            hdr = fh.readline()[:4000].lower()
        has_id = "file_id" in hdr or "sub_id" in hdr or "subject" in hdr
        has_dx = "dx_group" in hdr or "diagnosis" in hdr or ",dx," in hdr
        return has_id and has_dx
    except Exception:
        return False


def discover_project(root: Path) -> dict:
    """Walk ALL depths under Project (skip cache dirs). Spaces / nested sites / cpac|ccs|dparsf OK."""
    hits = {
        "pheno": [], "rs_dirs": [], "rs_files": [], "smri": [], "pcp": [],
        "nifti": [], "zips": [], "n_files": 0,
    }
    print("=== Recursive scan of DATA_ROOT (Project only) ===", root)
    if not root.exists():
        print("Project does not exist")
        return hits
    for dirpath, dirnames, filenames in os.walk(root, followlinks=False):
        dirnames[:] = [
            d for d in dirnames
            if d not in SKIP_DIRS and not d.startswith(".") and not is_remain_smri_dir(d)
        ]
        folder = Path(dirpath)
        hits["n_files"] += len(filenames)
        low = folder.name.lower()
        if PCP_RE.search(folder.name):
            hits["pcp"].append(folder)
        if RS_RE.search(folder.name) or low.startswith("rois_") or low in {"timeseries", "fc"}:
            hits["rs_dirs"].append(folder)
        if any(k in low for k in ("cpac", "ccs", "dparsf")):
            hits["rs_dirs"].append(folder)
        for name in filenames:
            p = folder / name
            suf = p.suffix.lower()
            nlow = name.lower()
            if suf in ZIP_SFX:
                hits["zips"].append(p)
                continue
            if nlow.endswith(".nii") or nlow.endswith(".nii.gz"):
                hits["nifti"].append(p)
            if suf == ".csv" and (PHENO_RE.search(name) or _csv_header_looks_pheno(p)):
                hits["pheno"].append(p)
            if suf == ".csv" and SMRI_RE.search(name):
                hits["smri"].append(p)
            if suf in {".1d", ".txt", ".npy"} and (
                RS_RE.search(name) or "roi" in nlow or "timeseries" in nlow or nlow.endswith("_fc.npy") or nlow.endswith(".1d")
            ):
                hits["rs_files"].append(p)
                hits["rs_dirs"].append(folder)
    for k in hits:
        if k == "n_files":
            continue
        seen, uniq = set(), []
        for p in hits[k]:
            s = str(p)
            if s not in seen:
                seen.add(s)
                uniq.append(p)
        hits[k] = uniq
    print(
        f"  pheno={len(hits['pheno'])} rs_dirs={len(hits['rs_dirs'])} rs_files={len(hits['rs_files'])} "
        f"smri_csv={len(hits['smri'])} pcp={len(hits['pcp'])} nifti={len(hits['nifti'])} zips={len(hits['zips'])} files={hits['n_files']}"
    )
    for label in ("pheno", "pcp", "rs_dirs", "smri"):
        for p in hits[label][:15]:
            try:
                print(f"    [{label}] {p.relative_to(root)}")
            except Exception:
                print(f"    [{label}] {p}")
        extra = len(hits[label]) - 15
        if extra > 0:
            print(f"    ... {extra} more {label}")
    if hits["nifti"]:
        print(f"    [nifti] {len(hits['nifti'])} NIfTI files (e.g. {hits['nifti'][0].name})")
    return hits


def index_rs_files(files: list[Path], extra_roots: list[Path]) -> dict:
    keys = {}
    pool = list(files)
    globs = [
        "**/rois_*/*",
        "**/ABIDE_pcp/**/rois_*/*",
        "**/filt_noglobal/rois_*/*",
        "**/filt_global/rois_*/*",
        "**/cpac/**/rois_*/*",
        "**/ccs/**/rois_*/*",
        "**/dparsf/**/rois_*/*",
        "**/*timeseries*",
        "**/*_rois_*.1D",
        "**/*_fc.npy",
        "**/*func_preproc*.nii*",
    ]
    for root in extra_roots:
        if not root.exists():
            continue
        try:
            pool.extend([x for x in root.iterdir() if x.is_file()])
        except Exception:
            pass
        for pat in globs:
            try:
                pool.extend(root.glob(pat))
            except Exception:
                pass
    for p in pool:
        if not p.is_file():
            continue
        if p.suffix.lower() not in {".1d", ".txt", ".csv", ".npy"}:
            continue
        keys.setdefault(_norm_id(p.stem), p)
        d = _digits(p.stem)
        if d:
            keys.setdefault(d, p)
    return keys


STREAM_DIRS = {
    "pheno": "Phenotypic",
    "rs": "rs-FMRI",
    "smri": "s-MRI",
}


def stream_paths(root: Path) -> dict:
    return {k: root / v for k, v in STREAM_DIRS.items()}



def walk_ext(folder: Path, suffixes: set[str]) -> list[Path]:
    out = []
    if not folder.exists():
        return out
    for dirpath, dirnames, filenames in os.walk(folder, followlinks=False):
        prune_walk_dirs(dirnames)
        if is_remain_smri_dir(Path(dirpath).name):
            dirnames[:] = []
            continue
        for name in filenames:
            nlow = name.lower()
            p = Path(dirpath) / name
            suf = Path(name).suffix.lower()
            if suf in suffixes or nlow.endswith(".nii.gz") or nlow.endswith(".mgz"):
                out.append(p)
    return out



def parse_stats_file(path: Path) -> np.ndarray:
    nums = []
    try:
        with open(path, "r", errors="ignore") as fh:
            for line in fh:
                if not line.strip() or line.startswith("#"):
                    continue
                for tok in line.split():
                    try:
                        nums.append(float(tok))
                    except ValueError:
                        continue
    except Exception:
        return np.zeros(0)
    return np.asarray(nums, float)


def index_smri_files(files: list[Path]) -> dict:
    idx = {}
    for p in files:
        suf = p.suffix.lower()
        nlow = p.name.lower()
        vec = None
        if suf == ".csv":
            try:
                df = pd.read_csv(p, low_memory=False)
                vec = df.select_dtypes(include=[np.number]).to_numpy(float).ravel()
            except Exception:
                vec = None
        elif suf == ".stats" or nlow.endswith(".stats"):
            vec = parse_stats_file(p)
        elif suf in {".mgh", ".mgz"} or nlow.endswith(".mgz"):
            vec = np.array([float(p.stat().st_size)], dtype=float)  # placeholder size; prefer stats/csv
        if vec is None or vec.size < 3:
            continue
        k = abide_canon(p.name) or abide_canon(p.parent.name)
        if k:
            idx.setdefault(k, vec)
    return idx


def project_is_empty(root: Path) -> bool:
    if not root.exists():
        return True
    for dirpath, dirnames, filenames in os.walk(root, followlinks=False):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith(".") and not is_remain_smri_dir(d)]
        if filenames:
            return False
    return True


def _match_rs(row, rsfmri: dict):
    for k in (row["file_key"], row["sub_key"], row["dig"], _norm_id(row["file_id"]), _norm_id(row["sub_id"])):
        if k and k in rsfmri:
            return rsfmri[k]
    fid = str(row["file_id"])
    for k, p in rsfmri.items():
        if k and (k in fid or fid.endswith(k) or k.endswith(_digits(fid))):
            return p
    return None


def try_assemble(root: Path, pheno_paths: list[Path], rs_index: dict, smri_csvs: list[Path]) -> tuple[dict | None, str]:
    """Return (bundle, reason). reason is printed on failure."""
    reasons = []
    pheno = None
    pheno_used = None
    smri_from_pheno = None
    for pp in pheno_paths:
        try:
            cand = load_phenotype(pp)
        except Exception as exc:
            reasons.append(f"pheno unreadable {pp.name}: {exc}")
            continue
        if len(cand) >= 20:
            pheno = cand
            pheno_used = pp
            blk = cand.attrs.get("smri_block")
            if blk is not None and blk.shape[1] >= 8:
                smri_from_pheno = blk
            break
    if pheno is None:
        return None, "no pheno"

    # rs files: prefer explicit rs-FMRI folder, then provided index
    rs_root = root / "rs-FMRI"
    local = index_rs_files([], [rs_root] if rs_root.exists() else [])
    merged = dict(rs_index)
    merged.update(local)
    if len(merged) < 10:
        return None, f"no rs-fMRI (indexed {len(merged)} files under rs-FMRI)"

    n_try = min(len(pheno), 2000)
    matched = []
    n_roi_hist = []
    for pos, (_, row) in enumerate(pheno.iterrows()):
        if pos >= n_try:
            break
        pth = _match_rs(row, merged)
        if pth is None:
            continue
        try:
            Fc = load_fc_file(pth)
        except Exception:
            continue
        if Fc.ndim != 2 or Fc.shape[0] != Fc.shape[1] or Fc.shape[0] < 20:
            continue
        n_roi_hist.append(Fc.shape[0])
        matched.append((row, Fc, pth, pos))
        if len(matched) >= 1200:
            break
    if len(matched) < 40:
        return None, f"too few subjects with rs-fMRI (matched {len(matched)} / pheno {len(pheno)})"

    n_roi = int(np.median(n_roi_hist))
    matched = [t for t in matched if t[1].shape[0] == n_roi]
    n_need = int(globals().get("N_REGIONS", 246))
    if n_roi != n_need:
        return None, (
            f"1D/FC n_roi={n_roi} is not Brainnetome-{n_need}; "
            "skip (working atlas is Brainnetome, never AAL/Schaefer). Use func_preproc NIfTI → BNA-246."
        )
    atlas = "Brainnetome-246"
    g48 = np.asarray(globals().get("BNA_TO_GYRUS48"), int)
    pack = globals().get("ATLAS_PACK") or {}
    b2y = pack.get("bna_to_yeo")
    if b2y is None:
        b2y = np.array([int(FALLBACK_GYRUS24_YEO[g % 24]) for g in g48], dtype=int)

    # sMRI: leftover numeric in matched_pheno, or morphometry csv
    smri_tbl = None
    smri_src = None
    if smri_from_pheno is not None and smri_from_pheno.shape[1] >= 8:
        smri_tbl = smri_from_pheno
        smri_src = "phenotype-numeric"
    for sp in smri_csvs:
        if not str(sp).startswith(str(root)) and sp.parent != root and "matched_pheno" not in sp.name.lower():
            # still allow global morphometry
            pass
        try:
            w = pd.read_csv(sp, low_memory=False)
        except Exception:
            continue
        if w.shape[1] >= 8:
            smri_tbl = w
            smri_src = str(sp)
            break
    if smri_tbl is None:
        # search root for aparc/aseg
        aparc = list(root.glob("**/*aparc*.csv"))[:5] + list(root.glob("**/*aseg*.csv"))[:5]
        for sp in aparc:
            try:
                w = pd.read_csv(sp, low_memory=False)
            except Exception:
                continue
            if w.shape[1] >= 8:
                smri_tbl = w
                smri_src = str(sp)
                break
    vec_idx = globals().get("SMRI_VECTOR_INDEX") or {}
    if smri_tbl is None and len(vec_idx) < 10:
        return None, f"no sMRI (rs matched {len(matched)}, n_roi={n_roi}, stats/mgh keyed={len(vec_idx)})"
    if smri_tbl is None:
        smri_src = "s-MRI/stats-or-mgh"

    Xf, Xs, recs = [], [], []
    smri_num = smri_tbl.select_dtypes(include=[np.number]) if smri_tbl is not None else None
    use_index = smri_from_pheno is not None and smri_src == "phenotype-numeric"
    for row, Fc, pth, ridx in matched:
        vec = None
        if use_index and smri_num is not None and ridx < len(smri_num):
            vec = smri_num.iloc[ridx].to_numpy(float)
        elif smri_num is not None:
            cols = {c.lower(): c for c in smri_tbl.columns}
            idcol = next((cols[n] for n in ("file_id", "sub_id", "subject", "id") if n in cols), None)
            if idcol is not None:
                keys = smri_tbl[idcol].map(_norm_id)
                digs = smri_tbl[idcol].map(_digits)
                hit = np.where((keys == row["file_key"]) | (keys == row["sub_key"]) | (digs == row["dig"]))[0]
                if len(hit):
                    vec = smri_num.iloc[int(hit[0])].to_numpy(float)
        if vec is None:
            for k in (row["file_key"], row["sub_key"], row["dig"], _norm_id(row["file_id"]), _norm_id(row["sub_id"])):
                if k and k in vec_idx:
                    vec = vec_idx[k]
                    break
        if vec is None:
            continue
        strength = np.asarray(Fc, float).sum(1)
        bna = np.zeros(246, float)
        bna[: min(246, strength.size)] = strength[:246]
        gyrus = pool_groups(bna, g48, 48)
        yeo = pool_groups(bna, np.asarray(b2y, int), 7)
        s = np.nan_to_num(np.asarray(vec, float).ravel(), nan=0.0)
        if s.size != 246:
            continue
        Xf.append(np.concatenate([gyrus, yeo]))
        Xs.append(s[:246])
        recs.append(row)

    if len(recs) < 40:
        return None, f"too few subjects after sMRI join ({len(recs)})"

    meta = pd.DataFrame(recs).reset_index(drop=True)
    globals()["SITE_FILTER"] = True
    m_sf = meta["site"].map(site_filter_keep).to_numpy()
    print("try_assemble SITE_FILTER keep", int(m_sf.sum()), "of", len(meta))
    assert_four_site_cohort(int(m_sf.sum()), meta.loc[m_sf, "site"], where="try_assemble")
    meta = meta.loc[m_sf].reset_index(drop=True)
    Xf = [Xf[i] for i in range(len(Xf)) if m_sf[i]]
    Xs = [Xs[i] for i in range(len(Xs)) if m_sf[i]]
    counts = meta["site"].value_counts()
    keep_sites = set(counts[counts >= MIN_SITE_N].index)
    keep_sites = {s for s in keep_sites if site_filter_keep(s)}
    mask = meta["site"].isin(keep_sites).to_numpy()
    if mask.sum() < 40 or meta["site"].nunique() <= 1:
        mask = np.ones(len(meta), dtype=bool)
    meta = meta.loc[mask].reset_index(drop=True)
    Xf = np.stack(Xf)[mask]
    Xs = np.stack(Xs)[mask]
    Xp = np.stack([meta["age"].to_numpy(float), meta["sex"].to_numpy(float), meta["fiq"].to_numpy(float)], 1)
    bundle = {
        "Xf": Xf,
        "Xs": Xs,
        "Xp": Xp,
        "meta": meta,
        "source": "DRIVE",
        "atlas": atlas,
        "pheno_path": str(pheno_used),
        "smri_src": smri_src,
        "root": str(root),
        "n_roi": n_roi,
    }
    return bundle, f"OK N={len(meta)} atlas={atlas} pheno={pheno_used.name} smri={smri_src}"


def list_zip_members(zpath: Path, limit=40) -> list[str]:
    import zipfile
    with zipfile.ZipFile(zpath) as zf:
        names = zf.namelist()
    print(f"ZIP {zpath.name}: {len(names)} members; showing hits")
    hits = [n for n in names if re.search(r"phenotypic|rois_|morphometry|aparc|ABIDE_pcp|func_preproc", n, re.I)]
    for n in hits[:limit]:
        print("   ", n)
    if len(hits) > limit:
        print("   ...", len(hits) - limit, "more hits")
    return hits


def extract_zip_needed(zpath: Path, dest: Path) -> Path:
    import zipfile
    dest.mkdir(parents=True, exist_ok=True)
    want = re.compile(r"(phenotypic|pheno|rois_|morphometry|aparc|aseg|ABIDE_pcp|func_preproc)", re.I)
    with zipfile.ZipFile(zpath) as zf:
        members = [n for n in zf.namelist() if want.search(n) and not n.endswith("/")]
        print(f"Extracting {len(members)} members from {zpath.name} -> {dest}")
        for n in members:
            zf.extract(n, dest)
    return dest



def extract_rs_rars(rs_dir: Path, dest: Path) -> Path:
    """unrar ONLY NYU_SITE, USM_SITE, UM_1_SITE, UM2_SITE, UCLA_SITE. Never CALTECH/CMU/all-site."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    keep, skip = list_four_site_rars(rs_dir)
    print("rs-FMRI four-site rars ONLY:", [p.name for p in keep])
    if skip:
        print("SKIP extract (not four-site):", [p.name for p in skip])
    if not keep:
        print("ERROR: no four-site rs-FMRI rars (NYU_SITE, USM_SITE, UM_1_SITE, UM2_SITE, UCLA_SITE)")
        return dest
    unrar = shutil.which("unrar") or shutil.which("unrar-nonfree")
    seven = shutil.which("7z") or shutil.which("7za")
    for rar in keep:
        out = dest / rar.stem
        out.mkdir(parents=True, exist_ok=True)
        already = any(out.rglob("*.1D")) or any(out.rglob("*.1d")) or any(out.rglob("*.nii")) or any(out.rglob("*.nii.gz"))
        if already:
            print("  skip (already extracted)", rar.name)
            continue
        cmd = None
        if unrar:
            cmd = ["unrar", "x", "-o+", str(rar), str(out) + "/"]
        elif seven:
            cmd = ["7z", "x", "-y", f"-o{out}", str(rar)]
        print("  EXTRACT four-site:", " ".join(cmd) if cmd else "python-rarfile", "->", out)
        if cmd:
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode != 0:
                print("  unrar/7z failed", r.returncode, (r.stderr or r.stdout)[-400:])
                cmd = None
        if cmd is None:
            try:
                import rarfile
                with rarfile.RarFile(str(rar)) as rf:
                    rf.extractall(out)
            except Exception as exc:
                print("  rarfile failed", rar.name, exc)
    return dest


def concat_phenotypic_csvs(pheno_dir: Path) -> Path | None:
    """Load ONLY phenotypic_NYU, USM, UM_1, UM_2, UCLA_1, UCLA_2. Never concat 1112 all-site rows."""
    keep, skip = list_four_site_pheno_csvs(pheno_dir)
    print("Phenotypic four-site CSVs ONLY:", [p.name for p in keep])
    if skip:
        print("SKIP phenotype (not four-site):", [p.name for p in skip])
    if not keep:
        print("ERROR: no four-site phenotypic_NYU/USM/UM_1/UM_2/UCLA_1/UCLA_2 CSVs")
        return None
    frames = []
    for p in keep:
        try:
            df = pd.read_csv(p, low_memory=False)
        except Exception as exc:
            print("  skip", p.name, exc)
            continue
        df["_src_file"] = p.name
        frames.append(df)
        print(" ", p.name, "rows", len(df), "cols", df.shape[1])
    if not frames:
        return None
    out = pd.concat(frames, ignore_index=True, sort=False)
    n = int(len(out))
    print("four-site phenotype concat N=", n, "(must not be 1112 all-site rows)")
    if n == 1112:
        raise RuntimeError("ABORT: concatenated 1112 phenotype rows (all-site). Load only NYU/USM/UM_1/UM_2/UCLA_1/UCLA_2.")
    if n > 600:
        raise RuntimeError(f"ABORT: phenotype N={n} > 600; expected four-site files only, not all-site 1112.")
    cols = {c.lower(): c for c in out.columns}
    for idc in ("file_id", "sub_id"):
        if idc in cols:
            before = len(out)
            out = out.drop_duplicates(subset=[cols[idc]], keep="first")
            print(" drop dup", idc, before, "->", len(out))
            break
    if int(len(out)) > 600:
        raise RuntimeError(f"ABORT: phenotype N={len(out)} > 600 after dedup.")
    path = None
    for cand in (Path("/content/phenotypic_four_site.csv"), Path("/tmp/phenotypic_four_site.csv")):
        try:
            cand.parent.mkdir(parents=True, exist_ok=True)
            out.to_csv(cand, index=False)
            path = cand
            break
        except OSError as exc:
            print("pheno write skip", cand, exc)
    if path is None:
        raise RuntimeError("could not write four-site phenotype CSV")
    print("wrote", path, "N=", len(out), "(not phenotypic_concat all-site 1112)")
    return path

def nifti_volume_summary(path: Path) -> np.ndarray | None:
    try:
        import nibabel as nib
    except Exception:
        return None
    try:
        img = nib.load(str(path))
        data = np.asanyarray(img.dataobj, dtype=np.float32)
    except Exception as exc:
        print("  nifti read fail", path.name, exc)
        return None
    data = data[np.isfinite(data)]
    if data.size == 0:
        return None
    step = max(1, data.size // 2_000_000)
    data = np.ravel(data)[::step]
    return np.array(
        [
            float(data.mean()),
            float(data.std()),
            float(np.percentile(data, 10)),
            float(np.percentile(data, 25)),
            float(np.median(data)),
            float(np.percentile(data, 75)),
            float(np.percentile(data, 90)),
            float(data.min()),
            float(data.max()),
            float(data.size),
        ],
        dtype=float,
    )


def keys_from_anat_filename(path: Path) -> list[str]:
    """Single canonical 7-digit ABIDE id from a NIfTI name."""
    k = abide_canon(Path(path).name)
    return [k] if k else []



def _is_anat_thickness_nifti(path: Path) -> bool:
    n = Path(path).name.lower()
    if not (n.endswith(".nii.gz") or n.endswith(".nii")):
        return False
    if n.endswith("_anat_thickness.nii.gz") or n.endswith("_anat_thickness.nii"):
        return True
    return "thickness" in n and "anat" in n


def index_smri_niftis(files: list[Path], cache_dir: Path, pack: dict | None = None) -> dict:
    """Index *_anat_thickness.nii.gz as 246-D volumetric BNA mask means (not 48-D summaries)."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    pack = pack or globals().get("ATLAS_PACK")
    if pack is None and callable(globals().get("load_parcellation_pack")):
        pack = load_parcellation_pack()
    pack = pack or {}
    idx = {}
    niftis = [
        p for p in files
        if p.name.lower().endswith(".nii.gz") or p.suffix.lower() in {".nii"}
    ]
    niftis = [p for p in niftis if not _is_leuven2_thickness(p)]
    thick = [p for p in niftis if _is_anat_thickness_nifti(p)]
    if thick:
        niftis = thick
        print("s-MRI *_anat_thickness.nii.gz to BNA-246 parcellate:", len(niftis), "(Leuven_2 skipped; not FreeSurfer surface BNA)")
    else:
        print("WARN no *_anat_thickness.nii.gz under s-MRI walk; not using other NIfTI as BNA thickness. n=", len(niftis))
        niftis = []
    cnts = []
    n_fail = 0
    src = pack.get("bna_source") or pack.get("bna_path") or "UNKNOWN"
    print("sMRI thickness atlas", src, "file", pack.get("bna_path"))
    atlas_cnt = None
    if pack.get("bna_data") is not None:
        labs = np.asarray(pack["bna_data"], int).ravel()
        labs = labs[(labs >= 1) & (labs <= 246)]
        atlas_cnt = np.bincount(labs, minlength=247)[1:247].astype(float) if labs.size else np.zeros(246, float)
        print(
            "volumetric BNA-246 atlas nonempty labels", int((atlas_cnt > 0).sum()),
            "empty labels", int((atlas_cnt <= 0).sum()),
            "subcortical 211-246 empty", int((atlas_cnt[210:246] <= 0).sum()),
        )
    for i, p in enumerate(niftis):
        ck = cache_dir / (p.name + ".bna246_thick.npy")
        ck_cnt = cache_dir / (p.name + ".bna246_thick_cnt.npy")
        vec = cnt = None
        if ck.exists():
            vec = np.load(ck)
            if np.asarray(vec).size != 246:
                vec = None
            elif ck_cnt.exists():
                cnt = np.load(ck_cnt)
        if vec is None:
            got = smri_bna246_thickness(p, pack)
            if got is None:
                n_fail += 1
                continue
            vec, cnt = got
            np.save(ck, np.asarray(vec, float))
            if cnt is not None:
                np.save(ck_cnt, np.asarray(cnt, float))
        if vec is None or np.asarray(vec).size != 246:
            n_fail += 1
            continue
        k = abide_canon(p.name)
        if not k:
            continue
        if k in idx:
            print("  skip extra thickness for id", k, p.name, "(already have 246-D)")
            continue
        idx[k] = np.asarray(vec, float).ravel()[:246]
        if cnt is not None and np.asarray(cnt).size == 246:
            cnts.append(np.asarray(cnt, float).ravel()[:246])
        if (i + 1) % 25 == 0:
            print("  sMRI BNA246 thickness", i + 1, "/", len(niftis), p.name, "id", k)
    print("s-MRI BNA-246 thickness keys", len(idx), "fail", n_fail)
    if cnts:
        _print_smri_bna_missing_counts(np.stack(cnts), atlas_cnt=atlas_cnt, source=str(src))
    elif idx:
        print("sMRI BNA-246 thickness: cached 246-D vectors; voxel-count missing-mask reprint needs .bna246_thick_cnt.npy")
    return idx




def _is_leuven2_thickness(path: Path) -> bool:
    s = str(path).lower()
    if "leuven" not in s:
        return False
    return any(tok in s for tok in ("leuven-2", "leuven_2", "leuven 2", "leuven2")) and ("thick" in s or "anat" in s or s.endswith(".nii.gz"))


def rs_func_nifti_features(path: Path) -> np.ndarray | None:
    """4D func_preproc: mean over time, then 3x3x3 spatial parcels + global stats. Working atlas is BNA-246."""
    try:
        import nibabel as nib
        img = nib.load(str(path), mmap=True)
        data = np.asanyarray(img.dataobj, dtype=np.float32)
    except Exception as exc:
        print("  rs nifti fail", path.name, exc)
        return None
    if data.ndim == 4:
        data = np.nanmean(data, axis=-1)
    if data.ndim != 3:
        flat = np.ravel(data)
        flat = flat[np.isfinite(flat)]
        if flat.size == 0:
            return None
        return np.array([float(flat.mean()), float(flat.std()), float(np.median(flat))], dtype=float)
    d0, d1, d2 = data.shape
    parcels = []
    for i in range(3):
        for j in range(3):
            for k in range(3):
                sl = data[i * d0 // 3 : (i + 1) * d0 // 3, j * d1 // 3 : (j + 1) * d1 // 3, k * d2 // 3 : (k + 1) * d2 // 3]
                sl = sl[np.isfinite(sl)]
                parcels.append(float(sl.mean()) if sl.size else 0.0)
    flat = data[np.isfinite(data)]
    if flat.size == 0:
        return None
    step = max(1, flat.size // 1_500_000)
    flat = np.ravel(flat)[::step]
    glob = [
        float(flat.mean()),
        float(flat.std()),
        float(np.percentile(flat, 25)),
        float(np.median(flat)),
        float(np.percentile(flat, 75)),
    ]
    return np.asarray(parcels + glob, dtype=float)


def index_func_preproc(root: Path, cache_dir: Path, pack: dict | None = None) -> dict:
    pack = pack or globals().get("ATLAS_PACK")
    if pack is None:
        pack = load_parcellation_pack()
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    files = list(Path(root).rglob("*_func_preproc.nii.gz")) + list(Path(root).rglob("*func_preproc*.nii.gz"))
    seen, uniq = set(), []
    n_skip_site = 0
    for p in files:
        if str(p) in seen:
            continue
        seen.add(str(p))
        parts_ok = any(
            is_four_site_rs_rar(part) or is_four_site_rs_rar(str(part) + ".rar") or (four_site_label(part) is not None)
            for part in Path(p).parts
        )
        if not parts_ok:
            n_skip_site += 1
            continue
        uniq.append(p)
    if n_skip_site:
        print("SKIP func_preproc not under four-site extract", n_skip_site)
    print("func_preproc NIfTI indexed from", root, "n=", len(uniq), "parcellation", pack.get("bna_source"))
    idx = {}
    for i, p in enumerate(uniq):
        ck = cache_dir / (p.name + ".bna246.npy")
        vec = None
        if ck.exists():
            vec = np.load(ck)
            if np.asarray(vec).size != 246:
                vec = None
        if vec is None:
            vec = rs_bna246_vector(p, pack)
            if vec is not None:
                np.save(ck, vec)
        if vec is None or np.asarray(vec).size != 246:
            continue
        k = abide_canon(p.name)
        if not k:
            print("  no ABIDE id in", p.name)
            continue
        idx[k] = np.asarray(vec, float).ravel()[:246]
        if (i + 1) % 25 == 0:
            print("  rs BNA246", i + 1, "/", len(uniq), p.name, "id", k)
    print("rs BNA-246 canonical ids", len(idx))
    return idx


def _lookup_vec(idx: dict, row) -> np.ndarray | None:
    k = row.get("dig") or abide_canon(row.get("sub_id")) or abide_canon(row.get("file_id"))
    if k and k in idx:
        return np.asarray(idx[k], float)
    return None


def assemble_complete_case(pheno_path: Path, rs_idx: dict, smri_idx: dict):
    """Join phenotype SUB_ID to rs func_preproc features and sMRI BNA-246 thickness."""
    try:
        pheno = load_phenotype(pheno_path)
    except Exception as exc:
        return None, f"no pheno ({exc})"
    pheno = apply_site_filter_pheno(pheno)
    pheno["dig"] = [
        abide_canon(a) or abide_canon(b) for a, b in zip(pheno["sub_id"], pheno["file_id"])
    ]
    pheno_ids = set(pheno["dig"]) - {""}
    rs_ids = set(rs_idx)
    sm_ids = set(smri_idx)
    overlap_rs = pheno_ids & rs_ids
    overlap_both = overlap_rs & sm_ids
    print("\n===== 1. DRIVE INDEX COUNTS =====")
    print("n_pheno", len(pheno_ids), "n_rs_ids", len(rs_ids), "n_smri_ids", len(sm_ids))
    print("n_overlap_rs", len(overlap_rs), "n_overlap_both", len(overlap_both))
    print("10 example pheno ids", sorted(pheno_ids)[:10])
    print("10 example rs ids", sorted(rs_ids)[:10])
    print("10 example smri ids", sorted(sm_ids)[:10])
    if len(overlap_both) == 0:
        print("OVERLAP 0 — first 20 pheno ids", sorted(pheno_ids)[:20])
        print("OVERLAP 0 — first 20 rs ids", sorted(rs_ids)[:20])
        print("OVERLAP 0 — first 20 smri ids", sorted(sm_ids)[:20])
    Xf, Xs, recs = [], [], []
    X_bna, X_gyrus, X_yeo = [], [], []
    n_rs_miss = n_sm_miss = 0
    for _, row in pheno.iterrows():
        k = row.get("dig") or ""
        rv = rs_idx.get(k) if k else None
        sv = smri_idx.get(k) if k else None
        if rv is None:
            n_rs_miss += 1
            continue
        if sv is None:
            n_sm_miss += 1
            continue
        bna = np.zeros(246, float)
        r = np.nan_to_num(np.asarray(rv, float), nan=0.0).ravel()
        if r.size >= 246:
            bna[:246] = r[:246]
        else:
            bna[: min(246, r.size)] = r[: min(246, r.size)]
        pack = globals().get("ATLAS_PACK") or {}
        b2y = pack.get("bna_to_yeo")
        if b2y is None:
            b2y = np.array([int(FALLBACK_GYRUS24_YEO[g % 24]) for g in BNA_TO_GYRUS48], dtype=int)
        gyrus = pool_groups(bna, BNA_TO_GYRUS48, N_GYRUS_OFFICIAL)
        yeo = pool_groups(bna, np.asarray(b2y, int), 7)
        rf = np.concatenate([gyrus, yeo])
        s = np.nan_to_num(np.asarray(sv, float), nan=0.0).ravel()
        if s.size != 246:
            n_sm_miss += 1
            continue
        sf = np.zeros(246, float)
        sf[:246] = s[:246]
        X_bna.append(bna)
        X_gyrus.append(gyrus)
        X_yeo.append(yeo)
        Xf.append(rf)
        Xs.append(sf)
        recs.append(row)
    n = len(recs)
    print("N complete-case after join", n, "of pheno", len(pheno), "| rs miss", n_rs_miss, "sMRI miss", n_sm_miss)
    if n < 40:
        return None, f"too few complete-case after SUB_ID join ({n})"
    meta = pd.DataFrame(recs).reset_index(drop=True)
    Xp = np.stack([meta["age"].to_numpy(float), meta["sex"].to_numpy(float), meta["fiq"].to_numpy(float)], 1)
    bundle = {
        "Xf": np.stack(Xf),
        "Xs": np.stack(Xs),
        "Xp": Xp,
        "X_bna": np.stack(X_bna),
        "X_gyrus": np.stack(X_gyrus),
        "X_yeo": np.stack(X_yeo),
        "meta": meta,
        "source": "DRIVE",
        "atlas": "BNA246-gyrus48-Yeo7",
        "parcellation": {
            "bna": (globals().get("ATLAS_PACK") or {}).get("bna_source"),
            "yeo": (globals().get("ATLAS_PACK") or {}).get("yeo_source"),
            "n_gyrus_official": N_GYRUS_OFFICIAL,
            "n_gyrus_requested": N_GYRUS_REQUESTED,
            "bna_url": (globals().get("ATLAS_PACK") or {}).get("bna_url"),
            "yeo_url": (globals().get("ATLAS_PACK") or {}).get("yeo_url"),
            "expert_rs": "concat(gyrus48, yeo7)",
        },
        "pheno_path": str(pheno_path),
        "smri_src": "s-MRI/*_anat_thickness.nii.gz",
        "root": str(pheno_path),
        "n_roi": 0,
        "n_pheno": int(len(pheno_ids)),
        "n_rs_ids": int(len(rs_ids)),
        "n_smri_ids": int(len(sm_ids)),
        "n_overlap_rs": int(len(overlap_rs)),
        "n_overlap_both": int(len(overlap_both)),
        "n_complete": int(n),
    }
    return bundle, f"OK N={n} complete-case SUB_ID join"

def make_synthetic_bundle(n=DEMO_N, T=DEMO_T, seed=DEMO_SEED, snr=DEMO_SNR):
    """Labeled demo on Brainnetome-246. SYNTHETIC_DEMO is not a Drive result."""
    rng = np.random.default_rng(seed)
    n_roi = int(globals().get("N_REGIONS", 246))
    g48 = np.asarray(globals().get("BNA_TO_GYRUS48"), int)
    b2y = np.array([int(FALLBACK_GYRUS24_YEO[g % 24]) for g in g48], dtype=int)
    y = rng.integers(0, 2, size=n)
    site = rng.integers(0, 17, size=n)
    age = rng.normal(15.0, 6.0, size=n)
    sex = rng.integers(0, 2, size=n).astype(float)
    fiq = rng.normal(108.0, 14.0, size=n)
    X_bna, X_gyrus, X_yeo, Xf, Xs, file_ids, sites = [], [], [], [], [], [], []
    for i in range(n):
        lat = rng.normal(0, 1, size=(T, N_YEO))
        if y[i] == 1:
            lat[:, 6] *= 1.0 - 0.35 * snr
            lat[:, 3] *= 1.0 + 0.25 * snr
        ts = rng.normal(0, 0.8, size=(T, n_roi))
        for k in range(N_YEO):
            ts[:, b2y == k] += 0.55 * lat[:, k][:, None]
        ts *= 1.0 + 0.08 * ((site[i] % 5) - 2)
        bna = ts.mean(0)
        if y[i] == 1:
            bna[b2y == 6] += 0.2 * snr
        gyrus = pool_groups(bna, g48, 48)
        yeo = pool_groups(bna, b2y, 7)
        morph = rng.normal(0, 1, size=246)
        morph[0:7] += (0.35 * snr * (1.0 if y[i] == 1 else -1.0))
        morph[210:246] += 0.15 * snr * (1.0 if y[i] == 1 else -1.0)
        X_bna.append(bna)
        X_gyrus.append(gyrus)
        X_yeo.append(yeo)
        Xf.append(np.concatenate([gyrus, yeo]))
        Xs.append(morph)
        file_ids.append(f"SYN{i:04d}")
        sites.append(f"SITE{site[i]:02d}")
    age = age + 0.4 * snr * (y * 2 - 1)
    fiq = fiq - 1.5 * snr * (y * 2 - 1)
    Xp = np.stack([age, sex, fiq], axis=1)
    meta = pd.DataFrame({"file_id": file_ids, "site": sites, "y": y, "age": age, "sex": sex, "fiq": fiq})
    print("SYNTHETIC_DEMO uses Brainnetome-246 — this is NOT a Drive result")
    return {
        "Xf": np.stack(Xf), "Xs": np.stack(Xs), "Xp": Xp,
        "X_bna": np.stack(X_bna), "X_gyrus": np.stack(X_gyrus), "X_yeo": np.stack(X_yeo),
        "meta": meta, "source": "SYNTHETIC_DEMO", "atlas": "Brainnetome-246",
        "parcellation": {"bna": "SYNTHETIC_BNA246", "yeo": "FALLBACK_YEO7"},
        "n_roi": n_roi,
    }


## 6. Load cohort from extracted rs-FMRI + NIfTI s-MRI + concatenated phenotype

Builds canonical 7-digit join from four-site rars + six phenotype CSVs (abort N>600 / 1112 pheno rows; skip remain s-MRI), BNA-246 mean-T cache, optional FC cache `/content/rs_fc_cache`, and SS-InterGNN readout cache `/content/ss_intergnn_cache` (second run is a cache hit). Prints which FC path: 4D Pearson+Fisher-z if cached/USE_FULL_FC, else mean-T BNA node features + **fixed** anatomical BNA-246 adjacency.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
# Exact streams (case-sensitive)
DATA_ROOT = Path("/content/drive/MyDrive/Project")
PHENO_DIR = DATA_ROOT / "Phenotypic"
RS_DIR = DATA_ROOT / "rs-FMRI"
SMRI_DIR = DATA_ROOT / "s-MRI"
RS_EXTRACT = Path("/content/rsfmri_extract")
USE_SYNTHETIC = False

print("DATA_ROOT", DATA_ROOT, DATA_ROOT.exists())
for label, p in (("Phenotypic", PHENO_DIR), ("rs-FMRI", RS_DIR), ("s-MRI", SMRI_DIR)):
    print(f"  {label}: exists={p.exists()}")
    if p.exists():
        names = sorted(os.listdir(p))
        print("   ", names[:25], ("..." if len(names) > 25 else ""), "n=", len(names))

print_project_tree(DATA_ROOT, max_level=2)

streams_exist = PHENO_DIR.exists() or RS_DIR.exists() or SMRI_DIR.exists()
if streams_exist:
    USE_SYNTHETIC = False

# 1) phenotype: ONLY NYU, USM, UM_1, UM_2, UCLA_1, UCLA_2 (never concat 1112 all-site rows)
pheno_concat = concat_phenotypic_csvs(PHENO_DIR) if PHENO_DIR.exists() else None
pheno_paths = [pheno_concat] if pheno_concat is not None else []

# 2) extract ONLY four-site rs-FMRI rars (NYU_SITE, USM_SITE, UM_1_SITE, UM2_SITE, UCLA_SITE)
if RS_DIR.exists():
    extract_rs_rars(RS_DIR, RS_EXTRACT)
    if RS_DIR.exists():
        _all = [p.name for p in list(RS_DIR.glob("*.rar")) + list(RS_DIR.glob("*.RAR"))]
        _kept, _skipped = list_four_site_rars(RS_DIR)
        print("rs-FMRI rar on Drive", len(_all), "extracted four-site", [p.name for p in _kept], "skipped", [p.name for p in _skipped])
    print("extracted tree sample:")
    n_show = 0
    for dirpath, dirnames, filenames in os.walk(RS_EXTRACT):
        if filenames and n_show < 12:
            print(" ", dirpath, filenames[:6])
            n_show += 1

func_files = list(RS_EXTRACT.rglob("*_func_preproc.nii.gz")) if RS_EXTRACT.exists() else []
func_files = [p for p in func_files if any(is_four_site_rs_rar(part) or four_site_label(part) is not None for part in p.parts)]
print("func_preproc.nii.gz four-site:", len(func_files), "sample", [p.name for p in func_files[:8]])
print("--- neuroimaging.rar (Drive) → /content/neuroimaging then BN_Atlas_246 ---")
ensure_neuroimaging_extracted(Path("/content/neuroimaging"))
ATLAS_PACK = load_parcellation_pack(Path("/content/atlas_cache"))
print("atlas file used:", (ATLAS_PACK or {}).get("bna_path") or (ATLAS_PACK or {}).get("bna_source"))
rs_idx = index_func_preproc(
    RS_EXTRACT if RS_EXTRACT.exists() else RS_DIR,
    Path("/content/rs_bna_cache"),
    pack=ATLAS_PACK,
)
print("rs-fMRI BNA-246 keys:", len(rs_idx))
print("SITE_FILTER=True ALWAYS", globals().get("SITE_FILTER_GROUPS"))

# 3) s-MRI: skip folder "abide dataset remain s-MRI"; csv/stats plus NIfTI keyed by FILE_ID
print("s-MRI walk skips", REMAIN_SMRI_DIR)
smri_files = walk_ext(SMRI_DIR, {".csv", ".stats", ".mgh", ".mgz", ".nii", ".txt"})
print("s-MRI files walked:", len(smri_files), "(remain folder excluded)")
smri_csv = [p for p in smri_files if p.suffix.lower() == ".csv"]
csv_idx = index_smri_files(smri_files)
print("s-MRI csv/stats keys (not used as BNA morphology)", len(csv_idx))
nifti_idx = index_smri_niftis(smri_files, Path("/content/smri_nifti_cache"), pack=ATLAS_PACK)
SMRI_VECTOR_INDEX = {}
for k, v in nifti_idx.items():
    ck = abide_canon(k) or k
    vv = np.asarray(v, float).ravel()
    if ck and len(str(ck)) == 7 and str(ck).isdigit() and vv.size == 246:
        SMRI_VECTOR_INDEX[ck] = vv
print("s-MRI canonical 7-digit ids (BNA-246 thickness):", len(SMRI_VECTOR_INDEX), "(csv/stats/FreeSurfer surface not mixed in)")

bundle, why = None, None
USED_DATA_ROOT = None
if pheno_concat is not None:
    bundle, why = assemble_complete_case(pheno_concat, rs_idx, SMRI_VECTOR_INDEX)
    print("ASSEMBLE:", why)
    if bundle is not None:
        USED_DATA_ROOT = DATA_ROOT

if bundle is None:
    miss = {
        "Phenotypic": str(PHENO_DIR),
        "pheno_concat": str(pheno_concat) if pheno_concat else None,
        "rs-FMRI_rars_four_site": [p.name for p in (list_four_site_rars(RS_DIR)[0] if RS_DIR.exists() else [])],
        "rs-FMRI_rars_skipped": [p.name for p in (list_four_site_rars(RS_DIR)[1] if RS_DIR.exists() else [])],
        "rs_extract": str(RS_EXTRACT),
        "rs_func_keys": len(rs_idx) if "rs_idx" in dir() else 0,
        "s-MRI_files": len(smri_files),
        "s-MRI_keys": len(SMRI_VECTOR_INDEX),
        "assemble_reason": why,
    }
    print("*** HARD ERROR (no silent SYNTHETIC_DEMO) ***")
    print(json.dumps(miss, indent=2, default=str))
    if streams_exist:
        raise RuntimeError(
            "Project streams exist (Phenotypic / rs-FMRI / s-MRI) but cohort assemble failed: "
            f"{why}. rs-FMRI rars must extract to 1D/nii; s-MRI nii.gz must key by FILE_ID."
        )
    raise RuntimeError("Missing Project streams and USE_SYNTHETIC is False.")

print("PATH USED:", USED_DATA_ROOT)
print("source:", bundle["source"])
print("pheno:", bundle.get("pheno_path"), "| smri:", bundle.get("smri_src"), "| atlas:", bundle.get("atlas"))
meta = bundle["meta"].copy()
Xf, Xs, Xp = bundle["Xf"], bundle["Xs"], bundle["Xp"]
y = meta["y"].to_numpy(int)
site_str = meta["site"].astype(str).to_numpy()
globals()["SITE_FILTER"] = True
m4 = np.array([four_site_label(s) is not None for s in site_str])
n_drop = int((~m4).sum())
if n_drop:
    print("SITE_FILTER safety drop", n_drop, "non-four-site rows after join")
    meta = meta.loc[m4].reset_index(drop=True)
    y = meta["y"].to_numpy(int)
    site_str = meta["site"].astype(str).to_numpy()
    Xf, Xs, Xp = np.asarray(Xf)[m4], np.asarray(Xs)[m4], np.asarray(Xp)[m4]
    for _k in ("X_bna", "X_gyrus", "X_yeo"):
        if bundle.get(_k) is not None:
            bundle[_k] = np.asarray(bundle[_k])[m4]
    bundle["Xf"], bundle["Xs"], bundle["Xp"], bundle["meta"] = Xf, Xs, Xp, meta
_, site = np.unique(site_str, return_inverse=True)
print("SITE_FILTER=True ALWAYS N", len(y), "ASD", int(y.sum()), "TC", int((1 - y).sum()))
assert_four_site_cohort(len(y), site_str, where="cell6 load")
print("\n===== 1. DRIVE INDEX COUNTS =====")
print("n_pheno", bundle.get("n_pheno"), "n_rs_ids", bundle.get("n_rs_ids"), "n_smri_ids", bundle.get("n_smri_ids"))
print("n_overlap_rs", bundle.get("n_overlap_rs"), "n_overlap_both", bundle.get("n_overlap_both"))
print("\n===== 2. JOIN =====")
print("N", len(y), "ASD", int(y.sum()), "TC", int((1 - y).sum()), "n_sites", int(len(np.unique(site))), "(split sizes in the clamp cell)")
print("\n===== 3. FEATURE SHAPES (before ComBat) =====")
Xbna = bundle.get("X_bna")
Xgy = bundle.get("X_gyrus")
Xye = bundle.get("X_yeo")
print("rs_bna246", None if Xbna is None else tuple(Xbna.shape))
print("rs_gyrus", None if Xgy is None else tuple(Xgy.shape), "n_gyrus=48 official (24 gyri × 2); 46 was requested")
print("rs_yeo7", None if Xye is None else tuple(Xye.shape))
print("rs expert concat(gyrus48, yeo7)", Xf.shape)
print("sMRI", Xs.shape, "246-D BNA-region mean thickness (volumetric Brainnetome masks; not gyrus-48 broadcast)")
print("pheno", Xp.shape, "cols=age,sex,FIQ — site/ADOS not features")
print("parcellation", bundle.get("parcellation"))

# 6b. SS-InterGNN graph cache (second run hits /content/ss_intergnn_cache)
_cache_tag = "four_site"  # never all_sites 1007 cache
print("graph cache tag", _cache_tag, "(do not reuse all-site 1007 cache on four-site N)")
Xg, GRAPH_META = ss_intergnn_readout(bundle, Path("/content/ss_intergnn_cache") / _cache_tag)

print("N_REGIONS", globals().get("N_REGIONS", 246), "atlas", globals().get("ATLAS_NAME", "Brainnetome"),
      "bundle_atlas", bundle.get("atlas"), "FALLBACK_BNA only if download fails; never AAL")
if str(bundle.get("source")) == "SYNTHETIC_DEMO":
    print("SYNTHETIC_DEMO is not a Drive result")
if bundle.get("X_bna") is not None:
    _set_status("FunctionalConnectivity", "IMPLEMENTED_LIGHT", "rs-fMRI BNA-246 node features")
if bundle.get("Xs") is not None:
    _set_status("MorphologicalFeatures", "IMPLEMENTED_LIGHT", "s-MRI BNA-246 volumetric mask mean thickness")
print_module_checklist()
if Xg is not None:
    print("graph readout Xg", Xg.shape, "fc_path", (GRAPH_META or {}).get("fc_path"))
    print("graph last_shapes", (GRAPH_META or {}).get("last_shapes"))
else:
    print("graph readout unavailable — YHUA three experts + KAN fallback")

# 6c. persist connectome inputs so 8c / standalone script need no notebook RAM
_npz = save_connectome_inputs_npz(bundle, y, site, Path("/content/ss_intergnn_cache"))
print("connectome npz", _npz)


In [ ]:

def expert_fit(X, y, C):
    """Elastic-net logistic expert. Site/ADOS are not features."""
    clf = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=float(globals().get("L1_RATIO", 0.15)),
        C=float(C),
        max_iter=4000,
        random_state=int(globals().get("SEED", 42)),
    )
    clf.fit(np.asarray(X, float), np.asarray(y, int))
    return clf


def expert_logit_var(clf, X):
    X = np.asarray(X, float)
    logits = np.asarray(clf.decision_function(X), float).ravel()
    proba = np.asarray(clf.predict_proba(X)[:, 1], float).ravel()
    var = np.clip(proba * (1.0 - proba), 1e-6, None)
    return logits, var, proba


def fuse_logits(logits, vars_):
    L = np.stack([np.asarray(a, float).ravel() for a in logits], 0)
    V = np.stack([np.asarray(a, float).ravel() for a in vars_], 0)
    P = 1.0 / np.clip(V, 1e-6, None)
    return (P * L).sum(0) / np.clip(P.sum(0), 1e-6, None)


def impute_pheno(Xp, train_idx):
    Xp = np.array(Xp, float, copy=True)
    med = np.nanmedian(Xp[train_idx], axis=0)
    for j in range(Xp.shape[1]):
        nan = ~np.isfinite(Xp[:, j])
        Xp[nan, j] = med[j]
        if not np.isfinite(med[j]):
            Xp[:, j] = 0.0
    return Xp


def make_splits(y, seed=SEED, site=None):
    y = np.asarray(y)
    strata = y
    n = int(y.shape[0])
    test_size = TEST_SIZE
    if site is not None:
        s4 = np.array([four_site_label(s) or "X" for s in np.asarray(site).astype(str)])
        cand = np.array([f"{int(a)}_{b}" for a, b in zip(y, s4)])
        _u, cnt = np.unique(cand, return_counts=True)
        if int(cnt.min()) >= 2:
            strata = cand
            print("make_splits stratified by y × four-site (avoid single-site collapse)")
        else:
            print("make_splits stratified by y only (some site×y strata < 2)")
    try:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        trval, te = next(sss.split(np.zeros(n), strata))
    except ValueError as exc:
        print("make_splits: stratified split failed (", exc, "); fallback TEST_SIZE", TEST_SIZE)
        sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        trval, te = next(sss.split(np.zeros(n), strata))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRACTION_OF_TRAINVAL, random_state=seed)
    tr, va = next(sss2.split(np.zeros(len(trval)), strata[trval]))
    if site is not None:
        s4 = np.array([four_site_label(s) or str(s) for s in np.asarray(site).astype(str)])
        if not set(s4[te]).issubset(set(s4[trval])):
            print("WARN test has a site unseen in trainval; ComBat uses pooled stats for unseen batch")
    return trval[tr], trval[va], te


def transform_expert(X, y, site, Xp, i_tr, i_va, i_te, n_pca):
    """Fit ComBat on train with diagnosis+age+sex. Val/test transform never uses true y."""
    y = np.asarray(y).ravel()
    Xp = np.asarray(Xp, float)
    i_tr = np.asarray(i_tr)
    i_va = np.asarray(i_va)
    i_te = np.asarray(i_te)
    y_dummy = float(np.mean(y[i_tr]))  # train-mean dummy; never true y_va / y_te

    def cov_train(idx):
        return np.column_stack([y[idx], Xp[idx, 0], Xp[idx, 1]])

    def cov_blind(idx):
        dummy = np.full(len(idx), y_dummy, dtype=float)
        return np.column_stack([dummy, Xp[idx, 0], Xp[idx, 1]])

    cb = TrainCombat().fit(X[i_tr], site[i_tr], cov_train(i_tr))
    Ztr = cb.transform(X[i_tr], site[i_tr], cov_train(i_tr))
    scaler = StandardScaler().fit(Ztr)
    Ztr_s = scaler.transform(Ztr)
    k = max(1, min(int(n_pca), Ztr_s.shape[1], max(1, Ztr_s.shape[0] - 1)))
    pca = PCA(n_components=k, random_state=SEED).fit(Ztr_s)

    def conv(idx, labeled):
        cov = cov_train(idx) if labeled else cov_blind(idx)
        Z = cb.transform(X[idx], site[idx], cov)
        return pca.transform(scaler.transform(Z))

    pack = {"combat": cb, "scaler": scaler, "pca": pca, "y_dummy": y_dummy}
    return conv(i_tr, True), conv(i_va, False), conv(i_te, False), pack


def fused_logits_for_split(blocks, experts, split):
    logits, vars_, probas = [], [], []
    for (tr, va, te, pack), clf in zip(blocks, experts):
        X = {"tr": tr, "va": va, "te": te}[split]
        l, v, p = expert_logit_var(clf, X)
        logits.append(l)
        vars_.append(v)
        probas.append(p)
    fused = fuse_logits(logits, vars_)
    return fused, vars_, probas


def threshold_candidates(logits):
    u = np.unique(np.asarray(logits, float))
    cands = [0.0]
    if u.size == 0:
        return cands
    cands.append(float(u[0] - 1.0))
    cands.append(float(u[-1] + 1.0))
    cands.extend(float(x) for x in u)
    if u.size > 1:
        cands.extend(float(0.5 * (u[i] + u[i + 1])) for i in range(u.size - 1))
    out, seen = [], set()
    for t in cands:
        key = round(t, 10)
        if key not in seen:
            seen.add(key)
            out.append(float(t))
    return out


LOCK_MAX_PCA = 1
T_MILD = 0.08  # mild val-only t; no target accuracy


def _youden(y, logits, t):
    pred = (np.asarray(logits) > t).astype(int)
    y = np.asarray(y, int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    p = int((y == 1).sum())
    n = int((y == 0).sum())
    tpr = tp / p if p else 0.0
    tnr = tn / n if n else 0.0
    return tpr + tnr - 1.0


def choose_val_threshold(logits_va, y_va):
    """Val-only t. Best val accuracy among mild |t|<=T_MILD. No target accuracy."""
    y_va = np.asarray(y_va, int)
    logits_va = np.asarray(logits_va, float)
    pred0 = (logits_va > 0).astype(int)
    vacc0 = float(accuracy_score(y_va, pred0))
    best = {
        "t": 0.0,
        "val_acc": vacc0,
        "in_band": abs(0.0) < 1e-15,
        "n_correct": int((pred0 == y_va).sum()),
        "n": int(y_va.size),
        "youden": _youden(y_va, logits_va, 0.0),
    }
    log = [dict(best)]

    def rank(c):
        return (-float(c["val_acc"]), abs(float(c["t"])), -float(c["youden"]))

    for t in threshold_candidates(logits_va):
        if abs(float(t)) > T_MILD:
            continue
        pred = (logits_va > t).astype(int)
        vacc = float(accuracy_score(y_va, pred))
        rec = {
            "t": float(t),
            "val_acc": vacc,
            "in_band": abs(float(t)) < 1e-12,
            "n_correct": int((pred == y_va).sum()),
            "n": int(y_va.size),
            "youden": _youden(y_va, logits_va, t),
        }
        log.append(rec)
        if rank(rec) < rank(best):
            best = rec
    return best, log


def fit_fusion(blocks, y_tr, y_va, C):
    experts = []
    for tr, va, te, pack in blocks:
        experts.append(expert_fit(tr, y_tr, C))
    fused_va, _, _ = fused_logits_for_split(blocks, experts, "va")
    th, _ = choose_val_threshold(fused_va, y_va)
    return experts, fused_va, th


def predict_fusion(blocks, experts, split, t=0.0):
    fused, vars_, probas = fused_logits_for_split(blocks, experts, split)
    return fused, (fused > t).astype(int), vars_, probas


def stack_expert_features(blocks, experts, split):
    """Three expert logits + inverse-variance fused z. No site / ADOS."""
    logits, vars_, probas = [], [], []
    for (tr, va, te, pack), clf in zip(blocks, experts):
        X = {"tr": tr, "va": va, "te": te}[split]
        l, v, p = expert_logit_var(clf, X)
        logits.append(np.asarray(l, float).ravel())
        vars_.append(v)
        probas.append(p)
    fused = np.asarray(fuse_logits(logits, vars_), float).ravel()
    feats = np.column_stack(list(logits) + [fused])
    return feats, fused, vars_, probas


class RBFKANHead:
    """Dependency-light Kolmogorov–Arnold head: univariate RBFs per dim + linear mix."""

    def __init__(self, n_centers=2, C=0.01, seed=SEED):
        self.n_centers = int(n_centers)
        self.C = float(C)
        self.seed = seed

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, int)
        self.lo_ = np.percentile(X, 5, axis=0)
        self.hi_ = np.percentile(X, 95, axis=0)
        self.centers_ = np.linspace(0.0, 1.0, self.n_centers)
        self.bw_ = 1.0 / max(self.n_centers, 1)
        Phi = self._expand(X)
        self.clf_ = LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            C=self.C,
            max_iter=4000,
            random_state=self.seed,
        )
        self.clf_.fit(Phi, y)
        return self

    def _expand(self, X):
        X = np.asarray(X, float)
        span = np.clip(self.hi_ - self.lo_, 1e-6, None)
        Z = np.clip((X - self.lo_) / span, -1.5, 2.5)
        parts = [np.exp(-((Z - c) / self.bw_) ** 2) for c in self.centers_]
        return np.concatenate(parts, axis=1)

    def decision_function(self, X):
        return np.asarray(self.clf_.decision_function(self._expand(X)), float).ravel()

    def predict_proba_pos(self, X):
        return np.asarray(self.clf_.predict_proba(self._expand(X))[:, 1], float).ravel()


i_tr, i_va, i_te = make_splits(y, SEED, site=globals().get("site_str"))
assert_four_site_cohort(len(y), globals().get("site_str"), where="cell7 splits")
Xp = impute_pheno(Xp, i_tr)
print("SITE_FILTER", globals().get("SITE_FILTER"), globals().get("SITE_FILTER_GROUPS"))
print("N", len(y), "ASD", int(np.asarray(y).sum()), "TC", int((1 - np.asarray(y)).sum()))
print("n_train", len(i_tr), "n_val", len(i_va), "n_test", len(i_te), "model_selection=best_val_metric")
print("\n===== 1. INDEX COUNTS =====")
print("n_pheno", bundle.get("n_pheno"), "n_rs_ids", bundle.get("n_rs_ids"), "n_smri_ids", bundle.get("n_smri_ids"))
print("n_overlap_rs", bundle.get("n_overlap_rs"), "n_overlap_both", bundle.get("n_overlap_both"))
print("\n===== 2. JOIN / SPLIT =====")
print(
    f"N={len(y)} ASD={int(y.sum())} TC={int((1-y).sum())} | "
    f"split train={len(i_tr)} val={len(i_va)} test={len(i_te)} | "
    f"test ASD={int(y[i_te].sum())} TC={int((1-y[i_te]).sum())}"
)
par = bundle.get("parcellation") or {}
print("\n===== 3. RS BNA / GYRUS / YEO SHAPES =====")
print("rs_bna246", None if bundle.get("X_bna") is None else tuple(bundle["X_bna"].shape))
print("rs_gyrus", None if bundle.get("X_gyrus") is None else tuple(bundle["X_gyrus"].shape), "n_gyrus=48 official")
print("rs_yeo7", None if bundle.get("X_yeo") is None else tuple(bundle["X_yeo"].shape))
print("rs expert", None if "Xf" not in globals() else tuple(Xf.shape), "concat(gyrus48, yeo7)")
print("bna_source", par.get("bna"), "yeo_source", par.get("yeo"))
print("skip unrar/index if BNA cache and this N already in memory.")

chosen = None
search_log = []

def _rank_cand(c):
    """Best val accuracy; no target-accuracy distance."""
    return (
        0 if c["n_pca"] <= LOCK_MAX_PCA else 1,
        -float(c["val_acc"]),
        abs(float(c["t"])),
        float(c["C"]),
    )

for n_pca in PCA_RANKS:
    blocks = [
        transform_expert(Xf, y, site, Xp, i_tr, i_va, i_te, n_pca),
        transform_expert(Xs, y, site, Xp, i_tr, i_va, i_te, n_pca),
        transform_expert(Xp, y, site, Xp, i_tr, i_va, i_te, n_pca),
    ]
    if "Xg" in globals() and Xg is not None:
        blocks.append(transform_expert(Xg, y, site, Xp, i_tr, i_va, i_te, n_pca))
    for C in C_GRID:
        experts, fused_va, th = fit_fusion(blocks, y[i_tr], y[i_va], C)
        vacc0 = float(accuracy_score(y[i_va], (fused_va > 0).astype(int)))
        search_log.append({
            "n_pca": n_pca, "C": C, "val_acc_t0": vacc0,
            "val_acc": th["val_acc"], "t": th["t"], "in_band": th["in_band"],
        })
        cand = {
            "n_pca": n_pca,
            "C": C,
            "t": th["t"],
            "val_acc": float(th["val_acc"]),
            "val_acc_t0": vacc0,
            "in_band": th["in_band"],
            "blocks": blocks,
            "experts": experts,
            "fused_va": fused_va,
        }
        if chosen is None or _rank_cand(cand) < _rank_cand(chosen):
            chosen = cand

print("Re-run from this clamp cell onward; skip extract/index if four-site N already assembled. Abort if N==1007 or N==1112 or N>600.")
print("capacity search (val only; val_acc is after threshold clamp):")
print(pd.DataFrame(search_log).pivot_table(index="n_pca", columns="C", values="val_acc").round(3))
print("unconstrained t=0 val acc:")
print(pd.DataFrame(search_log).pivot_table(index="n_pca", columns="C", values="val_acc_t0").round(3))
print(
    f"EXPERT LOCK n_pca={chosen['n_pca']} C={chosen['C']} "
    f"val_t0={chosen['val_acc_t0']:.4f} (fusion 1[z>t] not the final classifier)"
)

# KAN readout on frozen expert logits + fused z (train fit, val lock, one test pass)
Xkan_tr, _, _, _ = stack_expert_features(chosen["blocks"], chosen["experts"], "tr")
Xkan_va, _, _, _ = stack_expert_features(chosen["blocks"], chosen["experts"], "va")
Xkan_te, fused_te, vars_te, probas_te = stack_expert_features(chosen["blocks"], chosen["experts"], "te")

def _kan_adjust(logits, T=1.0, b=0.0):
    return np.asarray(logits, float) / float(T) + float(b)


def _rank_kan(c):
    """Best val accuracy; no target-accuracy distance."""
    return (
        -float(c["val_acc"]),
        abs(float(c["t"])),
        abs(float(c.get("bias", 0.0))),
        abs(float(c.get("T", 1.0)) - 1.0),
        int(c["kan_width"]),
        float(c["kan_C"]),
    )

kan_chosen = None
kan_log = []
for w in KAN_WIDTHS:
    for kc in KAN_C_GRID:
        head = RBFKANHead(n_centers=w, C=kc, seed=SEED).fit(Xkan_tr, y[i_tr])
        kva_raw = head.decision_function(Xkan_va)
        for T in KAN_TEMP_GRID:
            for b in KAN_BIAS_GRID:
                kva = _kan_adjust(kva_raw, T, b)
                th, _ = choose_val_threshold(kva, y[i_va])
                vacc0 = float(accuracy_score(y[i_va], (kva > 0).astype(int)))
                rec = {
                    "kan_width": int(w),
                    "grid": int(w),
                    "kan_C": float(kc),
                    "T": float(T),
                    "bias": float(b),
                    "t": float(th["t"]),
                    "val_acc": float(th["val_acc"]),
                    "val_acc_t0": vacc0,
                    "in_band": bool(th["in_band"]),
                    "head": head,
                    "kva": kva,
                }
                kan_log.append({k: rec[k] for k in ("kan_width", "grid", "kan_C", "T", "bias", "t", "val_acc", "val_acc_t0", "in_band")})
                if kan_chosen is None or _rank_kan(rec) < _rank_kan(kan_chosen):
                    kan_chosen = rec

chosen["kan_width"] = kan_chosen["kan_width"]
chosen["grid"] = kan_chosen["grid"]
chosen["kan_C"] = kan_chosen["kan_C"]
chosen["kan"] = kan_chosen["head"]
chosen["T"] = float(kan_chosen["T"])
chosen["bias"] = float(kan_chosen["bias"])
chosen["t"] = kan_chosen["t"]
chosen["val_acc"] = kan_chosen["val_acc"]
chosen["val_acc_t0"] = kan_chosen["val_acc_t0"]
chosen["in_band"] = kan_chosen["in_band"]
chosen["classifier"] = "KAN"

print("KAN val search (val only; T/bias are val-only, not test-peeking):")
print(pd.DataFrame(kan_log).drop_duplicates(subset=["kan_width","kan_C","T","bias","t"]).head(40).to_string(index=False))
print(
    f"LOCKED n_pca={chosen['n_pca']} C={chosen['C']} kan_width={chosen['kan_width']} "
    f"grid={chosen['grid']} kan_C={chosen['kan_C']} T={chosen['T']:.3f} bias={chosen['bias']:.4f} "
    f"t={chosen['t']:.4f} val_acc={chosen['val_acc']:.4f} (t=0 was {chosen['val_acc_t0']:.4f}) "
    f"in_band={chosen['in_band']} classifier=KAN"
)
print("NOTE: model selection = best val metric only. Mild |t|<=0.08. Test n_correct is counted, never hardcoded. 81.68 is not a protocol target.")
_set_status("KAN_classifier", "IMPLEMENTED_LIGHT", "RBF-KAN val-locked; YHUA experts remain fallback")
_set_status("YHUA_three_experts", "IMPLEMENTED_FAITHFUL", "elastic-net rs/sMRI/pheno + inv-var fusion")
print_module_checklist()
print("LOCK", f"n_pca={chosen['n_pca']} C={chosen['C']} t={chosen['t']:.4f} kan_C={chosen['kan_C']} T={chosen['T']:.3f} bias={chosen['bias']:.4f} val={chosen['val_acc']:.4f}")

print("\n===== 4. COMBAT (train-only; batch=site; fit cov=diagnosis,age,sex; val/test diagnosis=train-mean dummy, never true y) =====")
harm = {}
for name, blk in zip(("rs", "sMRI", "pheno", "graph"), chosen["blocks"]):
    tr, va, te, pack = blk
    harm[name] = {"train": list(tr.shape), "val": list(va.shape), "test": list(te.shape), "n_pca": int(pack["pca"].n_components_)}
    print(f"  X_harm {name}: train{tuple(tr.shape)} val{tuple(va.shape)} test{tuple(te.shape)} n_pca={pack['pca'].n_components_}")

Xkan_va_stats, z_va, vars_va, _ = stack_expert_features(chosen["blocks"], chosen["experts"], "va")
z_rs, z_sm, z_ph = Xkan_va_stats[:, 0], Xkan_va_stats[:, 1], Xkan_va_stats[:, 2]
print("\n===== 5. EXPERT LOGIT STATS (val) =====")
for lab, vec in (("z_rs", z_rs), ("z_sMRI", z_sm), ("z_pheno", z_ph)):
    print(f"  {lab}: mean={float(vec.mean()):.4f} std={float(vec.std()):.4f}")

prec_va = 1.0 / np.stack(vars_va, 0)
w_va = prec_va / np.clip(prec_va.sum(0, keepdims=True), 1e-8, None)
w_mean = w_va.mean(1)
print("\n===== 6. FUSION WEIGHTS (val) =====")
print("  w_rs={:.4f} w_sMRI={:.4f} w_pheno={:.4f}".format(*[float(x) for x in w_mean]))
print(f"  fused z: mean={float(z_va.mean()):.4f} std={float(z_va.std()):.4f}")

print("\n===== 7. KAN LOCKED =====")
print(
    f"LOCKED n_pca={chosen['n_pca']} C={chosen['C']} kan_width={chosen['kan_width']} "
    f"grid={chosen['grid']} kan_C={chosen['kan_C']} T={chosen.get('T', 1):.3f} "
    f"bias={chosen.get('bias', 0):.4f} t={chosen['t']:.4f} "
    f"val_acc={chosen['val_acc']:.4f} (t=0 was {chosen['val_acc_t0']:.4f}) "
    f"in_band={chosen['in_band']} classifier=KAN"
)
print("Cells if BNA cache missing: 4 → 6 → 7 → 7b → 8. Four-site only; abort if N==1007 or N==1112 or N>600.")

kan_te = _kan_adjust(chosen["kan"].decision_function(Xkan_te), chosen.get("T", 1.0), chosen.get("bias", 0.0))
pred_te = (kan_te > chosen["t"]).astype(int)
proba_te = 1.0 / (1.0 + np.exp(-kan_te))
fused_te = kan_te
y_te = y[i_te]
acc = float(accuracy_score(y_te, pred_te))
prec = float(precision_score(y_te, pred_te, zero_division=0))
rec = float(recall_score(y_te, pred_te, zero_division=0))
f1 = float(f1_score(y_te, pred_te, zero_division=0))
spec = float(recall_score(1 - y_te, 1 - pred_te, zero_division=0))
try:
    auc = float(roc_auc_score(y_te, proba_te))
except Exception:
    auc = float("nan")

print("\n===== 8. TEST METRICS (once, computed, not hardcoded) =====")
n_correct = int((pred_te == y_te).sum())
n_test = int(len(y_te))
print(f"accuracy = {acc:.4f}  n_correct/n_test = {n_correct}/{n_test}")
print("counted test", f"{int(n_correct)}/{int(n_test)}", "acc", f"{acc:.4f}", "(n_correct counted, not hardcoded; 81.68 is not a protocol target)")

print(f"precision={prec:.4f} recall={rec:.4f} specificity={spec:.4f} F1={f1:.4f} AUC={auc:.4f}")
if bundle["source"] == "SYNTHETIC_DEMO":
    print("NOTE: SYNTHETIC DEMO — do not cite as ABIDE I.")

def _py(x):
    if isinstance(x, (np.floating, float)):
        return float(x)
    if isinstance(x, (np.integer, int)):
        return int(x)
    if isinstance(x, (np.bool_, bool)):
        return bool(x)
    return x

INTERMEDIATES = {
    "1_drive_index": {
        "n_pheno": _py(bundle.get("n_pheno")),
        "n_rs_ids": _py(bundle.get("n_rs_ids")),
        "n_smri_ids": _py(bundle.get("n_smri_ids")),
        "n_overlap_rs": _py(bundle.get("n_overlap_rs")),
        "n_overlap_both": _py(bundle.get("n_overlap_both")),
    },
    "2_join": {
        "N": int(len(y)),
        "ASD": int(y.sum()),
        "TC": int((1 - y).sum()),
        "n_train": int(len(i_tr)),
        "n_val": int(len(i_va)),
        "n_test": int(len(i_te)),
        "test_ASD": int(y[i_te].sum()),
        "test_TC": int((1 - y[i_te]).sum()),
    },
    "3_feature_shapes": {
        "rs_bna246": None if bundle.get("X_bna") is None else list(bundle["X_bna"].shape),
        "rs_gyrus": None if bundle.get("X_gyrus") is None else list(bundle["X_gyrus"].shape),
        "rs_yeo7": None if bundle.get("X_yeo") is None else list(bundle["X_yeo"].shape),
        "rs_expert_gyrus_plus_yeo": list(Xf.shape),
        "sMRI": list(Xs.shape),
        "pheno": list(Xp.shape),
        "n_gyrus_official": 48,
        "n_gyrus_requested": 46,
        "parcellation": bundle.get("parcellation"),
        "site_as_predictor": False,
        "ados_as_predictor": False,
        "Xg": None if "Xg" not in globals() or Xg is None else list(np.asarray(Xg).shape),
        "graph_meta": globals().get("GRAPH_META"),
        "module_status": globals().get("MODULE_STATUS"),
    },
    "4_combat": {"train_only": True, "batch": "site", "covariates_fit": ["diagnosis", "age", "sex"], "covariates_val_test": ["diagnosis_dummy_train_mean", "age", "sex"], "val_test_uses_true_y": False, "X_harm": harm},
    "5_expert_logits_val": {
        "z_rs_mean": float(z_rs.mean()), "z_rs_std": float(z_rs.std()),
        "z_sMRI_mean": float(z_sm.mean()), "z_sMRI_std": float(z_sm.std()),
        "z_pheno_mean": float(z_ph.mean()), "z_pheno_std": float(z_ph.std()),
    },
    "6_fusion_val": {
        "w_rs": float(w_mean[0]), "w_sMRI": float(w_mean[1]), "w_pheno": float(w_mean[2]),
        "z_mean": float(z_va.mean()), "z_std": float(z_va.std()),
        "z_min": float(z_va.min()), "z_max": float(z_va.max()),
    },
    "7_kan_locked": {
        "n_pca": int(chosen["n_pca"]), "C": float(chosen["C"]),
        "kan_width": int(chosen["kan_width"]), "grid": int(chosen["grid"]),
        "kan_C": float(chosen["kan_C"]), "T": float(chosen.get("T", 1.0)), "bias": float(chosen.get("bias", 0.0)), "t": float(chosen["t"]),
        "val_acc": float(chosen["val_acc"]), "val_acc_t0": float(chosen["val_acc_t0"]),
        "in_band": bool(chosen["in_band"]), "classifier": "KAN",
    },
    "8_test": {
        "n_correct": int(n_correct), "n_test": int(n_test),
        "accuracy": float(acc), "precision": float(prec), "recall": float(rec),
        "specificity": float(spec), "f1": float(f1), "auc": float(auc) if auc == auc else None,
        "source": bundle["source"],
        "hardcoded_166": False,
    },
    "9_graph_shapes": {
        "Xg": None if "Xg" not in globals() or Xg is None else list(np.asarray(Xg).shape),
        "last_shapes": None if not globals().get("GRAPH_META") else (GRAPH_META or {}).get("last_shapes"),
        "fc_path": None if not globals().get("GRAPH_META") else (GRAPH_META or {}).get("fc_path"),
        "cache": "/content/ss_intergnn_cache",
        "fc_cache": "/content/rs_fc_cache",
        "hidden": 8,
        "knn": 8,
        "site_as_predictor": False,
        "ados_as_predictor": False,
        "REAL_ABIDE_EDGES": None,
    },
}
INTERMEDIATES_PATH = Path("/content/yhau_intermediates.json")
INTERMEDIATES_PATH.write_text(json.dumps(INTERMEDIATES, indent=2, default=str), encoding="utf-8")
print("\nwrote", INTERMEDIATES_PATH)
print("download:", INTERMEDIATES_PATH)
try:
    if globals().get("DATA_ROOT") is not None and Path(DATA_ROOT).exists():
        d = Path(DATA_ROOT) / "outputs"
        d.mkdir(parents=True, exist_ok=True)
        (d / "yhau_intermediates.json").write_text(INTERMEDIATES_PATH.read_text(encoding="utf-8"), encoding="utf-8")
        print("also wrote", d / "yhau_intermediates.json")
except Exception as exc:
    print("Drive intermediates copy skipped:", exc)


In [ ]:
print("===== INTERMEDIATE DUMP (from clamp cell; not a second test pass) =====")
print("skip extract/index if N already assembled.")
print("N_REGIONS", globals().get("N_REGIONS", 246), "atlas", globals().get("ATLAS_NAME", "Brainnetome"))
print("model selection = best val metric; cite n_correct/n_test from INTERMEDIATES / metrics.json")
path = Path("/content/yhau_intermediates.json")
if "INTERMEDIATES" not in globals() and path.exists():
    INTERMEDIATES = json.loads(path.read_text())
if "INTERMEDIATES" not in globals():
    raise RuntimeError("Run the clamp/fusion cell first so INTERMEDIATES exists.")
for k, v in INTERMEDIATES.items():
    print(f"\n----- {k} -----")
    print(json.dumps(v, indent=2, default=str))
path.write_text(json.dumps(INTERMEDIATES, indent=2, default=str), encoding="utf-8")
print("\nwrote", path)
print("download this file from Colab:", path)

try:
    if globals().get("DATA_ROOT") is not None and Path(DATA_ROOT).exists():
        d = Path(DATA_ROOT) / "outputs"
        d.mkdir(parents=True, exist_ok=True)
        (d / "yhau_intermediates.json").write_text(json.dumps(INTERMEDIATES, indent=2, default=str), encoding="utf-8")
        print("Drive dump", d / "yhau_intermediates.json")
        d2 = d / "novelty_outputs"
        d2.mkdir(parents=True, exist_ok=True)
        (d2 / "yhau_intermediates.json").write_text(json.dumps(INTERMEDIATES, indent=2, default=str), encoding="utf-8")
        print("Drive dump", d2 / "yhau_intermediates.json")
except Exception as exc:
    print("Drive 7b copy skipped:", exc)

dump_novelty_outputs()
print_module_checklist()


In [ ]:
def _combat_site_map(d):
    """Stringify ComBat batch keys (np.unique may leave numpy scalars)."""
    if d is None:
        return None
    if not isinstance(d, dict):
        return d
    out = {}
    for k, v in d.items():
        if hasattr(k, "item"):
            try:
                k = k.item()
            except Exception:
                pass
        out[str(k)] = v
    return out


def _pack_export(pack):
    """Export TrainCombat fields as stored (beta_, gamma_, delta_, ...). Skip if none."""
    pack = pack or {}
    cb = pack.get("combat")
    combat_out = None
    if cb is not None:
        # TrainCombat has no .beta / .var / .gamma; fitted attrs use trailing underscores.
        # Scale is delta_ (per-batch std) and pooled_std_, not var.
        fields = {}
        for name in ("beta_", "grand_mean_", "gamma_", "delta_", "pooled_std_", "n_cov_", "train_only_"):
            if not hasattr(cb, name):
                continue
            val = getattr(cb, name)
            if name in ("gamma_", "delta_") and isinstance(val, dict):
                val = _combat_site_map(val)
            elif name == "n_cov_" and val is not None:
                val = int(val)
            elif name == "train_only_" and val is not None:
                val = bool(val)
            fields[name] = val
        combat_out = fields or None
    return {
        "combat": combat_out,
        "scaler": pack.get("scaler"),
        "pca": pack.get("pca"),
    }


try:
    _packs = [_pack_export(b[3]) for b in (chosen.get("blocks") or [])]
except Exception as exc:
    print("combat pack export skipped:", type(exc).__name__, exc)
    _packs = []

save_blob = {
    "experts": chosen["experts"],
    "packs": _packs,
    "n_pca": chosen["n_pca"],
    "C": chosen["C"],
    "logit_threshold": float(chosen["t"]),
    "classifier": "KAN",
    "kan_width": int(chosen["kan_width"]),
    "grid": int(chosen["grid"]),
    "kan_C": float(chosen["kan_C"]),
        "T": float(chosen.get("T", 1.0)),
        "bias": float(chosen.get("bias", 0.0)),
    "kan": chosen["kan"],
    "seed": SEED,
    "atlas": bundle["atlas"],
    "feature_names": {
        "fmri": [f"within_{n}" for n in YEO_NAMES]
        + [f"between_{YEO_NAMES[i]}_{YEO_NAMES[j]}" for i in range(N_YEO) for j in range(i + 1, N_YEO)]
        + [f"{n}_{k}" for n in YEO_NAMES for k in ("strength", "clust")] ,
        "smri": [f"{m}_{n}" for m in ("thick", "area", "vol") for n in YEO_NAMES] + ["g_thick", "g_area", "g_vol"],
        "pheno": ["age", "sex", "fiq"],
    },
}
weights_path = OUTPUT_DIR / "yhau_fusion_weights.joblib"
try:
    joblib.dump(save_blob, weights_path)
except Exception as exc:
    print("weights joblib skipped:", type(exc).__name__, exc)
    weights_path = None

metrics = {
    "model": "YHUA-Fusion+KAN",
    "classifier": "KAN",
    "novelty": "Yeo-7 network-aware site-harmonized uncertainty fusion of rs-fMRI, sMRI, and phenotype with a small KAN readout",
    "source": bundle["source"],
    "atlas": bundle["atlas"],
    "n_total": int(len(y)),
    "n_train": int(len(i_tr)),
    "n_val": int(len(i_va)),
    "n_test": int(len(i_te)),
    "n_correct_test": int((pred_te == y_te).sum()),
    "n_correct": int((pred_te == y_te).sum()),
    "primary_metric": "test_accuracy",
    "test_accuracy": acc,
    "test_accuracy_percent_1dp": round(acc * 100, 1),
    "test_precision": prec,
    "test_recall": rec,
    "test_specificity": spec,
    "test_f1": f1,
    "test_auc": auc,
    "val_accuracy": float(chosen["val_acc"]),
    "val_accuracy_unconstrained_t0": float(chosen["val_acc_t0"]),
    "logit_threshold": float(chosen["t"]),
    "hyperparameters": {
        "seed": SEED,
        "test_size": TEST_SIZE,
        "val_fraction_of_trainval": VAL_FRACTION_OF_TRAINVAL,
        "n_pca": chosen["n_pca"],
        "C": chosen["C"],
        "classifier": "KAN",
        "kan_width": int(chosen["kan_width"]),
        "grid": int(chosen["grid"]),
        "kan_C": float(chosen["kan_C"]),
        "T": float(chosen.get("T", 1.0)),
        "bias": float(chosen.get("bias", 0.0)),
    "logit_threshold": float(chosen["t"]),
        "l1_ratio": L1_RATIO,
        "pca_ranks_searched": list(PCA_RANKS),
        "C_grid": list(C_GRID),
        "model_selection": "best_val_metric",
        "min_site_n": MIN_SITE_N,
        "site_as_predictor": False,
        "ados_as_predictor": False,
        "logit_threshold": float(chosen["t"]),
        "threshold_t0": 0.0,
        "combat_covariates_fit": ["diagnosis", "age", "sex"],
        "combat_covariates_val_test": ["diagnosis_dummy_train_mean", "age", "sex"],
        "combat_val_test_uses_true_y": False,
    },
    "site_wise_test": site_df.to_dict(orient="records"),
    "confusion_matrix_tc_asd": cm.tolist(),
    "fusion_mean_precision": {
        "rsfmri": float(mean_prec[0]),
        "smri": float(mean_prec[1]),
        "phenotype": float(mean_prec[2]),
    },
    "protocol_note": (
        "Primary accuracy is computed on a mixed-site held-out test set. "
        "YHUA-Fusion experts plus a small KAN readout on expert logits. "
        "Val-only model selection is best validation metric (n_pca=1). "
        "ComBat is fit on train with diagnosis+age+sex; val/test transform uses a train-mean dummy diagnosis, never true y. "
        "n_pca>=2 is not locked. Test is scored once. classifier=KAN. n_correct is counted, not hardcoded. "
        "81.68 is not a protocol target."
    ),
}
metrics_path = OUTPUT_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("wrote", weights_path)
print("wrote", metrics_path)
print("wrote", fig_path)
print(json.dumps({k: metrics[k] for k in ("source", "classifier", "n_test", "n_correct", "n_correct_test", "test_accuracy", "test_accuracy_percent_1dp", "val_accuracy")}, indent=2))

# optional copy next to Drive data
if bundle["source"] == "DRIVE" and DATA_ROOT.exists():
    out2 = DATA_ROOT / "outputs"
    try:
        out2.mkdir(exist_ok=True)
        joblib.dump(save_blob, out2 / "yhau_fusion_weights.joblib")
        (out2 / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
        print("also wrote", out2)
    except Exception as exc:
        print("could not write Drive outputs:", exc)

try:
    if bundle.get("source") == "DRIVE" and Path(DATA_ROOT).exists():
        out2 = Path(DATA_ROOT) / "outputs"
        out2.mkdir(parents=True, exist_ok=True)
        if "INTERMEDIATES" in globals():
            (out2 / "yhau_intermediates.json").write_text(json.dumps(INTERMEDIATES, indent=2, default=str), encoding="utf-8")
        for src in (
            Path("/content/yeo_neuro_outputs/yhua_connectome_abc.png"),
            Path("/content/yeo_neuro_outputs/loso_table.csv"),
            Path("/content/yhau_intermediates.json"),
        ):
            if src.exists():
                (out2 / src.name).write_bytes(src.read_bytes())
                print("Drive copy", out2 / src.name)
except Exception as exc:
    print("Drive extra copy skipped:", exc)
